# PageRank: Markov Chains and GPU Computing
## Experiments accompanying the bachelor's thesis

**Marco Mazzocco · Sapienza University of Rome · 2026**  
[Personal website](https://mazzocco51.github.io) · [Contact](mailto:mazzocco.dev@gmail.com)

This notebook brings together the experiments used in the thesis: power iteration,
endpoint Monte Carlo, sparse CUDA kernels, and contact recommendation on Flickr.
The code comes from the original research notebooks and their supporting scripts.
The guide is in English; original Italian comments and result keys are retained where
they help preserve the correspondence with the recorded experiments.

**One notebook is sufficient.** All required Python and CUDA source code is included
below. The `%%writefile` cells materialise it in `_pagerank_runtime/scripts/` when
executed. You do not need to download any separate `.py` files. Experiments run in
separate processes, so GPU allocations and variables do not leak between sections.

### Contents
1. [Setup and execution options](#setup)
2. [Recorded results](#recorded-results)
3. [Monte Carlo implementation and small CPU check](#monte-carlo)
4. [web-Google: CPU/GPU benchmark and convergence](#web-google)
5. [twitter_combined: Monte Carlo accuracy](#twitter)
6. [GAP-twitter: large-scale computation and CUDA kernels](#gap)
7. [Flickr: personalised contact recommendation](#flickr)
8. [Optional exploratory analyses](#exploratory)
9. [Run selected experiments](#run)
10. [Sources and organisation](#sources)

The default **Run All** performs a small CPU check and displays historical summaries;
it does **not** download datasets or start long GPU experiments. Enable the desired
flags in the next section to reproduce selected experiments. Recorded results are
labelled explicitly and are never presented as outputs from a new benchmark.


<a id="setup"></a>
## 1. Setup and execution options

Use Python 3.10+ with NumPy, SciPy, pandas, Matplotlib, requests and IPython.
For GPU experiments, use a CUDA-capable NVIDIA GPU and a compatible CuPy installation
(the original experiments used a Tesla T4). On Colab, select a GPU runtime first.
The GAP conversion additionally requires Linux, `gcc`, `tar` and `wget`.

The optional installation cell installs only the CPU dependencies. Install the CuPy
distribution matching your CUDA runtime separately, if it is not already available.
See the [CuPy installation guide](https://docs.cupy.dev/en/stable/install.html).

Data and new results are written under `pagerank_runs/`, excluded by the supplied
`.gitignore`. This notebook never needs access to the author's original folders.
GAP-twitter has about 1.47 billion edges: its archive is about 9.5 GB and the two
converted orientations occupy about 12.7 GB together. Allow additional disk, RAM
and GPU memory; the main conversion and kernel scripts check available memory.
The full Flickr experiment computes 5,000 personalised rankings and can take hours.


In [ ]:
INSTALL_CPU_DEPENDENCIES = False
RUN_CPU_CHECK = True
RUN_WEB_BENCHMARK = False
RUN_WEB_KERNEL_COMPARISON = False
RUN_CONVERGENCE = False
RUN_TWITTER_MONTE_CARLO = False
RUN_GAP_PREPARATION = False
RUN_GAP_POWER = False
RUN_GAP_KERNEL_COMPARISON = False
RUN_GAP_WEIGHTED_KERNELS = False  # supplementary scalar/vector comparison
RUN_GAP_MONTE_CARLO = False
RUN_GAP_VALIDATION = False       # graph checks + MC error against power iteration
RUN_FLICKR = False
RUN_FLICKR_EXPLORATORY = False   # quota policies; requires RUN_FLICKR
RUN_FLICKR_TIE_AUDIT = False     # 300-user GPU audit; requires RUN_FLICKR

from pathlib import Path
import os
import sys
import subprocess

NOTEBOOK_DIR = Path.cwd().resolve()
SCRIPT_DIR = NOTEBOOK_DIR / '_pagerank_runtime' / 'scripts'
RUN_DIR = NOTEBOOK_DIR / 'pagerank_runs'
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(exist_ok=True)

if INSTALL_CPU_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'numpy', 'scipy', 'pandas', 'matplotlib', 'requests', 'ipython'], check=True)
print('Python:', sys.version.split()[0])
print('Working directory:', NOTEBOOK_DIR)


In [ ]:
import importlib.util
import json
import shutil
import numpy as np
import pandas as pd
from IPython.display import display, Image

def run_program(name, section, *args, gpu=False, extra_env=None):
    """Run an included script with live output in its own experiment directory."""
    folder = RUN_DIR / section
    folder.mkdir(exist_ok=True)
    env = os.environ.copy()
    env['PYTHONUTF8'] = '1'
    env['PYTHONIOENCODING'] = 'utf-8'
    if extra_env:
        env.update(extra_env)
    if gpu:
        subprocess.run([sys.executable, '-c',
            'import cupy as cp; assert cp.cuda.runtime.getDeviceCount() > 0; '
            'print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])'],
            check=True, env=env)
    subprocess.run([sys.executable, '-u', str(SCRIPT_DIR / name), *map(str, args)],
                   cwd=folder, env=env, check=True)
    return folder

def download_web_google(folder):
    import urllib.request
    folder.mkdir(exist_ok=True)
    target = folder / 'web-Google.txt.gz'
    if not target.exists():
        partial = target.with_suffix('.gz.part')
        urllib.request.urlretrieve('https://snap.stanford.edu/data/web-Google.txt.gz', partial)
        partial.replace(target)
    return target


<a id="recorded-results"></a>
## 2. Recorded results — not a new execution

The following compact summaries are embedded from the saved research results.
They remain available without a GPU or a dataset download. Benchmark times depend
on the machine, software and session. In particular, the later three-run GAP kernel
comparison is a separate measurement series from the original large-graph timing.
The Flickr evaluation measures links observed later, not user acceptance of suggestions.


### Historical results visible without running the notebook

| web-Google implementation | Median time (ms) | Repetitions |
|---|---:|---:|
| CPU baseline | 1671.62 | 7 |
| GPU: `base` | 100.46 | 7 |
| GPU: `dot` | 91.61 | 7 |
| GPU: `controllo_5` | 45.59 | 7 |
| GPU: `alpha_nei_pesi` | 45.00 | 7 |

| Flickr method | Links identified | HitRate@10 (%) | Precision@10 (%) |
|---|---:|---:|---:|
| Global PageRank | 71 | 1.22 | 0.142 |
| Personalized PageRank | 491 | 6.96 | 0.982 |
| Degree + reciprocity | 554 | 7.06 | 1.108 |
| PPR + reciprocity | 795 | 10.12 | 1.590 |

Flickr: **5,000 users and 50,000 suggestions per method**. The combined method identifies more reciprocal links, while its non-reciprocal hits decrease from 441 to 213 compared with PPR alone.


In [ ]:
RECORDED = {'web': {'prove': [{'variante': 'alpha_nei_pesi', 'ripetizione': 1, 'seconds': 0.04567708200011111, 'iterazioni': 65},
    {'variante': 'dot', 'ripetizione': 1, 'seconds': 0.09199908400000822, 'iterazioni': 62},
    {'variante': 'base', 'ripetizione': 1, 'seconds': 0.09980295599984856, 'iterazioni': 62},
    {'variante': 'controllo_5', 'ripetizione': 1, 'seconds': 0.04618812700005037, 'iterazioni': 65},
    {'variante': 'base', 'ripetizione': 2, 'seconds': 0.10231559199996809, 'iterazioni': 62},
    {'variante': 'dot', 'ripetizione': 2, 'seconds': 0.09116880999999921, 'iterazioni': 62},
    {'variante': 'controllo_5', 'ripetizione': 2, 'seconds': 0.04601790199990319, 'iterazioni': 65},
    {'variante': 'alpha_nei_pesi', 'ripetizione': 2, 'seconds': 0.044369548000076975, 'iterazioni': 65},
    {'variante': 'base', 'ripetizione': 3, 'seconds': 0.100893688999804, 'iterazioni': 62},
    {'variante': 'dot', 'ripetizione': 3, 'seconds': 0.09076712000000953, 'iterazioni': 62},
    {'variante': 'controllo_5', 'ripetizione': 3, 'seconds': 0.04558850200010056, 'iterazioni': 65},
    {'variante': 'alpha_nei_pesi', 'ripetizione': 3, 'seconds': 0.045047050999983185, 'iterazioni': 65},
    {'variante': 'dot', 'ripetizione': 4, 'seconds': 0.09295321800004785, 'iterazioni': 62},
    {'variante': 'controllo_5', 'ripetizione': 4, 'seconds': 0.047999935999996524, 'iterazioni': 65},
    {'variante': 'alpha_nei_pesi', 'ripetizione': 4, 'seconds': 0.04485376599996016, 'iterazioni': 65},
    {'variante': 'base', 'ripetizione': 4, 'seconds': 0.09974267300003703, 'iterazioni': 62},
    {'variante': 'alpha_nei_pesi', 'ripetizione': 5, 'seconds': 0.04499798000006194, 'iterazioni': 65},
    {'variante': 'dot', 'ripetizione': 5, 'seconds': 0.09161217299993041, 'iterazioni': 62},
    {'variante': 'controllo_5', 'ripetizione': 5, 'seconds': 0.04541085599998951, 'iterazioni': 65},
    {'variante': 'base', 'ripetizione': 5, 'seconds': 0.1005303100000674, 'iterazioni': 62},
    {'variante': 'alpha_nei_pesi', 'ripetizione': 6, 'seconds': 0.04544452799996179, 'iterazioni': 65},
    {'variante': 'base', 'ripetizione': 6, 'seconds': 0.10001773000021785, 'iterazioni': 62},
    {'variante': 'dot', 'ripetizione': 6, 'seconds': 0.09535920399980569, 'iterazioni': 62},
    {'variante': 'controllo_5', 'ripetizione': 6, 'seconds': 0.04509939700005816, 'iterazioni': 65},
    {'variante': 'base', 'ripetizione': 7, 'seconds': 0.10045914899978925, 'iterazioni': 62},
    {'variante': 'alpha_nei_pesi', 'ripetizione': 7, 'seconds': 0.04455499799996687, 'iterazioni': 65},
    {'variante': 'controllo_5', 'ripetizione': 7, 'seconds': 0.04541739900014363, 'iterazioni': 65},
    {'variante': 'dot', 'ripetizione': 7, 'seconds': 0.09136134200002743, 'iterazioni': 62}]}, 'gap_kernel': {'trials': [{'variant': 'csr_vector', 'repetition': 1, 'seconds': 33.89882954699988, 'iterations': 49},
    {'variant': 'cupy', 'repetition': 1, 'seconds': 46.99863702499988, 'iterations': 49},
    {'variant': 'csr_vector_unweighted', 'repetition': 1, 'seconds': 45.44369302300038, 'iterations': 49},
    {'variant': 'cupy', 'repetition': 2, 'seconds': 48.987220671000614, 'iterations': 49},
    {'variant': 'csr_vector', 'repetition': 2, 'seconds': 49.134590288000254, 'iterations': 49},
    {'variant': 'csr_vector_unweighted', 'repetition': 2, 'seconds': 46.196455272000094, 'iterations': 49},
    {'variant': 'cupy', 'repetition': 3, 'seconds': 49.28925142400021, 'iterations': 49},
    {'variant': 'csr_vector', 'repetition': 3, 'seconds': 49.1782416799997, 'iterations': 49},
    {'variant': 'csr_vector_unweighted', 'repetition': 3, 'seconds': 46.3637710590001, 'iterations': 49}]}, 'flickr': [{'method': 'Personalized PageRank', 'collegamenti_recuperati': '491', 'utenti_con_successo_pct': '6.959999999999999', 'precisione_pct': '0.9820000000000002'},
    {'method': 'PageRank globale', 'collegamenti_recuperati': '71', 'utenti_con_successo_pct': '1.22', 'precisione_pct': '0.142'},
    {'method': 'Grado entrante', 'collegamenti_recuperati': '78', 'utenti_con_successo_pct': '1.3599999999999999', 'precisione_pct': '0.156'},
    {'method': 'Reciprocità + grado', 'collegamenti_recuperati': '554', 'utenti_con_successo_pct': '7.06', 'precisione_pct': '1.108'},
    {'method': 'Reciprocità + PPR', 'collegamenti_recuperati': '795', 'utenti_con_successo_pct': '10.12', 'precisione_pct': '1.59'},
    {'method': 'PPR + quota fissa (5)', 'collegamenti_recuperati': '711', 'utenti_con_successo_pct': '9.44', 'precisione_pct': '1.422'},
    {'method': 'PPR + quota adattiva', 'collegamenti_recuperati': '751', 'utenti_con_successo_pct': '9.68', 'precisione_pct': '1.5019999999999998'}]}


In [ ]:
web_trials = pd.DataFrame(RECORDED['web']['prove'])
web_summary = web_trials.groupby('variante')['seconds'].agg(['median', 'min', 'max']) * 1000
display(web_summary.rename_axis('GPU variant').rename(columns=lambda c: c + ' (ms)'))

labels = {
    'PageRank globale': 'Global PageRank',
    'Personalized PageRank': 'Personalized PageRank',
    'Reciprocità + grado': 'Degree + reciprocity',
    'Reciprocità + PPR': 'PPR + reciprocity',
}
flickr_summary = pd.DataFrame(RECORDED['flickr']).set_index('method').loc[list(labels)]
flickr_summary = flickr_summary[['collegamenti_recuperati', 'utenti_con_successo_pct', 'precisione_pct']].astype(float)
flickr_summary.columns = ['Links identified', 'HitRate@10 (%)', 'Precision@10 (%)']
flickr_summary.index = [labels[x] for x in flickr_summary.index]
display(flickr_summary)
print('Flickr population: 5,000 users, 10 suggestions per user (50,000 suggestions per method).')
gap_trials = pd.DataFrame(RECORDED['gap_kernel']['trials'])
display(gap_trials.groupby('variant')['seconds'].agg(['median', 'min', 'max']).rename(columns=lambda c: c + ' (s)'))


<a id="monte-carlo"></a>
## 3. Endpoint Monte Carlo and CPU check

Implements Algorithm 2: start uniformly, stop with probability $1-\alpha$ before
each step, and count endpoints. At a dangling node, the next node is uniform.
Walks are processed in batches. The same implementation supports NumPy and CuPy.
The small default check compares the iteration with a dense linear solve and checks
the stochastic estimate against that reference. It does not reproduce GPU timings.


In [ ]:
%%writefile _pagerank_runtime/scripts/montecarlo.py
"""
PageRank con il metodo Monte Carlo end-point, eseguito su GPU con CuPy.

Implementa l'Algoritmo 2 della tesi (Sezione 4.4), fonte: Avrachenkov et al.,
SIAM J. Numer. Anal. 45(2), 2007, Algoritmo 1 a p. 892.

Ogni cammino parte da una pagina scelta uniformemente; prima di ogni passo si
ferma con probabilita' 1 - alpha, altrimenti segue un link uscente scelto
uniformemente (o salta a una pagina uniforme se la pagina corrente non ha link
uscenti). La stima di r_j e' c_j / N, dove c_j e' il numero di cammini
terminati in j.

Su GPU gli N cammini vengono simulati insieme, a blocchi di `batch` cammini:
a ogni passo si estraggono i numeri casuali di tutti i cammini ancora attivi,
si registrano quelli che si fermano e si spostano gli altri. Non ci sono kernel
scritti a mano: solo operazioni vettoriali di CuPy.

Uso (Colab):
    !python montecarlo.py test        # grafo della Figura 3.2, controllo rapido
    !python montecarlo.py twitter     # SNAP twitter_combined: errore al crescere di N
    !python montecarlo.py gap         # SuiteSparse GAP-twitter: tempi e scala

twitter_combined: prima colonna sorgente, seconda destinazione.
GAP-twitter: seconda coordinata sorgente (follower), prima destinazione
(utente seguito). I pesi artificiali GAP sono ignorati.
"""

import argparse
import json
import os
import subprocess
import sys
import time
import tarfile
import signal

import numpy as np

try:
    import cupy as cp
    HAVE_CUPY = True
except ImportError:          # permette di provare la logica anche senza GPU
    cp = None
    HAVE_CUPY = False


# =============================================================================
# Utilita'
# =============================================================================

def get_xp(use_gpu):
    if use_gpu:
        if not HAVE_CUPY:
            sys.exit("CuPy non disponibile: su Colab scegli un runtime con GPU.")
        return cp
    return np


def sync(xp):
    if xp is not np:
        cp.cuda.Stream.null.synchronize()


class Timer:
    """Cronometro: eventi CUDA su GPU, perf_counter su CPU."""

    def __init__(self, xp):
        self.xp = xp

    def __enter__(self):
        if self.xp is np:
            self.t0 = time.perf_counter()
        else:
            self.start, self.end = cp.cuda.Event(), cp.cuda.Event()
            self.start.record()
        return self

    def __exit__(self, *exc):
        if self.xp is np:
            self.seconds = time.perf_counter() - self.t0
        else:
            self.end.record()
            self.end.synchronize()
            self.seconds = cp.cuda.get_elapsed_time(self.start, self.end) / 1e3


def to_device(arr, xp, dtype, chunk=1 << 26):
    """Copia un array (anche memmap) sul dispositivo a blocchi."""
    arr = np.asarray(arr) if not isinstance(arr, np.memmap) else arr
    if xp is np:
        return np.asarray(arr, dtype=dtype)
    out = cp.empty(arr.size, dtype=dtype)
    for s in range(0, arr.size, chunk):
        e = min(s + chunk, arr.size)
        out[s:e] = cp.asarray(np.asarray(arr[s:e], dtype=dtype))
    return out


# =============================================================================
# Grafo: liste uscenti O_i in formato compresso (indptr, indices)
# =============================================================================

def out_lists(src, dst, n):
    """Liste uscenti: le destinazioni di i sono indices[indptr[i]:indptr[i+1]].

    E' la struttura O_i della Sezione 4.4, memorizzata in due array:
    indptr (n+1 posizioni) e indices (m destinazioni).
    """
    outdeg = np.bincount(src, minlength=n)
    indptr = np.zeros(n + 1, dtype=np.int64)
    np.cumsum(outdeg, out=indptr[1:])
    order = np.argsort(src, kind="stable")
    indices = dst[order].astype(np.int32)
    return indptr, indices


def load_edge_list(path):
    """Legge archi orientati distinti, conserva gli autoanelli e rinumera i nodi."""
    e = np.loadtxt(path, dtype=np.int64, comments="#")
    e = np.atleast_2d(e)
    rows = len(e)
    e = np.unique(e, axis=0)
    print(f"Lista archi: {rows:,} righe, {len(e):,} archi distinti; "
          f"rimossi {rows-len(e):,} duplicati; "
          f"autoanelli conservati: {int(np.count_nonzero(e[:, 0] == e[:, 1]))}",
          flush=True)
    ids, flat = np.unique(e, return_inverse=True)
    src, dst = flat.reshape(e.shape).T
    return src.astype(np.int64), dst.astype(np.int64), ids.size, ids


# =============================================================================
# Monte Carlo end-point (Algoritmo 2)
# =============================================================================

def pagerank_montecarlo(indptr, indices, n, N, alpha=0.85, seed=0,
                        batch=10_000_000, xp=None):
    """Stima del PageRank con N cammini aleatori indipendenti.

    indptr, indices: liste uscenti gia' sul dispositivo.
    Restituisce (stima r_hat come vettore float64, numero totale di passi).
    """
    xp = xp or (cp if HAVE_CUPY else np)
    rng = xp.random.RandomState(seed)
    counts = xp.zeros(n, dtype=xp.int64)
    total_steps = 0

    for start in range(0, N, batch):
        B = min(batch, N - start)
        ends = xp.empty(B, dtype=xp.int32)          # pagina finale di ogni cammino
        walk = xp.arange(B, dtype=xp.int32)         # identificativi dei cammini attivi
        pos = rng.randint(0, n, size=B, dtype=xp.int64)   # riga 4: partenza uniforme

        while walk.size > 0:
            # riga 5: un nuovo campione uniforme; se >= alpha il cammino si ferma
            go = rng.random_sample(walk.size) < alpha
            stop = ~go
            ends[walk[stop]] = pos[stop]            # riga 10 (conteggio a fine blocco)
            walk, pos = walk[go], pos[go]
            if walk.size == 0:
                break
            total_steps += int(walk.size)

            # righe 6-9: link uscente uniforme, oppure pagina uniforme se O_i e' vuota
            begin = indptr[pos]
            deg = indptr[pos + 1] - begin
            u = rng.random_sample(walk.size)
            k = xp.minimum((u * deg).astype(xp.int64), xp.maximum(deg - 1, 0))
            has_links = deg > 0
            nxt = xp.where(has_links,
                           indices[xp.where(has_links, begin + k, 0)].astype(xp.int64),
                           rng.randint(0, n, size=walk.size, dtype=xp.int64))
            pos = nxt

        counts += xp.bincount(ends, minlength=n)

    return counts.astype(xp.float64) / N, total_steps   # riga 11


# =============================================================================
# Riferimento: power iteration in float64 (Algoritmo 1, stessa formula (4.2))
# =============================================================================

def pagerank_power_reference(src, dst, n, alpha=0.85, tol=1e-12, max_iter=5000):
    """PageRank "esatto" di riferimento su CPU, in float64.

    Con Delta_t < 1e-12 e alpha = 0,85, per la Sezione 4.3 l'errore in norma 1
    e' al massimo alpha/(1-alpha) * 1e-12, circa 6e-12: trascurabile rispetto
    all'errore statistico del Monte Carlo.
    """
    import scipy.sparse as sp
    outdeg = np.bincount(src, minlength=n).astype(np.float64)
    w = 1.0 / outdeg[src]
    Pt = sp.csr_matrix((w, (dst, src)), shape=(n, n))     # r <- Pt @ r
    a = (outdeg == 0).astype(np.float64)
    r = np.full(n, 1.0 / n)
    for t in range(1, max_iter + 1):
        b = (1.0 - alpha + alpha * (r @ a)) / n
        r_new = alpha * (Pt @ r) + b
        delta = np.abs(r_new - r).sum()
        r = r_new
        if delta < tol:
            return r, t, delta
    return r, max_iter, delta


# =============================================================================
# Esperimenti
# =============================================================================

def run_test(args):
    """Grafo della Figura 3.2 (Esempio 4.4): verifica rapida della correttezza."""
    xp = get_xp(not args.cpu)
    names = list("ABCDEFG")
    edges = [("A", "B"), ("A", "D"), ("B", "C"), ("B", "E"),
             ("C", "A"), ("E", "F"), ("F", "G"), ("G", "E")]
    idx = {v: i for i, v in enumerate(names)}
    src = np.array([idx[s] for s, _ in edges])
    dst = np.array([idx[d] for _, d in edges])
    n = len(names)

    r_ref, it, _ = pagerank_power_reference(src, dst, n, args.alpha)
    indptr, indices = out_lists(src, dst, n)
    d_indptr, d_indices = to_device(indptr, xp, np.int64), to_device(indices, xp, np.int32)

    N = args.N[0] if args.N else 10_000_000
    with Timer(xp) as tm:
        r_hat, steps = pagerank_montecarlo(d_indptr, d_indices, n, N, args.alpha,
                                           seed=args.seeds[0], batch=args.batch, xp=xp)
    r_hat = r_hat.get() if xp is not np else r_hat

    print(f"Grafo della Figura 3.2, alpha = {args.alpha}, N = {N:,}")
    print(f"{'nodo':>4} {'riferimento':>12} {'Monte Carlo':>12} {'dev.std':>9}")
    for i, v in enumerate(names):
        sd = np.sqrt(r_ref[i] * (1 - r_ref[i]) / N)
        print(f"{v:>4} {r_ref[i]:12.6f} {r_hat[i]:12.6f} {sd:9.2e}")
    err = np.abs(r_hat - r_ref).sum()
    print(f"errore L1 = {err:.2e}   somma = {r_hat.sum():.6f}")
    print(f"passi medi per cammino = {steps / N:.3f}  (atteso alpha/(1-alpha) = "
          f"{args.alpha / (1 - args.alpha):.3f})   tempo = {tm.seconds:.2f} s")


def run_twitter(args):
    """SNAP twitter_combined: errore in norma 1 e tempo al crescere di N."""
    xp = get_xp(not args.cpu)
    path = os.path.join(args.workdir, "twitter_combined.txt")
    if not os.path.exists(path):
        url = "https://snap.stanford.edu/data/twitter_combined.txt.gz"
        subprocess.run(["wget", "-q", "-nc", "-P", args.workdir, url], check=True)
        subprocess.run(["gunzip", "-kf", path + ".gz"], check=True)

    src, dst, n, ids = load_edge_list(path)
    m = src.size
    print(f"twitter_combined: n = {n:,}  m = {m:,}  "
          f"penzolanti = {(np.bincount(src, minlength=n) == 0).sum():,}")

    t0 = time.perf_counter()
    r_ref, it, delta = pagerank_power_reference(src, dst, n, args.alpha)
    print(f"riferimento (power iteration float64): {it} iterazioni, "
          f"Delta = {delta:.1e}, {time.perf_counter() - t0:.1f} s")

    indptr, indices = out_lists(src, dst, n)
    d_indptr, d_indices = to_device(indptr, xp, np.int64), to_device(indices, xp, np.int32)
    r_ref_dev = xp.asarray(r_ref)

    # riscaldamento: la prima chiamata compila i kernel di CuPy
    pagerank_montecarlo(d_indptr, d_indices, n, min(100_000, args.batch),
                        args.alpha, seed=12345, batch=args.batch, xp=xp)
    sync(xp)

    rows = []
    print(f"\n{'N':>12} {'seme':>5} {'tempo [s]':>10} {'passi/cammino':>14} "
          f"{'errore L1':>11} {'err. max':>10}")
    for N in args.N:
        for seed in args.seeds:
            with Timer(xp) as tm:
                r_hat, steps = pagerank_montecarlo(d_indptr, d_indices, n, N,
                                                   args.alpha, seed=seed,
                                                   batch=args.batch, xp=xp)
            err_l1 = float(xp.abs(r_hat - r_ref_dev).sum())
            err_max = float(xp.abs(r_hat - r_ref_dev).max())
            total = float(r_hat.sum())
            rows.append(dict(N=N, seed=seed, seconds=tm.seconds,
                             steps_per_walk=steps / N, err_l1=err_l1,
                             err_max=err_max, sum=total))
            print(f"{N:12,} {seed:5d} {tm.seconds:10.3f} {steps / N:14.3f} "
                  f"{err_l1:11.3e} {err_max:10.2e}")

    # pendenza di log(errore) rispetto a log(N): la teoria prevede circa -1/2
    Ns = np.array([r["N"] for r in rows], dtype=float)
    errs = np.array([r["err_l1"] for r in rows])
    slope = np.polyfit(np.log10(Ns), np.log10(errs), 1)[0]
    print(f"\npendenza log-log dell'errore L1 rispetto a N: {slope:.3f} "
          f"(attesa circa -0,5 quando N e' molto maggiore di n = {n:,})")

    out = dict(grafo="twitter_combined", n=int(n), m=int(m), alpha=args.alpha,
               preprocessing="unique_directed_edges_keep_self_loops",
               self_loops=int(np.count_nonzero(src == dst)),
               riferimento=dict(iterazioni=int(it), delta=float(delta)),
               device=device_name(xp), batch=args.batch, pendenza=slope, prove=rows)
    save_results(out, args, "montecarlo_twitter")
    plot_error(rows, args)


def device_name(xp):
    if xp is np:
        return "CPU (numpy)"
    props = cp.cuda.runtime.getDeviceProperties(0)
    name = props["name"]
    return name.decode() if isinstance(name, bytes) else name


def save_results(out, args, name):
    path = os.path.join(args.workdir, name + ".json")
    with open(path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"risultati salvati in {path}")


def plot_error(rows, args):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    Ns = np.array(sorted({r["N"] for r in rows}), dtype=float)
    mean_err = [np.mean([r["err_l1"] for r in rows if r["N"] == N]) for N in Ns]
    plt.figure(figsize=(6, 4.2))
    for r in rows:
        plt.loglog(r["N"], r["err_l1"], "o", color="0.6", ms=4)
    plt.loglog(Ns, mean_err, "o-", color="C0", label="errore L1 medio")
    plt.loglog(Ns, mean_err[0] * np.sqrt(Ns[0] / Ns), "--", color="C1",
               label=r"andamento $N^{-1/2}$")
    plt.xlabel("numero di cammini N")
    plt.ylabel(r"$\|\hat r - \bar r\|_1$")
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    path = os.path.join(args.workdir, "montecarlo_errore.pdf")
    plt.savefig(path)
    print(f"grafico salvato in {path}")


# =============================================================================
# GAP-twitter: conversione in liste USCENTI e prova su larga scala
# =============================================================================

CONV_OUT_C = r"""
/* Converte GAP-twitter.mtx (letto da stdin) in liste USCENTI:
 *   passata 1: conta i link uscenti di ogni sorgente i e scrive follow_indptr.bin
 *   passata 2: scrive la destinazione j-1 di ogni arco nella posizione di i
 * GAP: seconda coordinata = follower sorgente, prima = utente seguito.
 */
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <inttypes.h>

#define CHECK(x) do { if (!(x)) { fprintf(stderr, "errore: %s\n", #x); return 2; } } while (0)

static long read_header(char *line, int size, long *nnz) {
    long nr = 0, nc = 0;
    while (fgets(line, size, stdin)) {
        if (line[0] == '%') continue;
        sscanf(line, "%ld %ld %ld", &nr, &nc, nnz);
        break;
    }
    return nr;
}

static int parse(char *p, long *i, long *j) {
    *i = 0; *j = 0;
    while (*p == ' ' || *p == '\t') p++;
    if (*p < '0' || *p > '9') return 0;
    while (*p >= '0' && *p <= '9') *i = *i * 10 + (*p++ - '0');
    while (*p == ' ' || *p == '\t') p++;
    while (*p >= '0' && *p <= '9') *j = *j * 10 + (*p++ - '0');
    long tmp = *i; *i = *j; *j = tmp; /* follower -> seguito */
    return 1;
}

int main(int argc, char **argv) {
    if (argc < 3) { fprintf(stderr, "uso: conv_out 1|2 cartella\n"); return 1; }
    int pass = atoi(argv[1]);
    const char *dir = argv[2];
    static char iob[1 << 24];
    setvbuf(stdin, iob, _IOFBF, sizeof iob);
    char line[256], path[1024];
    long nnz = 0;
    long n = read_header(line, sizeof line, &nnz);
    CHECK(n > 0 && n <= INT32_MAX && nnz >= 0);
    fprintf(stderr, "n=%ld  nnz=%ld\n", n, nnz);

    if (pass == 1) {
        int64_t *deg = calloc(n + 1, sizeof(int64_t));
        CHECK(deg);
        long i, j, done = 0;
        while (fgets(line, sizeof line, stdin)) {
            if (!parse(line, &i, &j)) continue;
            CHECK(i >= 1 && i <= n && j >= 1 && j <= n);
            deg[i - 1]++;
            if (++done % 200000000 == 0) fprintf(stderr, "  %ld archi\n", done);
        }
        CHECK(!ferror(stdin) && done == nnz);
        int64_t *indptr = malloc(8LL * (n + 1));
        CHECK(indptr);
        indptr[0] = 0;
        for (long v = 0; v < n; v++) indptr[v + 1] = indptr[v] + deg[v];
        snprintf(path, sizeof path, "%s/follow_indptr.bin.part", dir);
        FILE *f = fopen(path, "wb"); CHECK(f);
        CHECK(fwrite(indptr, 8, n + 1, f) == (size_t)(n + 1));
        CHECK(fclose(f) == 0);
        fprintf(stderr, "passata 1: %ld archi\n", done);
        free(deg); free(indptr);
    } else {
        int64_t *cur = malloc(8LL * (n + 1));
        int64_t *bounds = malloc(8LL * (n + 1));
        CHECK(cur && bounds);
        snprintf(path, sizeof path, "%s/follow_indptr.bin", dir);
        FILE *f = fopen(path, "rb");
        CHECK(f);
        if (fread(cur, 8, n + 1, f) != (size_t)(n + 1)) { fprintf(stderr, "indptr corto\n"); return 2; }
        rewind(f);
        CHECK(fread(bounds, 8, n + 1, f) == (size_t)(n + 1));
        fclose(f);
        int64_t m = cur[n];
        CHECK(cur[0] == 0 && m == nnz);
        for (long v = 0; v < n; v++) CHECK(cur[v] >= 0 && cur[v] <= cur[v+1]);
        fprintf(stderr, "Costruzione in RAM: %.2f GiB (array e cursori)\n",
                (4.0*m + 16.0*(n+1))/(1024.0*1024*1024));
        int32_t *ind = malloc(m ? 4ULL * m : 1);
        CHECK(ind);
        long i, j, done = 0;
        while (fgets(line, sizeof line, stdin)) {
            if (!parse(line, &i, &j)) continue;
            CHECK(i >= 1 && i <= n && j >= 1 && j <= n);
            CHECK(cur[i - 1] < bounds[i]);
            ind[cur[i - 1]++] = (int32_t)(j - 1);
            if (++done % 200000000 == 0) fprintf(stderr, "  %ld archi\n", done);
        }
        CHECK(!ferror(stdin) && done == m);
        for (long v = 0; v < n; v++) CHECK(cur[v] == bounds[v+1]);
        fprintf(stderr, "Scrittura sequenziale su disco...\n");
        snprintf(path, sizeof path, "%s/follow_indices.bin.part", dir);
        f = fopen(path, "wb"); CHECK(f);
        CHECK(fwrite(ind, 4, m, f) == (size_t)m);
        CHECK(fclose(f) == 0);
        free(ind); free(cur); free(bounds);
        fprintf(stderr, "passata 2: %ld archi\n", done);
    }
    return 0;
}
"""


def build_gap_out_lists(workdir):
    tar = os.path.join(workdir, "GAP-twitter.tar.gz")
    ip = os.path.join(workdir, "follow_indptr.bin")
    ic = os.path.join(workdir, "follow_indices.bin")
    marker = os.path.join(workdir, "follow_complete.json")
    if os.path.exists(marker) and os.path.exists(ip) and os.path.exists(ic):
        with open(marker) as f:
            meta = json.load(f)
        if (meta.get("orientation") != "follower_to_followed"
                or meta.get("converter") != "ram-follow-v1"):
            raise RuntimeError("Orientamento o versione dei file GAP non verificati")
        if (os.path.getsize(ip) != 8 * (meta["n"] + 1)
                or os.path.getsize(ic) != 4 * meta["m"]):
            raise RuntimeError("Dimensioni dei file convertiti non valide")
        print("liste uscenti gia' presenti, salto la conversione")
        return ip, ic
    if not os.path.exists(tar):
        print("scarico GAP-twitter (circa 9,5 GB)...")
        subprocess.run(["wget", "-c", "-q", "-O", tar,
                        "https://sparse.tamu.edu/MM/GAP/GAP-twitter.tar.gz"], check=True)
    # Legge soltanto l'intestazione; non estrae l'intero archivio.
    with tarfile.open(tar, "r|gz") as archive:
        for member in archive:
            if member.name.endswith("GAP-twitter.mtx"):
                with archive.extractfile(member) as stream:
                    for line in stream:
                        if line.startswith(b"%") or not line.strip():
                            continue
                        n, nc, m = map(int, line.split())
                        break
                break
        else:
            raise RuntimeError("GAP-twitter.mtx assente nell'archivio")
    if n != nc or n <= 0 or m < 0:
        raise RuntimeError("Intestazione del grafo non valida")

    # Recupera anche la prima passata prodotta dalla vecchia versione.
    first_ok = False
    if os.path.exists(ip) and os.path.getsize(ip) == 8 * (n + 1):
        ptr = np.memmap(ip, dtype=np.int64, mode="r")
        first_ok = int(ptr[0]) == 0 and int(ptr[-1]) == m
        for s in range(0, n, 1_000_000):
            e = min(n, s + 1_000_000)
            first_ok = first_ok and bool(np.all(ptr[s:e] <= ptr[s+1:e+1]))
        del ptr
    if first_ok:
        print("Prima passata valida: la riutilizzo.", flush=True)
    if os.path.exists(ic):
        print("File destinazioni senza conferma di completamento: verra' ricostruito.", flush=True)

    if os.path.exists("/proc/meminfo"):
        with open("/proc/meminfo") as f:
            available = next(int(line.split()[1]) * 1024 for line in f
                             if line.startswith("MemAvailable:"))
        required = 4 * m + 16 * (n + 1) + 1024**3
        print(f"RAM disponibile: {available / 2**30:.2f} GiB; "
              f"richiesta con margine: {required / 2**30:.2f} GiB", flush=True)
        if available < required:
            raise RuntimeError("RAM disponibile insufficiente: liberare memoria prima della conversione")

    src_c = os.path.join(workdir, "conv_follow.c")
    exe = os.path.join(workdir, "conv_follow")
    with open(src_c, "w") as f:
        f.write(CONV_OUT_C)
    subprocess.run(["gcc", "-O2", "-o", exe, src_c], check=True)
    for p in (("2",) if first_ok else ("1", "2")):
        t0 = time.perf_counter()
        # Niente shell: controlla entrambi i processi e li ferma all'interruzione.
        reader = subprocess.Popen(["tar", "-xzOf", tar, "--wildcards", "*GAP-twitter.mtx"],
                                  stdout=subprocess.PIPE, start_new_session=True)
        writer = None
        try:
            writer = subprocess.Popen([exe, p, workdir], stdin=reader.stdout,
                                      start_new_session=True)
            reader.stdout.close()
            if writer.wait() != 0:
                raise RuntimeError(f"Conversione passata {p} fallita")
            if reader.wait() != 0:
                raise RuntimeError("Lettura archivio fallita: risultato non valido")
        finally:
            for process in (writer, reader):
                if process is not None:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
        target = ip if p == "1" else ic
        os.replace(target + ".part", target)
        print(f"passata {p}: {time.perf_counter() - t0:.0f} s")
    with open(marker + ".part", "w") as f:
        json.dump(dict(n=n, m=m, converter="ram-follow-v1", orientation="follower_to_followed"), f)
    os.replace(marker + ".part", marker)
    return ip, ic


def run_gap(args):
    """GAP-twitter: tempo, passi e coerenza fra semi diversi (nessun riferimento esatto)."""
    xp = get_xp(not args.cpu)
    ip, ic = build_gap_out_lists(args.workdir)
    indptr = np.memmap(ip, dtype=np.int64, mode="r")
    indices = np.memmap(ic, dtype=np.int32, mode="r")
    n, m = indptr.size - 1, indices.size
    outdeg_zero = int(np.count_nonzero(np.diff(indptr) == 0))
    print(f"GAP-twitter: n = {n:,}  m = {m:,}  senza link uscenti = {outdeg_zero:,}")

    t0 = time.perf_counter()
    d_indptr = to_device(indptr, xp, np.int64)
    d_indices = to_device(indices, xp, np.int32)
    sync(xp)
    print(f"caricamento su {device_name(xp)}: {time.perf_counter() - t0:.1f} s")
    if xp is not np:
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"memoria GPU libera: {free / 2**30:.2f} GiB su {total / 2**30:.2f}")

    pagerank_montecarlo(d_indptr, d_indices, n, min(1_000_000, args.batch),
                        args.alpha, seed=12345, batch=args.batch, xp=xp)
    sync(xp)

    rows = []
    for N in args.N:
        estimates = []
        for seed in args.seeds:
            with Timer(xp) as tm:
                r_hat, steps = pagerank_montecarlo(d_indptr, d_indices, n, N,
                                                   args.alpha, seed=seed,
                                                   batch=args.batch, xp=xp)
            estimates.append(r_hat)
            rows.append(dict(N=N, seed=seed, seconds=tm.seconds,
                             steps_per_walk=steps / N,
                             steps_per_second=steps / tm.seconds,
                             sum=float(r_hat.sum())))
            print(f"N = {N:,}  seme {seed}: {tm.seconds:.1f} s, "
                  f"{steps / N:.3f} passi/cammino, {steps / tm.seconds:.3e} passi/s")
        if len(estimates) >= 2:
            # due stime indipendenti: la loro distanza misura la dispersione
            d = float(xp.abs(estimates[0] - estimates[1]).sum())
            rows[-1]["l1_fra_due_semi"] = d
            print(f"   distanza L1 fra le stime di due semi: {d:.3e}")
        r0 = estimates[0].get() if xp is not np else estimates[0]
        top = np.argpartition(-r0, 10)[:10]          # su CPU: evita un ordinamento
        top = top[np.argsort(-r0[top])]              # completo in memoria GPU
        print(f"   top-10 (indici 0-based del file): {top.tolist()}")
        del r0
        del estimates
        if xp is not np:
            cp.get_default_memory_pool().free_all_blocks()

    out = dict(grafo="GAP-twitter", orientation="follower_to_followed", n=int(n), m=int(m), alpha=args.alpha,
               device=device_name(xp), batch=args.batch, prove=rows)
    save_results(out, args, "montecarlo_gap_follow")


# =============================================================================

def main():
    p = argparse.ArgumentParser(description="PageRank Monte Carlo su GPU")
    p.add_argument("grafo", choices=["test", "twitter", "gap"])
    p.add_argument("--N", type=lambda s: [int(float(x)) for x in s.split(",")],
                   help="numeri di cammini separati da virgole, es. 1e5,1e6,1e7")
    p.add_argument("--seeds", type=lambda s: [int(x) for x in s.split(",")],
                   default=[1, 2, 3])
    p.add_argument("--alpha", type=float, default=0.85)
    p.add_argument("--batch", type=int, default=10_000_000,
                   help="cammini simulati insieme (limita la memoria)")
    p.add_argument("--workdir", default="/content" if os.path.isdir("/content") else ".")
    p.add_argument("--cpu", action="store_true", help="usa numpy invece di CuPy (solo prove)")
    args = p.parse_args()

    if args.grafo == "twitter" and args.N is None:
        args.N = [10**6, 10**7, 10**8]
    if args.grafo == "gap":
        args.N = args.N or [10**8, 10**9]
        if args.seeds == [1, 2, 3]:
            args.seeds = [1, 2]

    {"test": run_test, "twitter": run_twitter, "gap": run_gap}[args.grafo](args)


if __name__ == "__main__":
    main()


In [ ]:
if RUN_CPU_CHECK:
    import importlib.util
    import scipy.sparse as sparse
    spec = importlib.util.spec_from_file_location('thesis_mc', SCRIPT_DIR / 'montecarlo.py')
    mc = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mc)
    # Directed graph with a dangling node and a closed cycle before teleportation.
    src = np.array([0, 0, 1, 1, 2, 4, 5, 6])
    dst = np.array([1, 3, 2, 4, 0, 5, 6, 4])
    n_toy, alpha_toy = 7, 0.85
    rank, iterations, delta = mc.pagerank_power_reference(src, dst, n_toy, alpha_toy)
    degrees = np.bincount(src, minlength=n_toy)
    transition = np.zeros((n_toy, n_toy))
    transition[src, dst] = 1.0 / degrees[src]
    transition[degrees == 0] = 1.0 / n_toy
    exact = np.linalg.solve(np.eye(n_toy) - alpha_toy * transition.T,
                            np.full(n_toy, (1.0 - alpha_toy) / n_toy))
    np.testing.assert_allclose(rank, exact, atol=1e-11, rtol=0)
    ptr, indices = mc.out_lists(src, dst, n_toy)
    estimate, steps = mc.pagerank_montecarlo(ptr, indices, n_toy, 200_000,
                                            alpha_toy, seed=2026, batch=50_000, xp=np)
    assert np.isfinite(estimate).all() and (estimate >= 0).all()
    np.testing.assert_allclose(estimate.sum(), 1.0, atol=1e-14)
    error = float(np.abs(estimate - exact).sum())
    assert error < 0.025, error  # generous deterministic smoke-check threshold
    print(f'CPU check passed: power L1 error={np.abs(rank-exact).sum():.3g}; MC L1 error={error:.4g}')


<a id="web-google"></a>
## 4. web-Google: CPU/GPU power iteration and convergence

The benchmark retains the original float32 weights, float64 stopping reduction,
seven repetitions, full warm-up and randomised GPU-variant order. It measures wall
time to stopping, including convergence checks and excluding data preparation,
upload and validation. The CPU baseline is also repeated seven times.

GPU variants: mask-based dangling sum; dot-product dangling sum; checking every
five iterations; and incorporating alpha in the weights. The optional kernel
comparison runs in this same prepared process, as required by its original code.
The convergence figures use CPU float64, independently of the timing experiment.


In [ ]:
%%writefile _pagerank_runtime/scripts/web_benchmark.py
import sys
import numpy as np
import scipy
import cupy as cp

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("CuPy:", cp.__version__)
print("GPU disponibili:", cp.cuda.runtime.getDeviceCount())

from pathlib import Path
import urllib.request
import time
import scipy.sparse as sp

cartella = Path.cwd()
archivio = cartella / "web-Google.txt.gz"

if not archivio.exists():
    urllib.request.urlretrieve(
        "https://snap.stanford.edu/data/web-Google.txt.gz",
        archivio
    )

# Prima colonna: sorgente; seconda: destinazione.
archi = np.loadtxt(archivio, dtype=np.int64, comments="#")
etichette, indici = np.unique(archi, return_inverse=True)
src, dst = indici.reshape(archi.shape).T
n = len(etichette)
m = len(src)

gradi = np.bincount(src, minlength=n)
penzolanti = gradi == 0

# Trasposta della matrice dei link: r_nuovo = alpha * Pt @ r + b.
pesi64 = 1.0 / gradi[src].astype(np.float64)
Pt64 = sp.csr_matrix(
    (pesi64, (dst, src)), shape=(n, n), dtype=np.float64
)
Pt64.sort_indices()

print(f"Nodi: {n:,}; archi: {m:,}; penzolanti: {penzolanti.sum():,}")

alpha = 0.85
r_ref = np.full(n, 1.0 / n, dtype=np.float64)
inizio = time.perf_counter()

for t in range(1, 5001):
    massa = r_ref[penzolanti].sum()
    nuovo = alpha * (Pt64 @ r_ref)
    nuovo += (1.0 - alpha + alpha * massa) / n
    delta = np.abs(nuovo - r_ref).sum()
    r_ref = nuovo
    if delta < 1e-12:
        break
else:
    raise RuntimeError("Il riferimento non ha raggiunto la soglia.")

tempo_ref = time.perf_counter() - inizio
controllo = alpha * (Pt64 @ r_ref)
controllo += (1.0 - alpha + alpha * r_ref[penzolanti].sum()) / n
residuo_ref = np.abs(controllo - r_ref).sum()

np.save(cartella / "web_google_riferimento_float64.npy", r_ref)

print(f"Riferimento: {t} iterazioni, {tempo_ref:.3f} s")
print(f"Delta finale: {delta:.3e}")
print(f"Residuo: {residuo_ref:.3e}")
print(f"Somma dei punteggi: {r_ref.sum():.12f}")

import cupyx.scipy.sparse as csp

# Pesi calcolati direttamente in float32, uguali per tutte le varianti.
pesi32 = np.float32(1.0) / gradi[src].astype(np.float32)
Pt32 = sp.csr_matrix(
    (pesi32, (dst, src)), shape=(n, n), dtype=np.float32
)
Pt32.sort_indices()

Pt_gpu = csp.csr_matrix(Pt32)
mask_gpu = cp.asarray(penzolanti)
a_gpu = mask_gpu.astype(cp.float32)

# Quarta variante: alpha incorporato nei pesi, una sola volta.
Pt_alpha_gpu = Pt_gpu.copy()
Pt_alpha_gpu.data *= cp.float32(alpha)

alpha32 = cp.float32(alpha)
n32 = cp.float32(n)
teletrasporto32 = cp.float32(1.0 - alpha)

def aggiornamento(r, variante):
    if variante == "base":
        massa = r[mask_gpu].sum()
    else:
        massa = cp.dot(r, a_gpu)

    b = (teletrasporto32 + alpha32 * massa) / n32

    if variante == "alpha_nei_pesi":
        nuovo = Pt_alpha_gpu @ r
    else:
        nuovo = Pt_gpu @ r
        nuovo *= alpha32

    nuovo += b
    return nuovo

varianti = [
    ("base", 1),
    ("dot", 1),
    ("controllo_5", 5),
    ("alpha_nei_pesi", 5),
]

# Verifica dell'aggiornamento contro la CPU float64.
r0_cpu = np.full(n, 1.0 / n, dtype=np.float64)
atteso = alpha * (Pt64 @ r0_cpu)
atteso += (
    1.0 - alpha + alpha * r0_cpu[penzolanti].sum()
) / n

for nome, frequenza in varianti:
    r0_gpu = cp.full(n, 1.0 / n, dtype=cp.float32)
    risultato = cp.asnumpy(aggiornamento(r0_gpu, nome))

    assert np.isfinite(risultato).all()
    assert risultato.min() >= 0
    np.testing.assert_allclose(
        risultato, atteso, rtol=2e-5, atol=1e-10
    )

    print(
        f"{nome}: test superato; "
        f"scarto L1 = {np.abs(risultato - atteso).sum():.3e}"
    )

import json
import platform

soglia = 1e-6
max_iter = 1000
ripetizioni = 7

def esegui(nome, frequenza):
    # Allocazione e inizializzazione escluse dal tempo.
    r = cp.full(n, 1.0 / n, dtype=cp.float32)
    cp.cuda.Stream.null.synchronize()
    inizio = time.perf_counter()

    for t in range(1, max_iter + 1):
        nuovo = aggiornamento(r, nome)

        if t % frequenza == 0 or t == max_iter:
            delta = float(cp.abs(nuovo - r).sum(dtype=cp.float64))
            if not np.isfinite(delta):
                raise RuntimeError(f"{nome}: incremento non finito")
            if delta < soglia:
                r = nuovo
                break

        r = nuovo
    else:
        raise RuntimeError(f"{nome}: soglia non raggiunta")

    cp.cuda.Stream.null.synchronize()
    secondi = time.perf_counter() - inizio
    return r, t, delta, secondi

# Riscaldamento completo di ciascuna variante.
for nome, frequenza in varianti:
    caldo, *_ = esegui(nome, frequenza)
    del caldo

risultati = []
ordine_rng = np.random.default_rng(2026)

for prova in range(1, ripetizioni + 1):
    # Alterniamo l'ordine per non favorire sempre la stessa variante.
    for indice in ordine_rng.permutation(len(varianti)):
        nome, frequenza = varianti[indice]
        r_gpu, t, delta, secondi = esegui(nome, frequenza)

        # Controlli fuori dal tempo misurato.
        r_cpu = cp.asnumpy(r_gpu).astype(np.float64)
        assert np.isfinite(r_cpu).all() and r_cpu.min() >= 0

        controllo = alpha * (Pt64 @ r_cpu)
        controllo += (
            1.0 - alpha + alpha * r_cpu[penzolanti].sum()
        ) / n

        risultati.append({
            "variante": nome,
            "ripetizione": prova,
            "controllo_ogni": frequenza,
            "iterazioni": t,
            "seconds": secondi,
            "ms_per_iterazione": 1000 * secondi / t,
            "delta": delta,
            "errore_l1_riferimento": float(np.abs(r_cpu - r_ref).sum()),
            "residuo_float64": float(np.abs(controllo - r_cpu).sum()),
            "somma": float(r_cpu.sum()),
        })
        del r_gpu

    print(f"Ripetizione {prova}/{ripetizioni} completata", flush=True)

print("\nVariante          iterazioni   tempo mediano [min, max] ms   errore L1 massimo")
for nome, _ in varianti:
    righe = [x for x in risultati if x["variante"] == nome]
    tempi = np.array([x["seconds"] for x in righe]) * 1000
    it = sorted({x["iterazioni"] for x in righe})
    errore = max(x["errore_l1_riferimento"] for x in righe)
    print(
        f"{nome:18s} {str(it):12s} "
        f"{np.median(tempi):8.2f} [{tempi.min():.2f}, {tempi.max():.2f}]"
        f"   {errore:.3e}"
    )

gpu_nome = cp.cuda.runtime.getDeviceProperties(0)["name"]
if isinstance(gpu_nome, bytes):
    gpu_nome = gpu_nome.decode()

documento = {
    "dataset": "web-Google",
    "n": n,
    "m": m,
    "alpha": alpha,
    "soglia_incremento": soglia,
    "dtype": "float32",
    "riduzione_delta": "float64",
    "gpu": gpu_nome,
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "cupy": cp.__version__,
    "protocollo": (
        "Tempo a parete sincronizzato fino all'arresto, inclusi controlli; "
        "esclusi preparazione, trasferimenti, inizializzazione e validazione. "
        "Riscaldamento completo; ordine casuale delle varianti."
    ),
    "prove": risultati,
}

percorso = cartella / "benchmark_web_google_gpu.json"
percorso.write_text(json.dumps(documento, indent=2))
print("\nSalvato:", percorso)

def esegui_cpu():
    r = np.full(n, 1.0 / n, dtype=np.float32)
    inizio = time.perf_counter()

    for t in range(1, max_iter + 1):
        massa = r[penzolanti].sum()
        nuovo = Pt32 @ r
        nuovo *= np.float32(alpha)
        nuovo += (
            np.float32(1.0 - alpha) + np.float32(alpha) * massa
        ) / np.float32(n)

        delta = float(np.abs(nuovo - r).sum(dtype=np.float64))
        r = nuovo
        if not np.isfinite(delta):
            raise RuntimeError("Incremento CPU non finito")
        if delta < soglia:
            break
    else:
        raise RuntimeError("CPU: soglia non raggiunta")

    secondi = time.perf_counter() - inizio
    return r, t, delta, secondi

# Riscaldamento, escluso dalle misure.
esegui_cpu()

prove_cpu = []
for prova in range(1, 8):
    r_cpu, t, delta, secondi = esegui_cpu()

    # Validazione esclusa dal tempo.
    assert np.isfinite(r_cpu).all() and r_cpu.min() >= 0
    errore = float(np.abs(r_cpu.astype(np.float64) - r_ref).sum())

    prove_cpu.append({
        "ripetizione": prova,
        "seconds": secondi,
        "iterazioni": t,
        "delta": delta,
        "errore_l1_riferimento": errore,
        "somma": float(r_cpu.sum(dtype=np.float64)),
    })
    print(f"CPU {prova}/7: {secondi:.3f} s, {t} iterazioni")

tempi_cpu = np.array([x["seconds"] for x in prove_cpu])
mediana_cpu = float(np.median(tempi_cpu))

print(
    f"\nCPU: mediana {mediana_cpu:.3f} s "
    f"[{tempi_cpu.min():.3f}, {tempi_cpu.max():.3f}]"
)
print(
    "Scarto L1 massimo:",
    f"{max(x['errore_l1_riferimento'] for x in prove_cpu):.3e}"
)

for nome, _ in varianti:
    tempi_gpu = [
        x["seconds"] for x in risultati if x["variante"] == nome
    ]
    rapporto = mediana_cpu / np.median(tempi_gpu)
    print(f"CPU base / GPU {nome}: {rapporto:.2f}×")

# Registra anche il modello della CPU.
modello_cpu = "non rilevato"
cpu_info = Path("/proc/cpuinfo")
for riga in (cpu_info.read_text().splitlines() if cpu_info.exists() else []):
    if riga.startswith("model name"):
        modello_cpu = riga.split(":", 1)[1].strip()
        break

documento["cpu"] = {
    "modello": modello_cpu,
    "dtype": "float32",
    "controllo_ogni": 1,
    "prove": prove_cpu,
}
percorso.write_text(json.dumps(documento, indent=2))
print("\nJSON aggiornato:", percorso)
import os
if os.environ.get('THESIS_WEB_KERNELS') == '1':
    from benchmark_kernel_web import kernel_experiment
    kernel_experiment(Pt_gpu, Pt32, r_ref, penzolanti, alpha)


In [ ]:
%%writefile _pagerank_runtime/scripts/benchmark_kernel_web.py
"""Eseguire con %run -i nel notebook web-Google dopo il benchmark esistente.
CSR-Scalar / CSR-Vector: Bell & Garland, NVR-2008-004, sezione 4.3.
Implementazione propria con shuffle sincronizzati per la riduzione di warp.
Solo SpMV: alpha e termine uniforme restano operazioni separate e identiche.
"""
import json
import time
from pathlib import Path
import numpy as np
import cupy as cp
import scipy.sparse as sps
import cupyx.scipy.sparse as cps

CUDA_CSR = r'''
extern "C" __global__ void csr_scalar(
    int n, const int* ptr, const int* col, const float* val,
    const float* x, float* y) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;
    if (row >= n) return;
    float sum = 0.0f;
    for (int k = ptr[row]; k < ptr[row+1]; ++k)
        sum += val[k] * x[col[k]];
    y[row] = sum;
}
extern "C" __global__ void csr_vector(
    int n, const int* ptr, const int* col, const float* val,
    const float* x, float* y) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int row = tid / 32;
    int lane = tid % 32;
    if (row >= n) return; // Tutto il warp esce insieme.
    float sum = 0.0f;
    for (int k = ptr[row] + lane; k < ptr[row+1]; k += 32)
        sum += val[k] * x[col[k]];
    for (int offset = 16; offset > 0; offset /= 2)
        sum += __shfl_down_sync(0xffffffff, sum, offset);
    if (lane == 0) y[row] = sum;
}
'''


def kernel_experiment(A, host_A, reference, dangling_mask, alpha=.85):
    assert A.dtype == cp.float32
    assert A.indptr.dtype == A.indices.dtype == cp.int32
    kernels = {name: cp.RawKernel(CUDA_CSR, name)
               for name in ('csr_scalar', 'csr_vector')}

    def product(name, matrix, x):
        if name == 'cupy':
            return matrix @ x
        rows = matrix.shape[0]
        out = cp.empty(rows, dtype=cp.float32)
        blocks = (rows+255)//256 if name == 'csr_scalar' else (rows+7)//8
        kernels[name]((blocks,), (256,),
                      (np.int32(rows), matrix.indptr, matrix.indices,
                       matrix.data, x, out))
        return out

    # Test non multiplo della dimensione di blocco/warp: righe vuote, 31,
    # 32, 33 elementi, e una riga da 4097 elementi con piu' giri del warp.
    rng = np.random.default_rng(17)
    lengths = np.array([0,1,2,31,32,33,65,257,4097,0,7])
    ptr = np.r_[0,np.cumsum(lengths)].astype(np.int32)
    columns = np.concatenate([np.arange(k,dtype=np.int32) for k in lengths])
    values = rng.uniform(.01,1,len(columns)).astype(np.float32)
    small = sps.csr_matrix((values,columns,ptr),shape=(len(lengths),4097))
    small_gpu = cps.csr_matrix(small)
    for vector in (rng.random(4097).astype(np.float32),
                   rng.uniform(-1,1,4097).astype(np.float32)):
        expected = small.astype(np.float64) @ vector.astype(np.float64)
        scale = abs(small.astype(np.float64)) @ abs(vector.astype(np.float64))
        for name in kernels:
            got = cp.asnumpy(product(name,small_gpu,cp.asarray(vector)))
            assert np.all(np.isfinite(got))
            assert np.all(abs(got-expected) <= 1e-5*scale+1e-7), name
    print('Test GPU su righe vuote/corte/lunghe e vettori con segno: superati.',flush=True)

    nodes = A.shape[0]
    mask = cp.asarray(dangling_mask)
    for vector in (np.full(nodes,1/nodes,dtype=np.float32),
                   rng.random(nodes).astype(np.float32)/np.float32(nodes)):
        expected = host_A.astype(np.float64) @ vector.astype(np.float64)
        for name in ('cupy', *kernels):
            got = cp.asnumpy(product(name,A,cp.asarray(vector)))
            np.testing.assert_allclose(got,expected,rtol=5e-5,atol=1e-12)
    print('Confronto SpMV web-Google con SciPy float64: superato.',flush=True)

    def solve(name):
        r = cp.full(nodes,1/nodes,dtype=cp.float32)
        cp.cuda.Stream.null.synchronize()
        start = time.perf_counter()
        for iteration in range(1,1001):
            mass = r[mask].sum()
            b = (cp.float32(1-alpha)+cp.float32(alpha)*mass)/cp.float32(nodes)
            nxt = product(name,A,r)
            nxt *= cp.float32(alpha)
            nxt += b
            delta = float(cp.abs(nxt-r).sum(dtype=cp.float64))
            if not np.isfinite(delta): raise RuntimeError('Delta non finito')
            r = nxt
            if delta < 1e-6: break
        else:
            raise RuntimeError(f'{name}: soglia non raggiunta')
        cp.cuda.Stream.null.synchronize()
        seconds = time.perf_counter()-start
        return r, iteration, delta, seconds

    names = ['cupy','csr_scalar','csr_vector']
    for name in names:
        warm, *_ = solve(name)
        del warm
    rows = []
    order = np.random.default_rng(2026)
    for repetition in range(1,8):
        for index in order.permutation(3):
            name = names[index]
            rank, iterations, delta, seconds = solve(name)
            host = cp.asnumpy(rank).astype(np.float64)
            assert np.isfinite(host).all() and host.min() >= 0
            error = float(np.abs(host-reference).sum())
            assert error < 1e-5, (name,error)
            rows.append(dict(variant=name,repetition=repetition,
                             seconds=seconds,iterations=iterations,delta=delta,
                             error_l1_reference=error,sum=float(host.sum())))
            del rank
        print(f'Ripetizione {repetition}/7 completata',flush=True)
    for name in names:
        group = [r for r in rows if r['variant']==name]
        times = np.array([r['seconds'] for r in group])*1000
        print(f"{name:12s}: {np.median(times):.2f} ms "
              f"[{times.min():.2f}, {times.max():.2f}], "
              f"iterazioni {sorted({r['iterations'] for r in group})}, "
              f"scarto L1 max {max(r['error_l1_reference'] for r in group):.3e}")
    props = cp.cuda.runtime.getDeviceProperties(0)
    gpu = props['name']
    if isinstance(gpu,bytes): gpu=gpu.decode()
    output = dict(dataset='web-Google',n=nodes,m=A.nnz,alpha=alpha,
                  dtype='float32',gpu=gpu,cupy=cp.__version__,block_threads=256,
                  tolerance=1e-6,check_every=1,dangling='mask',
                  protocol='Wall time synchronized; initialization, compilation, warmup and validation excluded; 7 randomized repetitions',
                  source='https://www.nvidia.com/docs/io/66889/nvr-2008-004.pdf section 4.3',
                  tests_passed=True,trials=rows)
    path=Path('./benchmark_kernel_web.json')
    path.write_text(json.dumps(output,indent=2))
    print('Salvato:',path,flush=True)


if __name__ == '__main__':
    required = ['Pt_gpu','Pt32','r_ref','penzolanti','alpha']
    if any(name not in globals() for name in required):
        raise RuntimeError('Usare %run -i ./benchmark_kernel_web.py nel notebook gia preparato')
    kernel_experiment(Pt_gpu,Pt32,r_ref,penzolanti,alpha)


In [ ]:
%%writefile _pagerank_runtime/scripts/figure_convergenza.py
"""
Figure della convergenza della power iteration al variare di alpha (Capitolo 5).

Da eseguire su Colab con runtime T4 (stessa configurazione della Tabella 5.1).
Il calcolo e' solo su CPU, in float64, sul grafo web-Google.

Uso in una cella di Colab:
    !wget -q https://snap.stanford.edu/data/web-Google.txt.gz
    !gunzip -f web-Google.txt.gz
    !python -u figure_convergenza_colab.py

Produce:
    fig_convergenza.pdf / .png   errore ||r(t) - r||_1 in funzione di t
    fig_fattore.pdf / .png       fattore di riduzione misurato contro alpha
    convergenza.json             tutti i numeri usati nelle figure

Notazione come nella tesi: r(t) iterati, r PageRank (qui: riferimento
calcolato con molte iterazioni in float64), t indice di iterazione.
"""

import json
import math
import platform
import subprocess
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp

GRAPH_PATH = sys.argv[1] if len(sys.argv) > 1 else "web-Google.txt"

ALPHAS_FIG1 = [0.50, 0.70, 0.85, 0.95]
ALPHAS_FIG2 = [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.99]
T_MAX = 150          # iterazioni mostrate e usate per la regressione
FLOOR = 1e-13        # la curva si interrompe al primo valore sotto questa soglia:
                     # piu' in basso contano gli arrotondamenti di float64
FIT_START = 10       # si scartano le prime iterazioni (transitorio)
FIT_STOP = 1e-11     # la regressione usa solo errori sopra questa soglia
REF_TARGET = 1e-17   # il riferimento fa almeno log(REF_TARGET)/log(alpha) iterazioni
REF_MIN = 300


# ------------------------------------------------------------------ grafo

def load_graph(path):
    """Lista di archi SNAP (righe 'sorgente destinazione', commenti con #).
    Gli identificatori vengono rinumerati in 0..n-1."""
    e = pd.read_csv(path, sep=r"\s+", comment="#", header=None,
                    names=["s", "d"], dtype=np.int64, engine="c").to_numpy()
    ids, inv = np.unique(e.ravel(), return_inverse=True)
    inv = inv.reshape(e.shape).astype(np.int64)
    src, dst = inv[:, 0], inv[:, 1]
    n = len(ids)
    distinct = len(np.unique(src * n + dst))
    selfloops = int(np.count_nonzero(src == dst))
    return src, dst, n, distinct, selfloops


def transition_transposed(src, dst, n):
    """Matrice con elemento (j, i) = 1/d_i per ogni arco i -> j (trasposta di P).
    Eventuali archi ripetuti verrebbero sommati: il numero di archi distinti
    viene stampato per controllo."""
    outdeg = np.bincount(src, minlength=n)
    w = 1.0 / outdeg[src]
    M = sp.csr_matrix((w, (dst, src)), shape=(n, n), dtype=np.float64)
    return M, outdeg == 0


def step(M, dangling, r, alpha, n):
    massa = r[dangling].sum()
    return alpha * (M @ r) + (alpha * massa + (1.0 - alpha)) / n


def reference(M, dangling, alpha, n):
    """Molte iterazioni in float64 a partire dal vettore uniforme.
    Restituisce anche l'ultimo incremento e la stima della Sezione 4.3,
    alpha/(1-alpha) * incremento (valida in aritmetica esatta)."""
    iters = max(REF_MIN, math.ceil(math.log(REF_TARGET) / math.log(alpha)))
    r = np.full(n, 1.0 / n)
    delta = float("nan")
    for _ in range(iters):
        r_new = step(M, dangling, r, alpha, n)
        delta = float(np.abs(r_new - r).sum())
        r = r_new
    return r, iters, delta, alpha / (1 - alpha) * delta


def history(M, dangling, alpha, n, r_ref):
    r = np.full(n, 1.0 / n)
    h = np.empty(T_MAX + 1)
    h[0] = np.abs(r - r_ref).sum()          # h[t] = ||r(t) - r||_1, t = 0..T_MAX
    for t in range(1, T_MAX + 1):
        r = step(M, dangling, r, alpha, n)
        h[t] = np.abs(r - r_ref).sum()
    return h


def fit_factor(h):
    """Pendenza di log(errore) fra t = FIT_START e l'ultimo t con errore
    sopra FIT_STOP; il fattore di riduzione per iterazione e' exp(pendenza)."""
    t = np.arange(len(h))
    ok = (t >= FIT_START) & (h > FIT_STOP)
    # solo il tratto iniziale contiguo sopra la soglia
    last = np.argmax(h <= FIT_STOP) if (h <= FIT_STOP).any() else len(h)
    ok &= t <= last
    slope = np.polyfit(t[ok], np.log(h[ok]), 1)[0]
    return math.exp(slope), int(t[ok][0]), int(t[ok][-1])


# ----------------------------------------------------------------- stile

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["cmr10", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True,
    "axes.unicode_minus": False,
    "font.size": 10,
    "legend.fontsize": 9,
    "pdf.fonttype": 42,
})
virgola = FuncFormatter(lambda x, _: f"{x:.2f}".replace(".", ","))


def env_info():
    info = dict(python=platform.python_version(), numpy=np.__version__,
                scipy=scipy.__version__, pandas=pd.__version__,
                matplotlib=matplotlib.__version__)
    try:
        with open("/proc/cpuinfo") as f:
            info["cpu"] = next(l.split(":", 1)[1].strip()
                               for l in f if l.startswith("model name"))
        with open("/proc/meminfo") as f:
            kb = int(next(l.split()[1] for l in f if l.startswith("MemTotal")))
        info["ram_GB"] = round(kb * 1024 / 1e9, 2)
    except Exception:
        pass
    try:
        info["gpu"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception:
        info["gpu"] = None
    return info


# ------------------------------------------------------------------ main

def main():
    env = env_info()
    print("Ambiente:", env, flush=True)

    t0 = time.perf_counter()
    src, dst, n, distinct, selfloops = load_graph(GRAPH_PATH)
    M, dangling = transition_transposed(src, dst, n)
    print(f"Grafo: n={n:,}  righe={len(src):,}  archi distinti={distinct:,}  "
          f"autoanelli={selfloops}  penzolanti={int(dangling.sum()):,}  "
          f"({time.perf_counter()-t0:.1f} s)", flush=True)

    alphas = sorted(set(ALPHAS_FIG1) | set(ALPHAS_FIG2))
    res = {}
    for a in alphas:
        t0 = time.perf_counter()
        r_ref, it_ref, d_ref, bound = reference(M, dangling, a, n)
        h = history(M, dangling, a, n, r_ref)
        rho, lo, hi = fit_factor(h)
        below = np.flatnonzero(h < 1e-8)
        res[a] = dict(h=h, rho=rho, fit=(lo, hi), ref_iters=it_ref,
                      ref_delta=d_ref, ref_bound=bound,
                      t_1e8=int(below[0]) if below.size else None)
        cut = np.flatnonzero(h <= FLOOR)
        res[a]["cut"] = int(cut[0]) if cut.size else None
        # controllo della Sezione 4.3: ||r(t) - r|| <= alpha^t ||r(0) - r||
        t_all = np.arange(T_MAX + 1)
        res[a]["bound_ok"] = bool(np.all(h <= h[0] * a ** t_all * (1 + 1e-9) + 1e-15))
        print(f"alpha={a:.2f}  curva interrotta a t={res[a]['cut']}  riferimento: {it_ref} iterazioni, ultimo "
              f"incremento {d_ref:.1e}, stima 4.3 {bound:.1e} | fattore misurato "
              f"{rho:.4f} (t da {lo} a {hi}) | errore < 1e-8 a t={res[a]['t_1e8']} "
              f"| stima alpha^t rispettata: {res[a]['bound_ok']} "
              f"| {time.perf_counter()-t0:.1f} s", flush=True)

    # ---------------------------------------------- Figura: convergenza
    fig, ax = plt.subplots(figsize=(5.6, 3.9))
    colors = plt.cm.viridis(np.linspace(0, 0.85, len(ALPHAS_FIG1)))
    for a, c in zip(ALPHAS_FIG1, colors):
        h = res[a]["h"]
        t = np.arange(T_MAX + 1)
        stop = np.argmax(h <= FLOOR) if (h <= FLOOR).any() else T_MAX + 1
        ax.semilogy(t[:stop], h[:stop], color=c, lw=1.6,
                    label=rf"$\alpha$ = {a:.2f}".replace(".", ","))
        # stima della Sezione 4.3: ||r(t) - r||_1 <= alpha^t ||r(0) - r||_1
        ax.semilogy(t, h[0] * a ** t, color=c, ls="--", lw=0.9, alpha=0.7)
    ax.plot([], [], color="grey", ls="--", lw=0.9,
            label=r"$\alpha^t\,\|\bar{r}(0) - \bar{r}\|_1$")
    ax.set_ylim(FLOOR / 10, 2.0)
    ax.set_xlim(0, T_MAX)
    ax.set_xlabel(r"iterazione $t$")
    ax.set_ylabel(r"$\|\bar{r}(t) - \bar{r}\|_1$")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3,
              frameon=False, handlelength=1.8, columnspacing=1.2)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("fig_convergenza.pdf", bbox_inches="tight")
    fig.savefig("fig_convergenza.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    # ------------------------------------- Figura: fattore di riduzione
    fig, ax = plt.subplots(figsize=(5.0, 3.9))
    grid = np.linspace(0.0, 1.0, 200)
    ax.plot(grid, grid, "k--", lw=1.0, label=r"$\alpha$")
    ax.plot(ALPHAS_FIG2, [res[a]["rho"] for a in ALPHAS_FIG2], "o",
            color="tab:red", ms=5, label="fattore misurato")
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel("fattore di riduzione per iterazione")
    ax.xaxis.set_major_formatter(virgola)
    ax.yaxis.set_major_formatter(virgola)
    lo = min(0.45, min(res[a]["rho"] for a in ALPHAS_FIG2) - 0.05)
    ax.set_xlim(lo, 1.0)
    ax.set_ylim(lo, 1.0)
    ax.set_aspect("equal")
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig("fig_fattore.pdf", bbox_inches="tight")
    fig.savefig("fig_fattore.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    out = dict(
        graph=GRAPH_PATH, n=n, rows=len(src), distinct_arcs=distinct,
        self_loops=selfloops, dangling=int(dangling.sum()), dtype="float64",
        start="uniforme 1/n", T_MAX=T_MAX, floor=FLOOR,
        errore_indice="errore[t] = norma 1 di r(t) - r, per t = 0..T_MAX",
        fit=f"regressione di log errore per t >= {FIT_START} ed errore > {FIT_STOP}",
        environment=env,
        alphas={f"{a:.2f}": dict(
            fattore_misurato=res[a]["rho"], tratto_regressione=res[a]["fit"],
            riferimento_iterazioni=res[a]["ref_iters"],
            riferimento_ultimo_incremento=res[a]["ref_delta"],
            riferimento_stima_4_3=res[a]["ref_bound"],
            iterazioni_errore_sotto_1e8=res[a]["t_1e8"],
            curva_interrotta_a_t=res[a]["cut"],
            stima_alpha_t_rispettata=res[a]["bound_ok"],
            errore=res[a]["h"].tolist()) for a in alphas},
    )
    with open("convergenza.json", "w") as f:
        json.dump(out, f, indent=1)
    print("Salvati: fig_convergenza.pdf/.png, fig_fattore.pdf/.png, convergenza.json")


if __name__ == "__main__":
    main()


<a id="twitter"></a>
## 5. twitter_combined: Monte Carlo accuracy

Uses the Monte Carlo implementation in Section 3, with $N=10^6,10^7,10^8$,
seeds 1, 2, 3 and batches of one million walks. The reference is computed by CPU
power iteration in float64. Graph edges are deduplicated, self-loops retained,
and original node identifiers mapped to contiguous indices.
The source-to-destination order is already follower → followed in this dataset.


<a id="gap"></a>
## 6. GAP-twitter: graph preparation, power iteration and kernels

The Matrix Market file stores the reverse of the intended follower → followed
relation. Both orientations are built explicitly:

* `follow_*`: outgoing lists in the follower → followed direction (Monte Carlo).
* `out_*`: incoming rows for that same direction (transposed adjacency for SpMV).

Artificial edge weights are ignored. The node index space is retained, including
isolated identifiers. Power iteration and Monte Carlo use the same graph semantics.
The unweighted vector kernel reads indices but does not store one float per edge;
the outgoing-degree scaling is applied to the score vector before SpMV.

Run preparation before any GAP experiment, unless the generated files are already
present in `pagerank_runs/gap/`. `RUN_GAP_VALIDATION` additionally needs the rank
vector produced by `RUN_GAP_POWER`. The full MC comparison includes up to one
billion walks. Degree checks validate necessary orientation invariants; they are
not, by themselves, a proof of equality of the entire edge sets.


In [ ]:
%%writefile _pagerank_runtime/scripts/prepara_gap_kernel.py
"""Ricostruisce solo i file GAP nei due versi; non simula cammini."""
import os, json, subprocess, tarfile, time, signal
import numpy as np
import montecarlo as mc
CONV_OUT_C = r"""
/* Converte GAP-twitter.mtx (letto da stdin) in liste USCENTI:
 *   passata 1: conta i link uscenti di ogni sorgente i e scrive out_indptr.bin
 *   passata 2: scrive la destinazione j-1 di ogni arco nella posizione di i
 * GAP: seconda coordinata = follower sorgente, prima = utente seguito.
 */
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <inttypes.h>

#define CHECK(x) do { if (!(x)) { fprintf(stderr, "errore: %s\n", #x); return 2; } } while (0)

static long read_header(char *line, int size, long *nnz) {
    long nr = 0, nc = 0;
    while (fgets(line, size, stdin)) {
        if (line[0] == '%') continue;
        sscanf(line, "%ld %ld %ld", &nr, &nc, nnz);
        break;
    }
    return nr;
}

static int parse(char *p, long *i, long *j) {
    *i = 0; *j = 0;
    while (*p == ' ' || *p == '\t') p++;
    if (*p < '0' || *p > '9') return 0;
    while (*p >= '0' && *p <= '9') *i = *i * 10 + (*p++ - '0');
    while (*p == ' ' || *p == '\t') p++;
    while (*p >= '0' && *p <= '9') *j = *j * 10 + (*p++ - '0');
    /* Mantieni prima coordinata -> seconda: CSR entrante del verso desiderato. */
    return 1;
}

int main(int argc, char **argv) {
    if (argc < 3) { fprintf(stderr, "uso: conv_out 1|2 cartella\n"); return 1; }
    int pass = atoi(argv[1]);
    const char *dir = argv[2];
    static char iob[1 << 24];
    setvbuf(stdin, iob, _IOFBF, sizeof iob);
    char line[256], path[1024];
    long nnz = 0;
    long n = read_header(line, sizeof line, &nnz);
    CHECK(n > 0 && n <= INT32_MAX && nnz >= 0);
    fprintf(stderr, "n=%ld  nnz=%ld\n", n, nnz);

    if (pass == 1) {
        int64_t *deg = calloc(n + 1, sizeof(int64_t));
        CHECK(deg);
        long i, j, done = 0;
        while (fgets(line, sizeof line, stdin)) {
            if (!parse(line, &i, &j)) continue;
            CHECK(i >= 1 && i <= n && j >= 1 && j <= n);
            deg[i - 1]++;
            if (++done % 200000000 == 0) fprintf(stderr, "  %ld archi\n", done);
        }
        CHECK(!ferror(stdin) && done == nnz);
        int64_t *indptr = malloc(8LL * (n + 1));
        CHECK(indptr);
        indptr[0] = 0;
        for (long v = 0; v < n; v++) indptr[v + 1] = indptr[v] + deg[v];
        snprintf(path, sizeof path, "%s/out_indptr.bin.part", dir);
        FILE *f = fopen(path, "wb"); CHECK(f);
        CHECK(fwrite(indptr, 8, n + 1, f) == (size_t)(n + 1));
        CHECK(fclose(f) == 0);
        fprintf(stderr, "passata 1: %ld archi\n", done);
        free(deg); free(indptr);
    } else {
        int64_t *cur = malloc(8LL * (n + 1));
        int64_t *bounds = malloc(8LL * (n + 1));
        CHECK(cur && bounds);
        snprintf(path, sizeof path, "%s/out_indptr.bin", dir);
        FILE *f = fopen(path, "rb");
        CHECK(f);
        if (fread(cur, 8, n + 1, f) != (size_t)(n + 1)) { fprintf(stderr, "indptr corto\n"); return 2; }
        rewind(f);
        CHECK(fread(bounds, 8, n + 1, f) == (size_t)(n + 1));
        fclose(f);
        int64_t m = cur[n];
        CHECK(cur[0] == 0 && m == nnz);
        for (long v = 0; v < n; v++) CHECK(cur[v] >= 0 && cur[v] <= cur[v+1]);
        fprintf(stderr, "Costruzione in RAM: %.2f GiB (array e cursori)\n",
                (4.0*m + 16.0*(n+1))/(1024.0*1024*1024));
        int32_t *ind = malloc(m ? 4ULL * m : 1);
        CHECK(ind);
        long i, j, done = 0;
        while (fgets(line, sizeof line, stdin)) {
            if (!parse(line, &i, &j)) continue;
            CHECK(i >= 1 && i <= n && j >= 1 && j <= n);
            CHECK(cur[i - 1] < bounds[i]);
            ind[cur[i - 1]++] = (int32_t)(j - 1);
            if (++done % 200000000 == 0) fprintf(stderr, "  %ld archi\n", done);
        }
        CHECK(!ferror(stdin) && done == m);
        for (long v = 0; v < n; v++) CHECK(cur[v] == bounds[v+1]);
        fprintf(stderr, "Scrittura sequenziale su disco...\n");
        snprintf(path, sizeof path, "%s/out_indices.bin.part", dir);
        f = fopen(path, "wb"); CHECK(f);
        CHECK(fwrite(ind, 4, m, f) == (size_t)m);
        CHECK(fclose(f) == 0);
        free(ind); free(cur); free(bounds);
        fprintf(stderr, "passata 2: %ld archi\n", done);
    }
    return 0;
}
"""


def build_gap_out_lists(workdir):
    tar = os.path.join(workdir, "GAP-twitter.tar.gz")
    ip = os.path.join(workdir, "out_indptr.bin")
    ic = os.path.join(workdir, "out_indices.bin")
    marker = os.path.join(workdir, "out_complete.json")
    if os.path.exists(marker) and os.path.exists(ip) and os.path.exists(ic):
        with open(marker) as f:
            meta = json.load(f)
        if (meta.get("orientation") != "followed_to_follower"
                or meta.get("converter") != "ram-v1"):
            raise RuntimeError("Orientamento o versione dei file GAP non verificati")
        if (os.path.getsize(ip) != 8 * (meta["n"] + 1)
                or os.path.getsize(ic) != 4 * meta["m"]):
            raise RuntimeError("Dimensioni dei file convertiti non valide")
        print("liste uscenti gia' presenti, salto la conversione")
        return ip, ic
    if not os.path.exists(tar):
        print("scarico GAP-twitter (circa 9,5 GB)...")
        subprocess.run(["wget", "-c", "-q", "-O", tar,
                        "https://sparse.tamu.edu/MM/GAP/GAP-twitter.tar.gz"], check=True)
    # Legge soltanto l'intestazione; non estrae l'intero archivio.
    with tarfile.open(tar, "r|gz") as archive:
        for member in archive:
            if member.name.endswith("GAP-twitter.mtx"):
                with archive.extractfile(member) as stream:
                    for line in stream:
                        if line.startswith(b"%") or not line.strip():
                            continue
                        n, nc, m = map(int, line.split())
                        break
                break
        else:
            raise RuntimeError("GAP-twitter.mtx assente nell'archivio")
    if n != nc or n <= 0 or m < 0:
        raise RuntimeError("Intestazione del grafo non valida")

    # Recupera anche la prima passata prodotta dalla vecchia versione.
    first_ok = False
    if os.path.exists(ip) and os.path.getsize(ip) == 8 * (n + 1):
        ptr = np.memmap(ip, dtype=np.int64, mode="r")
        first_ok = int(ptr[0]) == 0 and int(ptr[-1]) == m
        for s in range(0, n, 1_000_000):
            e = min(n, s + 1_000_000)
            first_ok = first_ok and bool(np.all(ptr[s:e] <= ptr[s+1:e+1]))
        del ptr
    if first_ok:
        print("Prima passata valida: la riutilizzo.", flush=True)
    if os.path.exists(ic):
        print("File destinazioni senza conferma di completamento: verra' ricostruito.", flush=True)

    if os.path.exists("/proc/meminfo"):
        with open("/proc/meminfo") as f:
            available = next(int(line.split()[1]) * 1024 for line in f
                             if line.startswith("MemAvailable:"))
        required = 4 * m + 16 * (n + 1) + 1024**3
        print(f"RAM disponibile: {available / 2**30:.2f} GiB; "
              f"richiesta con margine: {required / 2**30:.2f} GiB", flush=True)
        if available < required:
            raise RuntimeError("RAM disponibile insufficiente: liberare memoria prima della conversione")

    src_c = os.path.join(workdir, "conv_out.c")
    exe = os.path.join(workdir, "conv_out")
    with open(src_c, "w") as f:
        f.write(CONV_OUT_C)
    subprocess.run(["gcc", "-O2", "-o", exe, src_c], check=True)
    for p in (("2",) if first_ok else ("1", "2")):
        t0 = time.perf_counter()
        # Niente shell: controlla entrambi i processi e li ferma all'interruzione.
        reader = subprocess.Popen(["tar", "-xzOf", tar, "--wildcards", "*GAP-twitter.mtx"],
                                  stdout=subprocess.PIPE, start_new_session=True)
        writer = None
        try:
            writer = subprocess.Popen([exe, p, workdir], stdin=reader.stdout,
                                      start_new_session=True)
            reader.stdout.close()
            if writer.wait() != 0:
                raise RuntimeError(f"Conversione passata {p} fallita")
            if reader.wait() != 0:
                raise RuntimeError("Lettura archivio fallita: risultato non valido")
        finally:
            for process in (writer, reader):
                if process is not None:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
        target = ip if p == "1" else ic
        os.replace(target + ".part", target)
        print(f"passata {p}: {time.perf_counter() - t0:.0f} s")
    with open(marker + ".part", "w") as f:
        json.dump(dict(n=n, m=m, converter="ram-v1", orientation="followed_to_follower"), f)
    os.replace(marker + ".part", marker)
    return ip, ic


if __name__ == '__main__':
    print('1/2: liste follower -> seguito', flush=True)
    mc.build_gap_out_lists(os.getcwd())
    print('2/2: CSR entrante per power iteration', flush=True)
    build_gap_out_lists(os.getcwd())
    print('Preparazione completata. Nessuna simulazione Monte Carlo eseguita.', flush=True)


In [ ]:
%%writefile _pagerank_runtime/scripts/power_gap_follow.py
"""Colab: PageRank follower -> seguito usando gli array GAP gia' convertiti.

out_* e' l'adiacenza del vecchio verso: in CSR rappresenta A^T del nuovo.
follow_indptr contiene i gradi uscenti corretti. Calcoliamo A^T @ (r / d).
Nessuna conversione del grafo, nessun download. Prima prova, non benchmark.
"""
import argparse
import json
import time
from pathlib import Path

import numpy as np


def main():
    import cupy as cp
    import cupyx.scipy.sparse as sp
    parser = argparse.ArgumentParser()
    parser.add_argument('--workdir', default='/content')
    parser.add_argument('--max-iter', type=int, default=300)
    parser.add_argument('--tol', type=float, default=1e-6)
    args = parser.parse_args()
    base = Path(args.workdir)
    alpha = 0.85

    def update(A, inverse_degree, r):
        # sum(r) - sum(r on non-dangling nodes) is deliberately NOT used.
        dm = cp.sum(cp.where(inverse_degree == 0, r, 0), dtype=cp.float64)
        y = A @ (r * inverse_degree)
        y *= cp.float32(alpha)
        y += ((1-alpha + alpha*dm)/r.size).astype(cp.float32)
        return y

    # Asymmetric graph: 1->0, 2->0, 2->1; node 0 is dangling.
    dense = np.array([[0, 1, 1], [0, 0, 1], [0, 0, 0]], dtype=np.float32)
    test_A = sp.csr_matrix(cp.asarray(dense))
    inv = cp.asarray([0, 1, .5], dtype=cp.float32)
    r = cp.full(3, 1/3, dtype=cp.float32)
    expected_step = alpha * (dense @ np.array([0, 1/3, 1/6])) + (1-alpha+alpha/3)/3
    np.testing.assert_allclose(cp.asnumpy(update(test_A, inv, r)), expected_step,
                               rtol=2e-6, atol=2e-7)
    S = np.array([[1/3, 1, .5], [1/3, 0, .5], [1/3, 0, 0]])
    exact = np.linalg.solve(np.eye(3)-alpha*S, np.full(3, (1-alpha)/3))
    for _ in range(150):
        r = update(test_A, inv, r)
    np.testing.assert_allclose(cp.asnumpy(r), exact, rtol=3e-6, atol=3e-7)
    print('TEST GPU superato: verso, penzolante e riferimento diretto.', flush=True)
    del test_A, inv, r
    cp.get_default_memory_pool().free_all_blocks()

    meta = json.loads((base/'follow_complete.json').read_text())
    old = json.loads((base/'out_complete.json').read_text())
    if meta.get('orientation') != 'follower_to_followed':
        raise RuntimeError('Orientamento follow non confermato')
    if old.get('converter') != 'ram-v1':
        raise RuntimeError('Array del vecchio verso non riconosciuti')
    n, m = meta['n'], meta['m']
    assert (n, m) == (old['n'], old['m'])
    assert max(n, m) < np.iinfo(np.int32).max
    for name, size in [('out_indptr.bin',8*(n+1)), ('out_indices.bin',4*m),
                       ('follow_indptr.bin',8*(n+1))]:
        assert (base/name).stat().st_size == size, name
    ptr = np.memmap(base/'out_indptr.bin', dtype=np.int64, mode='r')
    col = np.memmap(base/'out_indices.bin', dtype=np.int32, mode='r')
    follow = np.memmap(base/'follow_indptr.bin', dtype=np.int64, mode='r')
    assert ptr[0] == follow[0] == 0 and ptr[-1] == follow[-1] == m

    free, total = cp.cuda.runtime.memGetInfo()
    required = 8*m + 4*(n+1) + 7*4*n + 512*1024**2
    print(f'VRAM libera {free/2**30:.2f} GiB; stima con margine {required/2**30:.2f} GiB', flush=True)
    if free < required:
        raise RuntimeError('VRAM insufficiente; nessun file e stato modificato')

    t0 = time.perf_counter()
    block = 2_000_000
    def upload(host):
        device = cp.empty(host.size, dtype=cp.int32)
        for s in range(0, host.size, block):
            part = host[s:s+block]
            assert np.all(part >= 0)
            if host is col:
                assert np.all(part < n)
            device[s:s+block].set(np.asarray(part, dtype=np.int32))
        return device

    gpu_ptr = upload(ptr)
    gpu_col = upload(col)
    data = cp.ones(m, dtype=cp.float32)
    A = sp.csr_matrix((data, gpu_col, gpu_ptr), shape=(n,n), copy=False)
    inverse_degree = cp.empty(n, dtype=cp.float32)
    dangling = 0
    for s in range(0,n,block):
        e = min(n,s+block)
        degree = np.diff(follow[s:e+1])
        assert np.all(degree >= 0)
        dangling += int(np.count_nonzero(degree == 0))
        inv = np.zeros(e-s, dtype=np.float32)
        np.divide(1., degree, out=inv, where=degree>0)
        inverse_degree[s:e].set(inv)
    cp.cuda.Stream.null.synchronize()
    setup = time.perf_counter()-t0
    print(f'Caricato in {setup:.1f} s: n={n}, m={m}, penzolanti={dangling}', flush=True)
    print(f'VRAM libera: {cp.cuda.runtime.memGetInfo()[0]/2**30:.2f} GiB', flush=True)
    r = cp.full(n,1/n,dtype=cp.float32)
    for _ in range(2):
        warm = update(A,inverse_degree,r)
    cp.cuda.Stream.null.synchronize()
    del warm
    history = []
    t0 = time.perf_counter()
    for iteration in range(1,args.max_iter+1):
        nxt = update(A,inverse_degree,r)
        delta = float(cp.sum(cp.abs(nxt-r),dtype=cp.float64))
        if not np.isfinite(delta):
            raise RuntimeError('Incremento non finito')
        r = nxt
        history.append(delta)
        if iteration == 1 or iteration % 10 == 0 or delta < args.tol:
            print(f'Iterazione {iteration}: Delta={delta:.6e}',flush=True)
        if delta < args.tol:
            break
    cp.cuda.Stream.null.synchronize()
    elapsed = time.perf_counter()-t0
    check = update(A,inverse_degree,r)
    residual = float(cp.sum(cp.abs(check-r),dtype=cp.float64))
    mass = float(cp.sum(r,dtype=cp.float64))
    minimum = float(r.min())
    assert minimum >= 0 and bool(cp.isfinite(r).all())
    props = cp.cuda.runtime.getDeviceProperties(0)
    name = props['name']
    if isinstance(name,bytes): name = name.decode()
    result = dict(dataset='GAP-twitter',orientation='follower_to_followed',
                  n=n,m=m,dangling=dangling,alpha=alpha,dtype='float32',
                  method='CSR A_transpose @ (rank * inverse_outdegree)',
                  device=name,cupy=cp.__version__,setup_seconds=setup,
                  solve_seconds=elapsed,iterations=iteration,tolerance=args.tol,
                  reached_tolerance=delta<args.tol,delta=delta,
                  fixed_point_residual=residual,sum=mass,minimum=minimum,
                  deltas=history)
    (base/'power_gap_follow.json').write_text(json.dumps(result,indent=2))
    # Chunked save avoids an extra full host copy.
    saved = np.lib.format.open_memmap(base/'power_gap_follow_rank.npy',mode='w+',
                                    dtype=np.float32,shape=(n,))
    for s in range(0,n,block): saved[s:s+block] = cp.asnumpy(r[s:s+block])
    saved.flush()
    print(json.dumps({k:v for k,v in result.items() if k!='deltas'},indent=2),flush=True)
    print('Salvati power_gap_follow.json e power_gap_follow_rank.npy',flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile _pagerank_runtime/scripts/benchmark_vector_unweighted_gap.py
"""GAP-twitter: Vector originale contro Vector senza valori espliciti, e CuPy.
Nessun download: richiede out_* e follow_* del runtime GAP originale.
"""
import argparse
import json
import time
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import cupy as cp
import cupyx.scipy.sparse as sp
import scipy.sparse as sps
cps = sp

CUDA_CSR = r'''
extern "C" __global__ void csr_scalar(
    int n, const int* ptr, const int* col, const float* val,
    const float* x, float* y) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;
    if (row >= n) return;
    float sum = 0.0f;
    for (int k = ptr[row]; k < ptr[row+1]; ++k)
        sum += val[k] * x[col[k]];
    y[row] = sum;
}
extern "C" __global__ void csr_vector(
    int n, const int* ptr, const int* col, const float* val,
    const float* x, float* y) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int row = tid / 32;
    int lane = tid % 32;
    if (row >= n) return; // Tutto il warp esce insieme.
    float sum = 0.0f;
    for (int k = ptr[row] + lane; k < ptr[row+1]; k += 32)
        sum += val[k] * x[col[k]];
    for (int offset = 16; offset > 0; offset /= 2)
        sum += __shfl_down_sync(0xffffffff, sum, offset);
    if (lane == 0) y[row] = sum;
}
extern "C" __global__ void csr_vector_unweighted(
    int n, const int* ptr, const int* col,
    const float* x, float* y) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int row = tid / 32;
    int lane = tid % 32;
    if (row >= n) return; // Tutto il warp esce insieme.
    float sum = 0.0f;
    for (int k = ptr[row] + lane; k < ptr[row+1]; k += 32)
        sum += x[col[k]];
    for (int offset = 16; offset > 0; offset /= 2)
        sum += __shfl_down_sync(0xffffffff, sum, offset);
    if (lane == 0) y[row] = sum;
}
'''


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--workdir', default='/content')
    parser.add_argument('--repeats', type=int, default=3)
    parser.add_argument('--max-iter', type=int, default=300)
    args = parser.parse_args()
    if args.repeats < 1: raise ValueError('repeats deve essere positivo')
    base = Path(args.workdir)
    alpha = .85
    # Fail before compilation if the large graph files are absent.
    required = ['out_complete.json','follow_complete.json',
                'out_indptr.bin','out_indices.bin','follow_indptr.bin']
    missing = [name for name in required if not (base/name).is_file()]
    if missing:
        raise RuntimeError('Usare il runtime GAP originale; file mancanti: '+', '.join(missing))
    kernels = {name: cp.RawKernel(CUDA_CSR, name)
               for name in ('csr_vector', 'csr_vector_unweighted')}

    def product(name, matrix, x):
        if name == 'cupy':
            return matrix @ x
        rows = matrix.shape[0]
        out = cp.empty(rows, dtype=cp.float32)
        blocks = (rows+255)//256 if name == 'csr_scalar' else (rows+7)//8
        arguments = (np.int32(rows), matrix.indptr, matrix.indices)
        if name != 'csr_vector_unweighted':
            arguments += (matrix.data,)
        kernels[name]((blocks,), (256,), arguments + (x, out))
        return out

    # Test non multiplo della dimensione di blocco/warp: righe vuote, 31,
    # 32, 33 elementi, e una riga da 4097 elementi con piu' giri del warp.
    rng = np.random.default_rng(17)
    lengths = np.array([0,1,2,31,32,33,65,257,4097,0,7])
    ptr = np.r_[0,np.cumsum(lengths)].astype(np.int32)
    columns = np.concatenate([np.arange(k,dtype=np.int32) for k in lengths])
    values = np.ones(len(columns), dtype=np.float32)
    small = sps.csr_matrix((values,columns,ptr),shape=(len(lengths),4097))
    small_gpu = cps.csr_matrix(small)
    for vector in (rng.random(4097).astype(np.float32),
                   rng.uniform(-1,1,4097).astype(np.float32)):
        expected = small.astype(np.float64) @ vector.astype(np.float64)
        scale = abs(small.astype(np.float64)) @ abs(vector.astype(np.float64))
        for name in kernels:
            got = cp.asnumpy(product(name,small_gpu,cp.asarray(vector)))
            assert np.all(np.isfinite(got))
            assert np.all(abs(got-expected) <= 1e-5*scale+1e-7), name
    print('Test GPU su righe vuote/corte/lunghe e vettori con segno: superati.',flush=True)

    del small_gpu
    cp.get_default_memory_pool().free_all_blocks()
    meta = json.loads((base/'follow_complete.json').read_text())
    old = json.loads((base/'out_complete.json').read_text())
    if meta.get('orientation') != 'follower_to_followed' or meta.get('converter') != 'ram-follow-v1':
        raise RuntimeError('Orientamento follow non confermato')
    if old.get('converter') != 'ram-v1' or old.get('orientation') != 'followed_to_follower':
        raise RuntimeError('Array del vecchio verso non riconosciuti')
    n, m = meta['n'], meta['m']
    assert (n, m) == (old['n'], old['m'])
    assert max(n, m) < np.iinfo(np.int32).max
    for name, size in [('out_indptr.bin',8*(n+1)), ('out_indices.bin',4*m),
                       ('follow_indptr.bin',8*(n+1))]:
        assert (base/name).stat().st_size == size, name
    ptr = np.memmap(base/'out_indptr.bin', dtype=np.int64, mode='r')
    col = np.memmap(base/'out_indices.bin', dtype=np.int32, mode='r')
    follow = np.memmap(base/'follow_indptr.bin', dtype=np.int64, mode='r')
    assert ptr[0] == follow[0] == 0 and ptr[-1] == follow[-1] == m

    free, total = cp.cuda.runtime.memGetInfo()
    required = 8*m + 4*(n+1) + 7*4*n + 512*1024**2
    print(f'VRAM libera {free/2**30:.2f} GiB; stima con margine {required/2**30:.2f} GiB', flush=True)
    if free < required:
        raise RuntimeError('VRAM insufficiente; nessun file e stato modificato')

    t0 = time.perf_counter()
    block = 2_000_000
    def upload(host):
        device = cp.empty(host.size, dtype=cp.int32)
        for s in range(0, host.size, block):
            part = host[s:s+block]
            assert np.all(part >= 0)
            if host is col:
                assert np.all(part < n)
            device[s:s+block].set(np.asarray(part, dtype=np.int32))
        return device

    gpu_ptr = upload(ptr)
    gpu_col = upload(col)
    A = None
    active = None
    inverse_degree = cp.empty(n, dtype=cp.float32)
    dangling = 0
    for s in range(0,n,block):
        e = min(n,s+block)
        degree = np.diff(follow[s:e+1])
        assert np.all(degree >= 0)
        dangling += int(np.count_nonzero(degree == 0))
        inv = np.zeros(e-s, dtype=np.float32)
        np.divide(1., degree, out=inv, where=degree>0)
        inverse_degree[s:e].set(inv)
    cp.cuda.Stream.null.synchronize()
    setup = time.perf_counter()-t0
    print(f'Caricato in {setup:.1f} s: n={n}, m={m}, penzolanti={dangling}', flush=True)
    print(f'VRAM libera: {cp.cuda.runtime.memGetInfo()[0]/2**30:.2f} GiB', flush=True)

    def update(name, r):
        mass = cp.sum(cp.where(inverse_degree == 0, r, 0), dtype=cp.float64)
        y = product(name, A, r * inverse_degree)
        y *= cp.float32(alpha)
        y += ((1-alpha+alpha*mass)/n).astype(cp.float32)
        return y

    def activate(name):
        # Release ALL owners of the values array, then release cached pool blocks.
        # Allocation and CSR construction are excluded from solve timing.
        nonlocal A, active
        cp.cuda.Stream.null.synchronize()
        A = None
        active = None
        cp.get_default_memory_pool().free_all_blocks()
        if name == 'csr_vector_unweighted':
            A = SimpleNamespace(shape=(n,n), indptr=gpu_ptr, indices=gpu_col)
        else:
            A = sp.csr_matrix((cp.ones(m,dtype=cp.float32),gpu_col,gpu_ptr),
                              shape=(n,n),copy=False)
            assert A.indices.data.ptr == gpu_col.data.ptr
            assert A.indptr.data.ptr == gpu_ptr.data.ptr
        active = name
        cp.cuda.Stream.null.synchronize()
        free_bytes, total_bytes = cp.cuda.runtime.memGetInfo()
        return dict(values_array_bytes=0 if name=='csr_vector_unweighted' else 4*m,
                    graph_array_bytes=4*(n+1)+4*m+(0 if name=='csr_vector_unweighted' else 4*m),
                    device_free_bytes_after_activation=int(free_bytes),
                    device_total_bytes=int(total_bytes),
                    pool_live_bytes_after_activation=int(cp.get_default_memory_pool().used_bytes()))

    def warmup(name):
        initial = cp.full(n,1/n,dtype=cp.float32)
        warm = update(name,initial)
        cp.cuda.Stream.null.synchronize()
        del initial, warm

    names = ['cupy','csr_vector','csr_vector_unweighted']
    # Actual graph: compare the two Vector products on exactly the same input.
    activate('csr_vector')
    test_x = cp.full(n,1/n,dtype=cp.float32) * inverse_degree
    expected = product('csr_vector',A,test_x)
    activate('csr_vector_unweighted')
    actual = product('csr_vector_unweighted',A,test_x)
    step_l1 = float(cp.abs(actual-expected).sum(dtype=cp.float64))
    step_max = float(cp.abs(actual-expected).max())
    if not bool(cp.isfinite(actual).all()) or step_l1 > 1e-6:
        raise RuntimeError(f'Confronto Vector con/senza valori fallito: L1={step_l1}')
    print(f'SpMV GAP, Vector con/senza valori: L1={step_l1:.3e}, max={step_max:.3e}',flush=True)
    del test_x, expected, actual

    def solve(name):
        rank = cp.full(n, 1/n, dtype=cp.float32)
        cp.cuda.Stream.null.synchronize()
        start = time.perf_counter()
        for iteration in range(1,args.max_iter+1):
            nxt = update(name, rank)
            delta = float(cp.sum(cp.abs(nxt-rank),dtype=cp.float64))
            if not np.isfinite(delta): raise RuntimeError('Delta non finito')
            rank = nxt
            if delta < 1e-6: break
        cp.cuda.Stream.null.synchronize()
        seconds = time.perf_counter()-start
        return rank, iteration, delta, seconds

    # A fresh library solution is a numerical comparison, not exact PageRank.
    activate('cupy')
    warmup('cupy')
    print('Calcolo riferimento CuPy...',flush=True)
    reference, _, reference_delta, _ = solve('cupy')
    if reference_delta >= 1e-6: raise RuntimeError('Riferimento non convergente')
    reference_host = cp.asnumpy(reference)
    del reference
    rows = []
    path = base/'benchmark_vector_unweighted_gap.json'
    props = cp.cuda.runtime.getDeviceProperties(0)
    gpu = props['name']
    if isinstance(gpu,bytes): gpu=gpu.decode()
    result = dict(dataset='GAP-twitter',orientation='follower_to_followed',
        n=n,m=m,dangling=dangling,alpha=alpha,dtype='float32',gpu=gpu,
        cupy=cp.__version__,setup_seconds=setup,repetitions=args.repeats,
        tolerance=1e-6,check_every=1,block_threads=256,
        matrix='Unweighted A transpose; inverse outdegree applied to vector',
        protocol='Synchronized wall time, uniform initialization excluded; one update warmup before every trial; randomized order; matrix activation/allocation and validation excluded; per-iteration temporaries included',
        reference='Fresh CuPy float32 solve with Delta < 1e-6, not exact PageRank',
        synthetic_tests_passed=True,real_graph_vector_comparison=dict(l1=step_l1,max_abs=step_max),
        expected_values_saving_bytes=4*m,
        memory_note='Graph arrays and live allocations after activation, before warmup; not peak solve memory. Values array released for unweighted variant. CuPy matrix recreated outside timing for residual validation.',
        source='https://www.nvidia.com/docs/io/66889/nvr-2008-004.pdf section 4.3',
        trials=rows)
    order = np.random.default_rng(2026)
    for repetition in range(1,args.repeats+1):
        for index in order.permutation(3):
            name = names[index]
            print(f'Avvio {name}, ripetizione {repetition}/{args.repeats}',flush=True)
            memory = activate(name)
            print(f'  array valori: {memory["values_array_bytes"]/2**30:.2f} GiB; '
                  f'VRAM libera: {memory["device_free_bytes_after_activation"]/2**30:.2f} GiB',flush=True)
            warmup(name)
            rank, iterations, delta, seconds = solve(name)
            valid = bool(cp.isfinite(rank).all()) and float(rank.min()) >= 0
            mass = float(cp.sum(rank,dtype=cp.float64))
            # Residual evaluated with the library for every variant.
            # The weighted matrix is reconstructed only AFTER timing ends.
            if active == 'csr_vector_unweighted': activate('cupy')
            check = update('cupy',rank)
            residual = float(cp.sum(cp.abs(check-rank),dtype=cp.float64))
            del check
            distance = 0.
            for s in range(0,n,block):
                part = cp.asnumpy(rank[s:s+block]).astype(np.float64)
                distance += float(np.abs(part-reference_host[s:s+block]).sum())
            rows.append(dict(variant=name,repetition=repetition,seconds=seconds,
                iterations=iterations,delta=delta,reached_tolerance=delta<1e-6,memory=memory,
                finite_nonnegative=valid,sum=mass,library_residual=residual,
                distance_l1_library_reference=distance,
                numerical_check_passed=valid and delta<1e-6 and abs(mass-1)<1e-5 and residual<1e-5 and distance<1e-5))
            path.write_text(json.dumps(result,indent=2))
            print(f'{name}: {seconds:.2f} s, {iterations} iterazioni, '
                  f'scarto L1 {distance:.3e}, residuo {residual:.3e}, somma {mass:.9f}',flush=True)
            del rank
            if not rows[-1]['numerical_check_passed']:
                raise RuntimeError('Verifica fallita; risultati parziali salvati')
    for name in names:
        times = [row['seconds'] for row in rows if row['variant']==name]
        print(f'{name}: mediana {np.median(times):.2f} s '
              f'[{min(times):.2f}, {max(times):.2f}]',flush=True)
    print(f'Array valori eliminabile: {4*m/2**30:.3f} GiB',flush=True)
    print('Salvato:',path,flush=True)

if __name__ == '__main__':
    main()


In [ ]:
%%writefile _pagerank_runtime/scripts/benchmark_kernel_gap.py
"""Benchmark GAP-twitter: stessi dati e aggiornamento, solo SpMV cambia.
Nessun download: richiede out_* e follow_* del runtime GAP originale.
"""
import argparse
import json
import time
from pathlib import Path
import numpy as np
import cupy as cp
import cupyx.scipy.sparse as sp
import scipy.sparse as sps
cps = sp

CUDA_CSR = r'''
extern "C" __global__ void csr_scalar(
    int n, const int* ptr, const int* col, const float* val,
    const float* x, float* y) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;
    if (row >= n) return;
    float sum = 0.0f;
    for (int k = ptr[row]; k < ptr[row+1]; ++k)
        sum += val[k] * x[col[k]];
    y[row] = sum;
}
extern "C" __global__ void csr_vector(
    int n, const int* ptr, const int* col, const float* val,
    const float* x, float* y) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int row = tid / 32;
    int lane = tid % 32;
    if (row >= n) return; // Tutto il warp esce insieme.
    float sum = 0.0f;
    for (int k = ptr[row] + lane; k < ptr[row+1]; k += 32)
        sum += val[k] * x[col[k]];
    for (int offset = 16; offset > 0; offset /= 2)
        sum += __shfl_down_sync(0xffffffff, sum, offset);
    if (lane == 0) y[row] = sum;
}
'''


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--workdir', default='/content')
    parser.add_argument('--repeats', type=int, default=3)
    parser.add_argument('--max-iter', type=int, default=300)
    args = parser.parse_args()
    if args.repeats < 1: raise ValueError('repeats deve essere positivo')
    base = Path(args.workdir)
    alpha = .85
    # Fail before compilation if the large graph files are absent.
    required = ['out_complete.json','follow_complete.json',
                'out_indptr.bin','out_indices.bin','follow_indptr.bin']
    missing = [name for name in required if not (base/name).is_file()]
    if missing:
        raise RuntimeError('Usare il runtime GAP originale; file mancanti: '+', '.join(missing))
    kernels = {name: cp.RawKernel(CUDA_CSR, name)
               for name in ('csr_scalar', 'csr_vector')}

    def product(name, matrix, x):
        if name == 'cupy':
            return matrix @ x
        rows = matrix.shape[0]
        out = cp.empty(rows, dtype=cp.float32)
        blocks = (rows+255)//256 if name == 'csr_scalar' else (rows+7)//8
        kernels[name]((blocks,), (256,),
                      (np.int32(rows), matrix.indptr, matrix.indices,
                       matrix.data, x, out))
        return out

    # Test non multiplo della dimensione di blocco/warp: righe vuote, 31,
    # 32, 33 elementi, e una riga da 4097 elementi con piu' giri del warp.
    rng = np.random.default_rng(17)
    lengths = np.array([0,1,2,31,32,33,65,257,4097,0,7])
    ptr = np.r_[0,np.cumsum(lengths)].astype(np.int32)
    columns = np.concatenate([np.arange(k,dtype=np.int32) for k in lengths])
    values = rng.uniform(.01,1,len(columns)).astype(np.float32)
    small = sps.csr_matrix((values,columns,ptr),shape=(len(lengths),4097))
    small_gpu = cps.csr_matrix(small)
    for vector in (rng.random(4097).astype(np.float32),
                   rng.uniform(-1,1,4097).astype(np.float32)):
        expected = small.astype(np.float64) @ vector.astype(np.float64)
        scale = abs(small.astype(np.float64)) @ abs(vector.astype(np.float64))
        for name in kernels:
            got = cp.asnumpy(product(name,small_gpu,cp.asarray(vector)))
            assert np.all(np.isfinite(got))
            assert np.all(abs(got-expected) <= 1e-5*scale+1e-7), name
    print('Test GPU su righe vuote/corte/lunghe e vettori con segno: superati.',flush=True)

    del small_gpu
    cp.get_default_memory_pool().free_all_blocks()
    meta = json.loads((base/'follow_complete.json').read_text())
    old = json.loads((base/'out_complete.json').read_text())
    if meta.get('orientation') != 'follower_to_followed':
        raise RuntimeError('Orientamento follow non confermato')
    if old.get('converter') != 'ram-v1':
        raise RuntimeError('Array del vecchio verso non riconosciuti')
    n, m = meta['n'], meta['m']
    assert (n, m) == (old['n'], old['m'])
    assert max(n, m) < np.iinfo(np.int32).max
    for name, size in [('out_indptr.bin',8*(n+1)), ('out_indices.bin',4*m),
                       ('follow_indptr.bin',8*(n+1))]:
        assert (base/name).stat().st_size == size, name
    ptr = np.memmap(base/'out_indptr.bin', dtype=np.int64, mode='r')
    col = np.memmap(base/'out_indices.bin', dtype=np.int32, mode='r')
    follow = np.memmap(base/'follow_indptr.bin', dtype=np.int64, mode='r')
    assert ptr[0] == follow[0] == 0 and ptr[-1] == follow[-1] == m

    free, total = cp.cuda.runtime.memGetInfo()
    required = 8*m + 4*(n+1) + 7*4*n + 512*1024**2
    print(f'VRAM libera {free/2**30:.2f} GiB; stima con margine {required/2**30:.2f} GiB', flush=True)
    if free < required:
        raise RuntimeError('VRAM insufficiente; nessun file e stato modificato')

    t0 = time.perf_counter()
    block = 2_000_000
    def upload(host):
        device = cp.empty(host.size, dtype=cp.int32)
        for s in range(0, host.size, block):
            part = host[s:s+block]
            assert np.all(part >= 0)
            if host is col:
                assert np.all(part < n)
            device[s:s+block].set(np.asarray(part, dtype=np.int32))
        return device

    gpu_ptr = upload(ptr)
    gpu_col = upload(col)
    data = cp.ones(m, dtype=cp.float32)
    A = sp.csr_matrix((data, gpu_col, gpu_ptr), shape=(n,n), copy=False)
    inverse_degree = cp.empty(n, dtype=cp.float32)
    dangling = 0
    for s in range(0,n,block):
        e = min(n,s+block)
        degree = np.diff(follow[s:e+1])
        assert np.all(degree >= 0)
        dangling += int(np.count_nonzero(degree == 0))
        inv = np.zeros(e-s, dtype=np.float32)
        np.divide(1., degree, out=inv, where=degree>0)
        inverse_degree[s:e].set(inv)
    cp.cuda.Stream.null.synchronize()
    setup = time.perf_counter()-t0
    print(f'Caricato in {setup:.1f} s: n={n}, m={m}, penzolanti={dangling}', flush=True)
    print(f'VRAM libera: {cp.cuda.runtime.memGetInfo()[0]/2**30:.2f} GiB', flush=True)

    def update(name, r):
        mass = cp.sum(cp.where(inverse_degree == 0, r, 0), dtype=cp.float64)
        y = product(name, A, r * inverse_degree)
        y *= cp.float32(alpha)
        y += ((1-alpha+alpha*mass)/n).astype(cp.float32)
        return y

    names = ['cupy','csr_scalar','csr_vector']
    # Compile and warm each update outside the timed region.
    initial = cp.full(n, 1/n, dtype=cp.float32)
    for name in names:
        warm = update(name, initial)
        cp.cuda.Stream.null.synchronize()
        del warm
    del initial

    def solve(name):
        rank = cp.full(n, 1/n, dtype=cp.float32)
        cp.cuda.Stream.null.synchronize()
        start = time.perf_counter()
        for iteration in range(1,args.max_iter+1):
            nxt = update(name, rank)
            delta = float(cp.sum(cp.abs(nxt-rank),dtype=cp.float64))
            if not np.isfinite(delta): raise RuntimeError('Delta non finito')
            rank = nxt
            if delta < 1e-6: break
        cp.cuda.Stream.null.synchronize()
        seconds = time.perf_counter()-start
        return rank, iteration, delta, seconds

    # A fresh library solution is a numerical comparison, not exact PageRank.
    reference, _, reference_delta, _ = solve('cupy')
    if reference_delta >= 1e-6: raise RuntimeError('Riferimento non convergente')
    reference_host = cp.asnumpy(reference)
    del reference
    rows = []
    path = base/'benchmark_kernel_gap.json'
    props = cp.cuda.runtime.getDeviceProperties(0)
    gpu = props['name']
    if isinstance(gpu,bytes): gpu=gpu.decode()
    result = dict(dataset='GAP-twitter',orientation='follower_to_followed',
        n=n,m=m,dangling=dangling,alpha=alpha,dtype='float32',gpu=gpu,
        cupy=cp.__version__,setup_seconds=setup,repetitions=args.repeats,
        tolerance=1e-6,check_every=1,block_threads=256,
        matrix='Unweighted A transpose; inverse outdegree applied to vector',
        protocol='Synchronized wall time, uniform initialization excluded; one update warmup per variant; randomized order; validation excluded',
        reference='Fresh CuPy float32 solve with Delta < 1e-6, not exact PageRank',
        synthetic_tests_passed=True,
        source='https://www.nvidia.com/docs/io/66889/nvr-2008-004.pdf section 4.3',
        trials=rows)
    order = np.random.default_rng(2026)
    for repetition in range(1,args.repeats+1):
        for index in order.permutation(3):
            name = names[index]
            print(f'Avvio {name}, ripetizione {repetition}/{args.repeats}',flush=True)
            rank, iterations, delta, seconds = solve(name)
            valid = bool(cp.isfinite(rank).all()) and float(rank.min()) >= 0
            mass = float(cp.sum(rank,dtype=cp.float64))
            # Residual evaluated with the library for every variant.
            check = update('cupy',rank)
            residual = float(cp.sum(cp.abs(check-rank),dtype=cp.float64))
            del check
            distance = 0.
            for s in range(0,n,block):
                part = cp.asnumpy(rank[s:s+block]).astype(np.float64)
                distance += float(np.abs(part-reference_host[s:s+block]).sum())
            rows.append(dict(variant=name,repetition=repetition,seconds=seconds,
                iterations=iterations,delta=delta,reached_tolerance=delta<1e-6,
                finite_nonnegative=valid,sum=mass,library_residual=residual,
                distance_l1_library_reference=distance))
            path.write_text(json.dumps(result,indent=2))
            print(f'{name}: {seconds:.2f} s, {iterations} iterazioni, '
                  f'scarto L1 {distance:.3e}, residuo {residual:.3e}, somma {mass:.9f}',flush=True)
            del rank
            if not valid or delta >= 1e-6:
                raise RuntimeError('Verifica fallita; risultati parziali salvati')
    for name in names:
        times = [row['seconds'] for row in rows if row['variant']==name]
        print(f'{name}: mediana {np.median(times):.2f} s '
              f'[{min(times):.2f}, {max(times):.2f}]',flush=True)
    print('Salvato:',path,flush=True)

if __name__ == '__main__':
    main()


In [ ]:
%%writefile _pagerank_runtime/scripts/verify_gap_follow.py
"""
GAP-twitter, verso follower -> utente seguito: verifiche sul grafo e errore del Monte Carlo.

PARTE 1 - verifiche (CPU, legge solo i file gia' presenti in /content):
  * il grafo nuovo (follow_*) e' esattamente il trasposto del vecchio (out_*):
      grado uscente nuovo  == grado entrante vecchio
      grado entrante nuovo == grado uscente vecchio
  * nodi isolati (grado entrante e uscente nulli) == 19.926.185 (pagina SuiteSparse)
  * nodi senza link uscenti nel verso nuovo == 25.889.280 (Monte Carlo e power iteration)
  * plausibilita': grado entrante massimo (follower) contro grado uscente massimo

PARTE 2 - solo se la parte 1 e' tutta superata (GPU):
  errore in norma 1 del Monte Carlo rispetto al PageRank della power iteration
  (power_gap_follow_rank.npy, errore stimato ~5e-6, trascurabile rispetto al Monte Carlo).

Uso in Colab:
    !python -u /content/verify_gap_follow.py
Opzioni: --solo-verifiche   salta la parte 2
         --N 1e8,1e9        numeri di cammini (default 1e8,1e9; seme 1)
Nessun file viene modificato: si scrive solo verify_gap_follow.json.
"""

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np

CHUNK = 1 << 27            # 134 milioni di elementi per blocco (~0,5 GB di int32)


# ----------------------------------------------------------------------------- utilita'

def open_graph(base, prefix, n, m):
    ip = base / f"{prefix}_indptr.bin"
    ic = base / f"{prefix}_indices.bin"
    for p, size in [(ip, 8 * (n + 1)), (ic, 4 * m)]:
        if not p.exists():
            present = sorted(x.name for x in base.glob("*.bin"))
            sys.exit(f"Manca {p.name}. File .bin presenti: {present}")
        if p.stat().st_size != size:
            sys.exit(f"{p.name}: dimensione {p.stat().st_size}, attesa {size}")
    return (np.memmap(ip, dtype=np.int64, mode="r"),
            np.memmap(ic, dtype=np.int32, mode="r"))


def in_degree(indices, n, label):
    """Conta quante volte ogni nodo compare come destinazione (a blocchi)."""
    deg = np.zeros(n, dtype=np.int64)
    t0 = time.perf_counter()
    for s in range(0, indices.size, CHUNK):
        part = np.asarray(indices[s:s + CHUNK])
        if part.min() < 0 or part.max() >= n:
            sys.exit(f"{label}: indice fuori intervallo nel blocco che inizia a {s}")
        deg += np.bincount(part, minlength=n)
        print(f"  {label}: {min(s + CHUNK, indices.size):,}/{indices.size:,} archi "
              f"({time.perf_counter() - t0:.0f} s)", flush=True)
    return deg


# ----------------------------------------------------------------------------- parte 1

def verifiche(base, n, m, isolati_attesi, penzolanti_attesi):
    old_ip, old_ic = open_graph(base, "out", n, m)       # verso vecchio: i -> j
    new_ip, new_ic = open_graph(base, "follow", n, m)    # verso nuovo:   j -> i

    out_old = np.diff(old_ip)
    out_new = np.diff(new_ip)
    print("Gradi entranti del verso vecchio...", flush=True)
    in_old = in_degree(old_ic, n, "vecchio")
    print("Gradi entranti del verso nuovo...", flush=True)
    in_new = in_degree(new_ic, n, "nuovo")

    r = {}
    r["archi_vecchio"] = int(out_old.sum())
    r["archi_nuovo"] = int(out_new.sum())
    r["uscente_nuovo_uguale_entrante_vecchio"] = bool(np.array_equal(out_new, in_old))
    r["entrante_nuovo_uguale_uscente_vecchio"] = bool(np.array_equal(in_new, out_old))
    r["isolati_vecchio"] = int(np.count_nonzero((out_old == 0) & (in_old == 0)))
    r["isolati_nuovo"] = int(np.count_nonzero((out_new == 0) & (in_new == 0)))
    r["penzolanti_nuovo"] = int(np.count_nonzero(out_new == 0))
    r["max_grado_entrante_nuovo"] = int(in_new.max())      # follower di un account
    r["max_grado_uscente_nuovo"] = int(out_new.max())      # account seguiti da un utente
    r["nodo_max_entrante_nuovo"] = int(in_new.argmax())

    checks = {
        "archi == m (entrambi i versi)": r["archi_vecchio"] == m == r["archi_nuovo"],
        "grado uscente nuovo == grado entrante vecchio": r["uscente_nuovo_uguale_entrante_vecchio"],
        "grado entrante nuovo == grado uscente vecchio": r["entrante_nuovo_uguale_uscente_vecchio"],
        f"isolati == {isolati_attesi:,} (entrambi i versi)":
            r["isolati_vecchio"] == isolati_attesi == r["isolati_nuovo"],
        f"penzolanti nel verso nuovo == {penzolanti_attesi:,}":
            r["penzolanti_nuovo"] == penzolanti_attesi,
    }
    print("\n=== PARTE 1: verifiche sul grafo ===")
    for k, ok in checks.items():
        print(f"  [{'OK' if ok else 'FALLITA'}] {k}")
    print(f"  isolati: vecchio {r['isolati_vecchio']:,}, nuovo {r['isolati_nuovo']:,}")
    print(f"  penzolanti nel verso nuovo: {r['penzolanti_nuovo']:,}")
    print(f"  plausibilita' (non e' una prova): grado entrante massimo "
          f"{r['max_grado_entrante_nuovo']:,} (nodo {r['nodo_max_entrante_nuovo']}), "
          f"grado uscente massimo {r['max_grado_uscente_nuovo']:,}")
    r["tutte_superate"] = all(checks.values())
    r["controlli"] = {k: bool(v) for k, v in checks.items()}
    del old_ip, old_ic, new_ic, in_old, in_new, out_old, out_new
    return r


# ----------------------------------------------------------------------------- parte 2

def pagerank_montecarlo(indptr, indices, n, N, alpha, seed, batch, xp):
    """Algoritmo 2 (Monte Carlo end-point), identico a montecarlo.py."""
    rng = xp.random.RandomState(seed)
    counts = xp.zeros(n, dtype=xp.int64)
    for start in range(0, N, batch):
        B = min(batch, N - start)
        ends = xp.empty(B, dtype=xp.int32)
        walk = xp.arange(B, dtype=xp.int32)
        pos = rng.randint(0, n, size=B, dtype=xp.int64)
        while walk.size > 0:
            go = rng.random_sample(walk.size) < alpha
            stop = ~go
            ends[walk[stop]] = pos[stop]
            walk, pos = walk[go], pos[go]
            if walk.size == 0:
                break
            begin = indptr[pos]
            deg = indptr[pos + 1] - begin
            u = rng.random_sample(walk.size)
            k = xp.minimum((u * deg).astype(xp.int64), xp.maximum(deg - 1, 0))
            has = deg > 0
            pos = xp.where(has, indices[xp.where(has, begin + k, 0)].astype(xp.int64),
                           rng.randint(0, n, size=walk.size, dtype=xp.int64))
        counts += xp.bincount(ends, minlength=n)
    return counts.astype(xp.float64) / N


def errore_montecarlo(base, n, m, Ns, seed, batch, alpha, use_cpu):
    if use_cpu:
        xp = np
    else:
        import cupy as cp
        xp = cp
    ref_path = base / "power_gap_follow_rank.npy"
    if not ref_path.exists():
        sys.exit("Manca power_gap_follow_rank.npy")
    info = json.loads((base / "power_gap_follow.json").read_text()) \
        if (base / "power_gap_follow.json").exists() else {}

    ip, ic = open_graph(base, "follow", n, m)

    def up(arr, dtype):
        if xp is np:
            return np.asarray(arr, dtype=dtype)
        out = xp.empty(arr.size, dtype=dtype)
        for s in range(0, arr.size, CHUNK):
            out[s:s + CHUNK] = xp.asarray(np.asarray(arr[s:s + CHUNK], dtype=dtype))
        return out

    t0 = time.perf_counter()
    d_ip, d_ic = up(ip, np.int64), up(ic, np.int32)
    ref = up(np.load(ref_path, mmap_mode="r"), np.float64)
    ref /= ref.sum()                     # la somma in float32 era 1,0000003
    if xp is not np:
        xp.cuda.Stream.null.synchronize()
    print(f"\nCaricamento: {time.perf_counter() - t0:.1f} s", flush=True)

    # riscaldamento (compilazione dei kernel CuPy)
    pagerank_montecarlo(d_ip, d_ic, n, min(1_000_000, batch), alpha, 12345, batch, xp)

    ref_top = (ref.get() if xp is not np else ref)
    ref_top10 = np.argpartition(-ref_top, 10)[:10]
    ref_top10 = set(ref_top10.tolist())

    rows = []
    print("=== PARTE 2: errore del Monte Carlo rispetto alla power iteration ===")
    for N in Ns:
        t0 = time.perf_counter()
        r_hat = pagerank_montecarlo(d_ip, d_ic, n, N, alpha, seed, batch, xp)
        if xp is not np:
            xp.cuda.Stream.null.synchronize()
        sec = time.perf_counter() - t0
        err = float(xp.abs(r_hat - ref).sum())
        err_max = float(xp.abs(r_hat - ref).max())
        h = r_hat.get() if xp is not np else r_hat
        top10 = np.argpartition(-h, 10)[:10]
        comuni = len(ref_top10 & set(top10.tolist()))
        rows.append(dict(N=N, seed=seed, seconds=sec, err_l1=err, err_max=err_max,
                         top10_in_comune_con_power=comuni))
        print(f"  N = {N:,}: {sec:.1f} s, errore L1 = {err:.4e}, errore max = {err_max:.2e}, "
              f"top-10 in comune con la power iteration = {comuni}/10", flush=True)
        del r_hat, h
        if xp is not np:
            xp.get_default_memory_pool().free_all_blocks()
    if len(rows) >= 2:
        a, b = rows[0], rows[-1]
        pend = np.log(b["err_l1"] / a["err_l1"]) / np.log(b["N"] / a["N"])
        print(f"  pendenza log-log dell'errore: {pend:.3f} "
              f"(circa -0,5 solo quando N e' molto maggiore di n = {n:,})")
    return dict(riferimento="power_gap_follow_rank.npy (float32, rinormalizzato a somma 1)",
                riferimento_delta=info.get("delta"), prove=rows)


# ----------------------------------------------------------------------------- main

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--workdir", default="/content")
    p.add_argument("--solo-verifiche", action="store_true")
    p.add_argument("--N", type=lambda s: [int(float(x)) for x in s.split(",")],
                   default=[10**8, 10**9])
    p.add_argument("--seed", type=int, default=1)
    p.add_argument("--batch", type=int, default=1_000_000)
    p.add_argument("--isolati-attesi", type=int, default=19_926_185)
    p.add_argument("--penzolanti-attesi", type=int, default=25_889_280)
    p.add_argument("--cpu", action="store_true", help="solo per prove locali")
    args = p.parse_args()
    base = Path(args.workdir)

    meta = json.loads((base / "follow_complete.json").read_text())
    if meta.get("orientation") != "follower_to_followed":
        sys.exit("follow_complete.json non dichiara follower_to_followed")
    n, m = meta["n"], meta["m"]
    print(f"n = {n:,}  m = {m:,}  verso: {meta['orientation']}", flush=True)

    out = dict(n=n, m=m, orientation=meta["orientation"])
    out["verifiche"] = verifiche(base, n, m, args.isolati_attesi, args.penzolanti_attesi)

    if not out["verifiche"]["tutte_superate"]:
        print("\nALMENO UNA VERIFICA E' FALLITA: parte 2 non eseguita.")
    elif args.solo_verifiche:
        print("\nVerifiche superate. Parte 2 saltata su richiesta.")
    else:
        out["errore_montecarlo"] = errore_montecarlo(base, n, m, args.N, args.seed,
                                                     args.batch, 0.85, args.cpu)

    (base / "verify_gap_follow.json").write_text(json.dumps(out, indent=2))
    print("\nSalvato verify_gap_follow.json")


if __name__ == "__main__":
    main()


<a id="flickr"></a>
## 7. Flickr: personalised contact recommendation

Initial graph: 2 November 2006. Evaluation: new links observed through 3 December
2006, between users already present at the initial date. Sample: 5,000 users with
at least ten initial contacts, excluding the 200 pilot users. The sampling seed,
dates, float64 computation, stopping threshold and tie-break rules are retained.
Future links are used only to evaluate recommendations, not to select users or rank
candidates. The pilot sample is reconstructed without rerunning its 200-user study.

The implementation retains five methods: global PageRank, incoming degree, PPR,
degree with reciprocity, and PPR with reciprocity. The thesis table reports four
of these. PPR teleportation and dangling mass both return to the query user.
The hybrid first ranks unfollowed followers by PPR and fills remaining positions
from the ordinary PPR ranking. Ties use ascending original user identifiers.

Results are checkpointed in blocks of 100 users. Precision@10 and HitRate@10 use
all 5,000 users. Confidence intervals use 10,000 paired user-level bootstrap samples.
The following cells contain the complete preparation and evaluation code; the
execution cell in Section 9 runs them sequentially in one isolated process.


### Environment


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_00.py
from IPython.display import display
import sys
import numpy as np
import scipy
import cupy as cp

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("CuPy:", cp.__version__)

device = cp.cuda.runtime.getDeviceProperties(0)
name = device["name"]
print("GPU:", name.decode() if isinstance(name, bytes) else name)

free, total = cp.cuda.runtime.memGetInfo()
print(f"Memoria GPU: {free / 1e9:.2f} GB liberi / {total / 1e9:.2f} GB")


### Download


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_01.py
from pathlib import Path
import requests
import gzip
import itertools

DATA_DIR = Path.cwd() / "flickr_ppr"
DATA_DIR.mkdir(exist_ok=True)

DATA_FILE = DATA_DIR / "flickr-growth.txt.gz"
URL = "https://socialnetworks.mpi-sws.org/data/flickr-growth.txt.gz"

if not DATA_FILE.exists():
    partial = DATA_FILE.with_suffix(".gz.part")

    with requests.get(URL, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()

        downloaded = 0
        last_report = 0

        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                output.write(chunk)
                downloaded += len(chunk)

                if downloaded - last_report >= 25 * 1024 * 1024:
                    print(f"Scaricati {downloaded / 1e6:.1f} MB")
                    last_report = downloaded

    # Verifica che la risposta sia un file gzip, non una pagina HTML.
    with partial.open("rb") as source:
        if source.read(2) != b"\x1f\x8b":
            raise ValueError("Il server non ha restituito un file gzip.")

    partial.replace(DATA_FILE)

print(f"\nFile: {DATA_FILE}")
print(f"Dimensione compressa: {DATA_FILE.stat().st_size / 1e6:.1f} MB")
print("\nPrime dieci righe:")

with gzip.open(DATA_FILE, "rt", encoding="ascii") as source:
    for line in itertools.islice(source, 10):
        print(repr(line.rstrip("\n")))


### Parse dated edges


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_02.py
import pandas as pd
import numpy as np
import json
from collections import Counter

EDGE_FILE = DATA_DIR / "edges_int32.bin"
PARTIAL_FILE = DATA_DIR / "edges_int32.bin.part"

date_counts = Counter()
total_edges = 0
self_loops = 0
min_id = None
max_id = None

chunks = pd.read_csv(
    DATA_FILE,
    sep="\t",
    header=None,
    names=["source", "target", "date"],
    dtype={"source": np.int64, "target": np.int64, "date": "string"},
    chunksize=1_000_000,
    compression="gzip",
)

with PARTIAL_FILE.open("wb") as output:
    for chunk in chunks:
        source = chunk["source"].to_numpy()
        target = chunk["target"].to_numpy()

        low = int(min(source.min(), target.min()))
        high = int(max(source.max(), target.max()))

        if low < 0 or high > np.iinfo(np.int32).max:
            raise ValueError("Identificativi fuori dal formato int32.")

        days = (
            pd.to_datetime(chunk["date"], format="%Y-%m-%d", errors="raise")
            .to_numpy(dtype="datetime64[D]")
            .astype(np.int32)
        )

        # Ogni record: sorgente, destinazione, giorno dal 1970-01-01.
        records = np.empty((len(chunk), 3), dtype=np.int32)
        records[:, 0] = source
        records[:, 1] = target
        records[:, 2] = days
        records.tofile(output)

        dates, counts = np.unique(days, return_counts=True)
        date_counts.update({
            int(day): int(count) for day, count in zip(dates, counts)
        })

        total_edges += len(chunk)
        self_loops += int(np.count_nonzero(source == target))
        min_id = low if min_id is None else min(min_id, low)
        max_id = high if max_id is None else max(max_id, high)

        if total_edges % 5_000_000 == 0:
            print(f"Lette {total_edges:,} righe")

PARTIAL_FILE.replace(EDGE_FILE)

def date_string(day):
    return str(np.datetime64("1970-01-01", "D") + np.timedelta64(day, "D"))

ordered_days = sorted(date_counts)
first_day = ordered_days[0]

metadata = {
    "records": total_edges,
    "min_id": min_id,
    "max_id": max_id,
    "self_loops": self_loops,
    "date_counts": {
        date_string(day): date_counts[day] for day in ordered_days
    },
}
(DATA_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print("\nRighe complessive:", f"{total_edges:,}")
print("Intervallo identificativi:", min_id, "—", max_id)
print("Autoarchi:", f"{self_loops:,}")
print("Date distinte:", len(ordered_days))
print("Prima data:", date_string(first_day))
print("Ultima data:", date_string(ordered_days[-1]))
print("Archi osservati alla prima data:", f"{date_counts[first_day]:,}")
print(f"File binario: {EDGE_FILE.stat().st_size / 1e6:.1f} MB")

print("\nPrime cinque date:")
for day in ordered_days[:5]:
    print(date_string(day), f"{date_counts[day]:,}")

print("\nUltime cinque date:")
for day in ordered_days[-5:]:
    print(date_string(day), f"{date_counts[day]:,}")


### Initial and future graphs


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_03.py
from scipy import sparse
import time

t_start = time.perf_counter()

edges = np.memmap(
    EDGE_FILE,
    dtype=np.int32,
    mode="r",
    shape=(total_edges, 3),
)

T0 = np.datetime64("2006-11-02", "D")
T1 = np.datetime64("2006-12-03", "D")
t0_day = int(T0.astype(np.int64))
t1_day = int(T1.astype(np.int64))

# Grafo iniziale: collegamenti osservati entro t0.
initial_edges = edges[edges[:, 2] <= t0_day, :2].copy()

# Identificativi originali degli utenti presenti nel grafo iniziale.
user_ids = np.unique(initial_edges)
n = len(user_ids)

# Conversione dagli identificativi originali a indici 0, ..., n-1.
id_map = np.full(max_id + 1, -1, dtype=np.int32)
id_map[user_ids] = np.arange(n, dtype=np.int32)

src = id_map[initial_edges[:, 0]]
dst = id_map[initial_edges[:, 1]]
raw_initial_count = len(src)

# G0[i, j] = 1 significa: i ha dichiarato j come contatto.
G0 = sparse.csr_matrix(
    (np.ones(len(src), dtype=np.float32), (src, dst)),
    shape=(n, n),
)
G0.sum_duplicates()
G0.sort_indices()

initial_duplicates = raw_initial_count - G0.nnz
G0.data.fill(1.0)  # Una relazione per coppia, anche se duplicata nel file.

del initial_edges, src, dst

out_degree = np.diff(G0.indptr).astype(np.int32)
in_degree = np.asarray(G0.sum(axis=0)).ravel().astype(np.int32)

# Righe della trasposta = liste dei collegamenti entranti.
GT0 = G0.T.tocsr()
GT0.sort_indices()

# Collegamenti osservati successivamente, fino a t1.
future_edges = edges[
    (edges[:, 2] > t0_day) & (edges[:, 2] <= t1_day), :2
].copy()

raw_future_count = len(future_edges)
src = id_map[future_edges[:, 0]]
dst = id_map[future_edges[:, 1]]

# Entrambi gli utenti devono essere presenti a t0.
present = (src >= 0) & (dst >= 0)
initial_users_count = int(present.sum())

# Non consideriamo un autoarco come obiettivo di raccomandazione.
valid = present & (src != dst)

future_graph = sparse.csr_matrix(
    (np.ones(int(valid.sum()), dtype=np.float32),
     (src[valid], dst[valid])),
    shape=(n, n),
)
future_graph.sum_duplicates()
future_graph.sort_indices()

future_duplicates = int(valid.sum()) - future_graph.nnz
future_graph.data.fill(1.0)

# Un arco di verifica deve essere assente nel grafo iniziale.
overlap = future_graph.multiply(G0)
already_present = overlap.nnz

Gnew = (future_graph - overlap).tocsr()
Gnew.eliminate_zeros()
Gnew.sort_indices()

del future_edges, src, dst, present, valid, future_graph, overlap

new_out_degree = np.diff(Gnew.indptr)

# u -> w è reciproco se w -> u esisteva già a t0.
reciprocal_count = Gnew.multiply(G0.T).nnz

def csr_mb(matrix):
    return (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    ) / 1e6

print(f"Periodo: {T0} → {T1}")
print(f"Utenti presenti a t0: {n:,}")
print(f"Archi distinti a t0: {G0.nnz:,}")
print(f"Duplicati iniziali: {initial_duplicates:,}")
print(f"Autoarchi iniziali: {np.count_nonzero(G0.diagonal()):,}")
print(f"Nodi penzolanti a t0: {np.count_nonzero(out_degree == 0):,}")

print(f"\nRighe nel periodo successivo: {raw_future_count:,}")
print(f"Fra utenti già presenti a t0: {initial_users_count:,}")
print(f"Duplicati successivi, esclusi autoarchi: {future_duplicates:,}")
print(f"Collegamenti successivi già presenti a t0: {already_present:,}")
print(f"Nuovi collegamenti di verifica: {Gnew.nnz:,}")
print(f"Di cui reciproci rispetto a t0: {reciprocal_count:,}")

eligible = out_degree >= 10
print(f"\nUtenti con almeno 10 contatti iniziali: {eligible.sum():,}")
print(
    "Di questi, con almeno un nuovo collegamento di verifica:",
    f"{np.count_nonzero(eligible & (new_out_degree > 0)):,}",
)

print(f"\nMemoria CSR uscente: {csr_mb(G0):.1f} MB")
print(f"Memoria CSR entrante: {csr_mb(GT0):.1f} MB")
print(f"Preparazione CPU: {time.perf_counter() - t_start:.1f} s")


### PPR implementation, numerical checks and pilot sample


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_04.py
import cupy as cp
from cupyx.scipy import sparse as gpu_sparse

def prepare_gpu_graph(graph):
    degrees = np.diff(graph.indptr)
    inv_degrees = np.zeros(graph.shape[0], dtype=np.float64)
    np.divide(1.0, degrees, out=inv_degrees, where=degrees > 0)

    incoming = gpu_sparse.csr_matrix(
        graph.T.tocsr().astype(np.float64)
    )
    return incoming, cp.asarray(inv_degrees), cp.asarray(degrees == 0)


def pagerank_gpu(incoming, inv_degrees, dangling,
                 user=None, alpha=0.85, tol=1e-8, max_iter=1000):
    """
    user=None: PageRank globale con ripartenza uniforme.
    user=int: Personalized PageRank con ripartenza su quell'utente.
              L'indice è interno al grafo, non l'ID originale.
    tol: soglia sulla differenza in norma 1 fra iterazioni.
    """
    n_local = incoming.shape[0]

    if not (0 < alpha < 1) or tol <= 0:
        raise ValueError("alpha o tolleranza non validi.")
    if user is not None:
        user = int(user)
        if not 0 <= user < n_local:
            raise ValueError("Indice utente non valido.")

    cp.cuda.get_current_stream().synchronize()
    start = time.perf_counter()

    rank = cp.full(n_local, 1.0 / n_local, dtype=cp.float64)

    for iteration in range(1, max_iter + 1):
        dangling_mass = cp.sum(rank[dangling], dtype=cp.float64)

        new_rank = alpha * incoming.dot(rank * inv_degrees)
        restart_mass = (1.0 - alpha) + alpha * dangling_mass

        if user is None:
            new_rank += restart_mass / n_local
        else:
            new_rank[user] += restart_mass

        delta = float(
            cp.sum(cp.abs(new_rank - rank), dtype=cp.float64).item()
        )
        rank = new_rank

        if not np.isfinite(delta):
            raise FloatingPointError("Scarto non finito.")
        if delta < tol:
            break
    else:
        raise RuntimeError(f"Nessun arresto entro {max_iter} iterazioni.")

    cp.cuda.get_current_stream().synchronize()
    elapsed = time.perf_counter() - start

    return rank, {
        "iterations": iteration,
        "delta": delta,
        "seconds": elapsed,
    }


# Verifica su un grafo di cinque nodi, due dei quali penzolanti.
toy = sparse.csr_matrix(
    (np.ones(4), ([0, 0, 1, 3], [1, 2, 2, 0])),
    shape=(5, 5),
)
toy_gpu = prepare_gpu_graph(toy)

toy_degrees = np.diff(toy.indptr)
toy_inv = np.zeros(5)
np.divide(1.0, toy_degrees, out=toy_inv, where=toy_degrees > 0)
toy_links = toy.toarray() * toy_inv[:, None]

for seed in (None, 0, 2):
    v = np.full(5, 1.0 / 5)
    if seed is not None:
        v.fill(0)
        v[seed] = 1.0

    transition = toy_links.copy()
    transition[toy_degrees == 0, :] = v

    reference = np.linalg.solve(
        np.eye(5) - 0.85 * transition.T,
        (1.0 - 0.85) * v,
    )
    calculated, info = pagerank_gpu(*toy_gpu, user=seed, tol=1e-12)
    error = np.linalg.norm(cp.asnumpy(calculated) - reference, ord=1)

    assert error < 1e-10, f"Verifica fallita: {error}"
    print(f"Verifica seed={seed}: errore L1 = {error:.3e}")

print("\nVerifiche superate.")

# Preparazione del grafo Flickr.
incoming_gpu, inv_degrees_gpu, dangling_gpu = prepare_gpu_graph(G0)

# Utenti scelti usando soltanto il grafo iniziale.
rng = np.random.default_rng(2026)
query_users = rng.choice(
    np.flatnonzero(out_degree >= 10),
    size=200,
    replace=False,
)
np.save(DATA_DIR / "query_user_ids.npy", user_ids[query_users])

# Prima esecuzione per compilazione e riscaldamento.
warm_rank, _ = pagerank_gpu(
    incoming_gpu, inv_degrees_gpu, dangling_gpu,
    user=int(query_users[0]),
)
del warm_rank

global_rank_gpu, global_info = pagerank_gpu(
    incoming_gpu, inv_degrees_gpu, dangling_gpu,
)

first_user = int(query_users[0])
personal_rank_gpu, personal_info = pagerank_gpu(
    incoming_gpu, inv_degrees_gpu, dangling_gpu,
    user=first_user,
)

print("\nPageRank globale:", global_info)
print("Somma dei punteggi:", float(global_rank_gpu.sum().item()))

print("\nUtente di prova, ID originale:", int(user_ids[first_user]))
print("Contatti iniziali:", int(out_degree[first_user]))
print("Personalized PageRank:", personal_info)
print("Somma dei punteggi:", float(personal_rank_gpu.sum().item()))
print("Punteggio minimo:", float(personal_rank_gpu.min().item()))


### Ranking and evaluation helpers


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_05.py
K = 10

def excluded_users(user):
    followed = G0.indices[G0.indptr[user]:G0.indptr[user + 1]]
    return np.union1d(followed, [user]).astype(np.int32)


def top_k_gpu(scores, excluded, k=K):
    """Esclude i contatti noti; a parità di punteggio precede l'ID minore."""
    k = min(k, scores.size - len(excluded))
    if k <= 0:
        return np.empty(0, dtype=np.int32)

    filtered = scores.copy()
    filtered[cp.asarray(excluded)] = -cp.inf

    threshold = cp.partition(filtered, filtered.size - k)[-k].item()

    above = cp.flatnonzero(filtered > threshold)
    tied = cp.flatnonzero(filtered == threshold)
    selected = cp.concatenate((above, tied[:k - above.size]))

    selected_cpu = cp.asnumpy(selected)
    values_cpu = cp.asnumpy(filtered[selected])
    order = np.lexsort((selected_cpu, -values_cpu))
    return selected_cpu[order].astype(np.int32)


# Controllo dello spareggio e dell'esclusione.
assert np.array_equal(
    top_k_gpu(
        cp.asarray([0.5, 0.5, 0.3, 0.5], dtype=cp.float64),
        np.asarray([0], dtype=np.int32),
        k=2,
    ),
    [1, 3],
)

degree_gpu = cp.asarray(in_degree, dtype=cp.float64)


def reciprocal_recommendations(user, excluded, popular, k=K):
    followers = GT0.indices[GT0.indptr[user]:GT0.indptr[user + 1]]
    candidates = np.setdiff1d(followers, excluded, assume_unique=True)

    # Grado entrante decrescente, poi ID crescente.
    order = np.lexsort((candidates, -in_degree[candidates]))
    selected = candidates[order[:k]].tolist()

    # Completamento fino a k, evitando ripetizioni.
    for candidate in popular:
        if len(selected) >= k:
            break
        if int(candidate) not in selected:
            selected.append(int(candidate))

    return np.asarray(selected, dtype=np.int32)


def evaluate_recommendations(user, recommendations):
    truth = Gnew.indices[Gnew.indptr[user]:Gnew.indptr[user + 1]]
    followers = GT0.indices[GT0.indptr[user]:GT0.indptr[user + 1]]

    hits = np.intersect1d(recommendations, truth, assume_unique=True)
    reciprocal_truth = np.intersect1d(truth, followers, assume_unique=True)
    reciprocal_hits = np.intersect1d(
        hits, reciprocal_truth, assume_unique=True
    )

    return {
        "hits": len(hits),
        "precision_at_10": len(hits) / K,
        "recall_at_10": len(hits) / len(truth) if len(truth) else np.nan,
        "hits_reciprocal": len(reciprocal_hits),
        "hits_nonreciprocal": len(hits) - len(reciprocal_hits),
        "new_links": len(truth),
        "new_reciprocal_links": len(reciprocal_truth),
    }


user = first_user
excluded = excluded_users(user)

popular = top_k_gpu(degree_gpu, excluded)

recommendations = {
    "Personalized PageRank": top_k_gpu(personal_rank_gpu, excluded),
    "PageRank globale": top_k_gpu(global_rank_gpu, excluded),
    "Grado entrante": popular,
    "Reciprocità + grado": reciprocal_recommendations(
        user, excluded, popular
    ),
}

rows = []
for method, suggested in recommendations.items():
    assert len(suggested) == K
    assert len(np.unique(suggested)) == K
    assert not np.intersect1d(suggested, excluded).size

    rows.append({
        "method": method,
        **evaluate_recommendations(user, suggested),
    })

print("Utente, ID originale:", int(user_ids[user]))
display(pd.DataFrame(rows).set_index("method"))

print("\nSuggerimenti, con identificativi originali:")
for method, suggested in recommendations.items():
    print(method, ":", user_ids[suggested].tolist())


### Confirmation sample


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_08.py
from pathlib import Path
import numpy as np

CONFIRM_DIR = Path(DATA_DIR) / "flickr_5000"
CONFIRM_DIR.mkdir(exist_ok=True)

N_CONFIRM = 5_000
CONFIRM_SEED = 20261004

pilot_ids = np.load(Path(DATA_DIR) / "query_user_ids.npy")
initial_degree = np.diff(G0.indptr)

eligible_confirm = np.flatnonzero(
    (initial_degree >= 10)
    & ~np.isin(user_ids, pilot_ids)
)

rng_confirm = np.random.default_rng(CONFIRM_SEED)
confirm_users = rng_confirm.choice(
    eligible_confirm, size=N_CONFIRM, replace=False
)

confirm_ids = user_ids[confirm_users]
sample_file = CONFIRM_DIR / "query_user_ids.npy"

if sample_file.exists():
    assert np.array_equal(np.load(sample_file), confirm_ids), (
        "La cartella contiene un campione diverso."
    )
else:
    np.save(sample_file, confirm_ids)

assert np.intersect1d(confirm_ids, pilot_ids).size == 0
assert np.unique(confirm_ids).size == N_CONFIRM

print("Utenti selezionati:", len(confirm_users))
print("Sovrapposizioni con il campione preliminare: 0")
print("Seme:", CONFIRM_SEED)
print("Campione salvato in:", sample_file)


### Hybrid method and protocol


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_09.py
import json

def _hybrid_from_followers(
    scores, followers, excluded, ppr_top, k=10
):
    candidates = np.setdiff1d(
        followers, excluded, assume_unique=True
    )

    if candidates.size:
        values = cp.asnumpy(scores[cp.asarray(candidates)])
        # Punteggio decrescente; a parità, identificativo crescente.
        order = np.lexsort((candidates, -values))
        selected = candidates[order[:k]].tolist()
    else:
        selected = []

    seen = set(selected)

    if len(selected) < k:
        for node in ppr_top:
            node = int(node)
            if node not in seen:
                selected.append(node)
                seen.add(node)
            if len(selected) == k:
                break

    if len(selected) != k:
        raise RuntimeError("Non sono disponibili abbastanza suggerimenti.")

    result = np.asarray(selected, dtype=np.int32)
    assert np.unique(result).size == k
    assert np.intersect1d(result, excluded).size == 0
    return result


def hybrid_recommendations(
    user, scores, excluded, ppr_top, k=10
):
    followers = GT0.indices[
        GT0.indptr[user]:GT0.indptr[user + 1]
    ]
    return _hybrid_from_followers(
        scores, followers, excluded, ppr_top, k
    )


# Gli indici interni seguono l'ordine degli identificativi originali.
assert np.all(user_ids[1:] > user_ids[:-1])

# Verifica: priorità ai follower, spareggio e completamento.
toy_scores = cp.asarray(
    [0.0, 0.3, 0.8, 0.1, 0.3, 0.9],
    dtype=cp.float64
)
toy_followers = np.asarray([0, 1, 2, 4], dtype=np.int32)
toy_excluded = np.asarray([0, 2], dtype=np.int32)
toy_ppr_top = np.asarray([5, 1, 4, 3], dtype=np.int32)

assert np.array_equal(
    _hybrid_from_followers(
        toy_scores, toy_followers, toy_excluded, toy_ppr_top, k=4
    ),
    [1, 4, 5, 3]
)

assert np.array_equal(
    _hybrid_from_followers(
        toy_scores,
        np.asarray([], dtype=np.int32),
        toy_excluded,
        toy_ppr_top,
        k=4
    ),
    toy_ppr_top
)

METHODS_CONFIRM = [
    "Personalized PageRank",
    "PageRank globale",
    "Grado entrante",
    "Reciprocità + grado",
    "Reciprocità + PPR",
]

assert str(T0) == "2006-11-02"
assert str(T1) == "2006-12-03"

protocol_confirm = {
    "dataset": "Flickr growth",
    "initial_date": str(T0),
    "evaluation_end": str(T1),
    "initial_nodes": int(G0.shape[0]),
    "initial_edges": int(G0.nnz),
    "n_users": N_CONFIRM,
    "sampling_seed": CONFIRM_SEED,
    "minimum_initial_out_degree": 10,
    "pilot_users_excluded": int(len(pilot_ids)),
    "sampling_uses_future_links": False,
    "k": 10,
    "alpha": 0.85,
    "precision": "float64",
    "stopping_rule": "L1 increment < 1e-8",
    "maximum_iterations": 1000,
    "ppr_dangling_mass": "return to the query user",
    "global_dangling_mass": "uniform redistribution",
    "excluded_candidates": "query user and initial outgoing contacts",
    "tie_break": "ascending original user ID",
    "reciprocal_definition": "reverse edge present at the initial date",
    "hybrid_rule": (
        "Unfollowed initial followers ranked by PPR; "
        "remaining positions filled from the PPR ranking."
    ),
    "methods": METHODS_CONFIRM,
    "precision_population": "all 5000 sampled users",
    "recall_population": "sampled users with at least one evaluation link",
    "component_precision_denominator": "5000 * 10 suggestions",
    "bootstrap_resamples": 10000,
    "bootstrap_seed": 20261005,
}

protocol_path = CONFIRM_DIR / "recommendation_protocol.json"

if protocol_path.exists():
    saved_protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
    assert saved_protocol == protocol_confirm, (
        "Esiste già un protocollo diverso nella cartella."
    )
else:
    protocol_path.write_text(
        json.dumps(protocol_confirm, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

print("Verifiche del metodo combinato superate.")
print("Metodi:", len(METHODS_CONFIRM))
print("Protocollo salvato in:", protocol_path)


### Checkpointed evaluation


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_10.py
import time
import pandas as pd
from IPython.display import display

K_CONFIRM = int(protocol_confirm["k"])
BLOCK_SIZE = 100
BLOCK_DIR = CONFIRM_DIR / "blocks"
BLOCK_DIR.mkdir(exist_ok=True)

assert np.array_equal(
    np.load(CONFIRM_DIR / "query_user_ids.npy"),
    user_ids[confirm_users]
)
assert json.loads(
    protocol_path.read_text(encoding="utf-8")
) == protocol_confirm

confirm_rows = []
confirm_timing_rows = []
pending_blocks = []

# Recupera gli eventuali blocchi già completati.
for start in range(0, N_CONFIRM, BLOCK_SIZE):
    stop = min(start + BLOCK_SIZE, N_CONFIRM)
    path = BLOCK_DIR / f"block_{start:05d}_{stop:05d}.json"
    expected_ids = user_ids[confirm_users[start:stop]].tolist()

    if path.exists():
        block = json.loads(path.read_text(encoding="utf-8"))
        assert block["user_ids"] == expected_ids
        assert [
            (row["user_id"], row["method"])
            for row in block["results"]
        ] == [
            (uid, method)
            for uid in expected_ids
            for method in METHODS_CONFIRM
        ]
        assert [
            row["user_id"] for row in block["timings"]
        ] == expected_ids

        confirm_rows.extend(block["results"])
        confirm_timing_rows.extend(block["timings"])
    else:
        pending_blocks.append((start, stop, path))

loaded_count = len(confirm_timing_rows)
print(f"Utenti già salvati: {loaded_count}/{N_CONFIRM}")

# Punteggi delle due alternative globali.
global_scores_confirm, global_info_confirm = pagerank_gpu(
    incoming_gpu, inv_degrees_gpu, dangling_gpu,
    user=None,
    alpha=protocol_confirm["alpha"],
    tol=1e-8,
    max_iter=protocol_confirm["maximum_iterations"]
)
degree_scores_confirm = cp.asarray(in_degree, dtype=cp.float64)

# Riscaldamento prima delle misure.
if pending_blocks:
    warm_user = int(confirm_users[pending_blocks[0][0]])
    warm_excluded = np.union1d(
        G0.indices[G0.indptr[warm_user]:G0.indptr[warm_user + 1]],
        np.asarray([warm_user], dtype=np.int32)
    )
    warm_rank, _ = pagerank_gpu(
        incoming_gpu, inv_degrees_gpu, dangling_gpu,
        user=warm_user,
        alpha=protocol_confirm["alpha"],
        tol=1e-8,
        max_iter=protocol_confirm["maximum_iterations"]
    )
    warm_top = top_k_gpu(warm_rank, warm_excluded, k=K_CONFIRM)
    hybrid_recommendations(
        warm_user, warm_rank, warm_excluded, warm_top, k=K_CONFIRM
    )
    cp.cuda.get_current_stream().synchronize()
    del warm_rank

run_start = time.perf_counter()
new_count = 0

for start, stop, block_path in pending_blocks:
    block_rows = []
    block_timings = []

    for internal_user in confirm_users[start:stop]:
        confirm_user = int(internal_user)
        original_id = int(user_ids[confirm_user])

        cp.cuda.get_current_stream().synchronize()
        query_start = time.perf_counter()

        followed = G0.indices[
            G0.indptr[confirm_user]:G0.indptr[confirm_user + 1]
        ]
        excluded = np.union1d(
            followed,
            np.asarray([confirm_user], dtype=np.int32)
        )

        rank_confirm, info_confirm = pagerank_gpu(
            incoming_gpu, inv_degrees_gpu, dangling_gpu,
            user=confirm_user,
            alpha=protocol_confirm["alpha"],
            tol=1e-8,
            max_iter=protocol_confirm["maximum_iterations"]
        )
        ppr_top = top_k_gpu(
            rank_confirm, excluded, k=K_CONFIRM
        )
        cp.cuda.get_current_stream().synchronize()
        ppr_query_seconds = time.perf_counter() - query_start

        hybrid_start = time.perf_counter()
        hybrid_top = hybrid_recommendations(
            confirm_user, rank_confirm, excluded,
            ppr_top, k=K_CONFIRM
        )
        cp.cuda.get_current_stream().synchronize()
        hybrid_selection_seconds = time.perf_counter() - hybrid_start

        mass_error = abs(float(cp.sum(rank_confirm).item()) - 1.0)
        assert mass_error < 1e-9

        global_top = top_k_gpu(
            global_scores_confirm, excluded, k=K_CONFIRM
        )
        degree_top = top_k_gpu(
            degree_scores_confirm, excluded, k=K_CONFIRM
        )

        followers = GT0.indices[
            GT0.indptr[confirm_user]:GT0.indptr[confirm_user + 1]
        ]
        reciprocal_degree_top = _hybrid_from_followers(
            degree_scores_confirm, followers,
            excluded, degree_top, k=K_CONFIRM
        )

        recommendations = [
            ppr_top,
            global_top,
            degree_top,
            reciprocal_degree_top,
            hybrid_top,
        ]

        # I collegamenti futuri vengono usati solo per la valutazione.
        truth = Gnew.indices[
            Gnew.indptr[confirm_user]:Gnew.indptr[confirm_user + 1]
        ]
        reciprocal_truth = np.intersect1d(
            truth, followers, assume_unique=True
        )

        for method, suggested in zip(
            METHODS_CONFIRM, recommendations
        ):
            assert len(suggested) == K_CONFIRM
            assert np.unique(suggested).size == K_CONFIRM
            assert np.intersect1d(suggested, excluded).size == 0

            matched = np.intersect1d(
                suggested, truth, assume_unique=True
            )
            reciprocal_matched = np.intersect1d(
                matched, reciprocal_truth, assume_unique=True
            )

            hits = int(matched.size)
            hits_reciprocal = int(reciprocal_matched.size)
            hits_nonreciprocal = hits - hits_reciprocal

            block_rows.append({
                "user_id": original_id,
                "method": method,
                "hits": hits,
                "precision_at_10": hits / K_CONFIRM,
                "precision_reciprocal_at_10": (
                    hits_reciprocal / K_CONFIRM
                ),
                "precision_nonreciprocal_at_10": (
                    hits_nonreciprocal / K_CONFIRM
                ),
                "recall_at_10": (
                    hits / int(truth.size) if truth.size else None
                ),
                "hit_at_10": int(hits > 0),
                "hits_reciprocal": hits_reciprocal,
                "hits_nonreciprocal": hits_nonreciprocal,
                "new_links": int(truth.size),
                "new_reciprocal_links": int(reciprocal_truth.size),
                "recommended_ids": json.dumps(
                    user_ids[suggested].tolist()
                ),
            })

        block_timings.append({
            "user_id": original_id,
            "initial_out_degree": int(followed.size),
            "iterations": int(info_confirm["iterations"]),
            "delta": float(info_confirm["delta"]),
            "pagerank_seconds": float(info_confirm["seconds"]),
            "ppr_query_seconds": ppr_query_seconds,
            "hybrid_selection_seconds": hybrid_selection_seconds,
            "hybrid_query_seconds": (
                ppr_query_seconds + hybrid_selection_seconds
            ),
            "mass_error": mass_error,
        })

        del rank_confirm

    block = {
        "user_ids": user_ids[confirm_users[start:stop]].tolist(),
        "results": block_rows,
        "timings": block_timings,
    }
    temporary = block_path.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(block, ensure_ascii=False, allow_nan=False),
        encoding="utf-8"
    )
    temporary.replace(block_path)

    confirm_rows.extend(block_rows)
    confirm_timing_rows.extend(block_timings)
    new_count += stop - start

    elapsed = time.perf_counter() - run_start
    completed = loaded_count + new_count
    remaining_seconds = (
        elapsed / new_count * (N_CONFIRM - completed)
    )
    print(
        f"{completed}/{N_CONFIRM} utenti salvati"
        f" — {elapsed / 60:.1f} min"
        f" — residui stimati: {remaining_seconds / 60:.1f} min",
        flush=True
    )

results_confirm_df = pd.DataFrame(confirm_rows)
timings_confirm_df = pd.DataFrame(confirm_timing_rows)

assert len(results_confirm_df) == N_CONFIRM * len(METHODS_CONFIRM)
assert not results_confirm_df.duplicated(["user_id", "method"]).any()
assert len(timings_confirm_df) == N_CONFIRM
assert timings_confirm_df["user_id"].nunique() == N_CONFIRM

summary_confirm = (
    results_confirm_df.groupby("method")
    .agg(
        precision_at_10=("precision_at_10", "mean"),
        precision_reciprocal_at_10=("precision_reciprocal_at_10", "mean"),
        precision_nonreciprocal_at_10=(
            "precision_nonreciprocal_at_10", "mean"
        ),
        recall_at_10=("recall_at_10", "mean"),
        hit_rate_at_10=("hit_at_10", "mean"),
        hits=("hits", "sum"),
        hits_reciprocal=("hits_reciprocal", "sum"),
        hits_nonreciprocal=("hits_nonreciprocal", "sum"),
    )
    .reindex(METHODS_CONFIRM)
)

assert np.allclose(
    summary_confirm["precision_at_10"],
    summary_confirm["precision_reciprocal_at_10"]
    + summary_confirm["precision_nonreciprocal_at_10"]
)

results_confirm_df.to_csv(
    CONFIRM_DIR / "recommendation_results.csv", index=False
)
timings_confirm_df.to_csv(
    CONFIRM_DIR / "recommendation_timings.csv", index=False
)
summary_confirm.to_csv(
    CONFIRM_DIR / "recommendation_summary.csv"
)

query_population = results_confirm_df[
    results_confirm_df["method"] == METHODS_CONFIRM[0]
]

print("\nEsperimento completato.")
print(
    "Utenti con nuovi collegamenti:",
    int((query_population["new_links"] > 0).sum()),
    "/", N_CONFIRM
)
print("Collegamenti di verifica:", int(query_population["new_links"].sum()))
print(
    "Di cui reciproci:",
    int(query_population["new_reciprocal_links"].sum())
)

print("\nMetriche in percentuale:")
display(
    summary_confirm[[
        "precision_at_10",
        "precision_reciprocal_at_10",
        "precision_nonreciprocal_at_10",
        "recall_at_10",
        "hit_rate_at_10",
    ]].mul(100).round(3)
)

print("\nCollegamenti recuperati:")
display(
    summary_confirm[[
        "hits", "hits_reciprocal", "hits_nonreciprocal"
    ]]
)

print("\nTempi delle query, in secondi:")
display(
    timings_confirm_df[[
        "pagerank_seconds",
        "ppr_query_seconds",
        "hybrid_query_seconds",
    ]].agg(["min", "median", "max"])
)

print("\nRisultati salvati in:", CONFIRM_DIR)


### Paired bootstrap for the five original methods


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_bootstrap.py
# Cella: bootstrap appaiato sui 5.000 utenti del campione di conferma.
# Usa solo i file salvati, quindi funziona anche dopo un riavvio della sessione.
# Nessuna operazione sulla GPU.
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CONFIRM_DIR = (Path.cwd() / "flickr_ppr" / "flickr_5000")  # cambiare se i file sono su Drive
RESULTS_PATH = CONFIRM_DIR / "recommendation_results.csv"

protocol = json.loads(
    (CONFIRM_DIR / "recommendation_protocol.json").read_text(encoding="utf-8")
)
METHODS = list(protocol["methods"])
N_USERS = int(protocol["n_users"])
K = int(protocol["k"])
BOOT_RESAMPLES = int(protocol["bootstrap_resamples"])  # 10000
BOOT_SEED = int(protocol["bootstrap_seed"])            # 20261005
CHUNK = 500  # ricampionamenti elaborati per volta, per limitare la memoria

results = pd.read_csv(RESULTS_PATH)
assert len(METHODS) == 5
assert set(results["method"]) == set(METHODS)
assert len(results) == N_USERS * len(METHODS)
assert not results.duplicated(["user_id", "method"]).any()


def user_by_method(column):
    """Una riga per utente, una colonna per metodo, nello stesso ordine."""
    table = results.pivot(index="user_id", columns="method", values=column)
    return table.reindex(columns=METHODS)


precision = user_by_method("precision_at_10")
users = precision.index
hit = user_by_method("hit_at_10").loc[users]
recall = user_by_method("recall_at_10").loc[users]

assert precision.shape == (N_USERS, len(METHODS))
assert not precision.isna().any().any()
assert not hit.isna().any().any()

P = precision.to_numpy(dtype=np.float64)
H = hit.to_numpy(dtype=np.float64)

# Il richiamo è definito solo per gli utenti con almeno un nuovo collegamento:
# per ogni utente deve essere definito per tutti i metodi o per nessuno.
assert (recall.isna().all(axis=1) == recall.isna().any(axis=1)).all()
active = recall.notna().all(axis=1).to_numpy()
R = recall.to_numpy(dtype=np.float64)[active]


def bootstrap_means(matrices, n_resamples, seed, chunk=CHUNK):
    """Ricampiona gli utenti con reinserimento.

    Lo stesso ricampionamento vale per tutti i metodi e per tutte le
    matrici passate insieme: il confronto è quindi appaiato per utente.
    """
    rng = np.random.default_rng(seed)
    n_rows = matrices[0].shape[0]
    assert all(m.shape[0] == n_rows for m in matrices)
    outputs = [np.empty((n_resamples, m.shape[1])) for m in matrices]

    done = 0
    while done < n_resamples:
        size = min(chunk, n_resamples - done)
        idx = rng.integers(0, n_rows, size=(size, n_rows))
        for out, m in zip(outputs, matrices):
            out[done:done + size] = m[idx].mean(axis=1)
        done += size
    return outputs


boot_P, boot_H = bootstrap_means([P, H], BOOT_RESAMPLES, BOOT_SEED)
(boot_R,) = bootstrap_means([R], BOOT_RESAMPLES, BOOT_SEED + 1)


def interval(samples):
    """Intervallo percentile al 95%."""
    return np.quantile(samples, [0.025, 0.975], axis=0)


p_low, p_high = interval(boot_P)
h_low, h_high = interval(boot_H)
r_low, r_high = interval(boot_R)

method_rows = []
for j, method in enumerate(METHODS):
    method_rows.append({
        "method": method,
        "precisione_pct": 100 * P[:, j].mean(),
        "precisione_low_pct": 100 * p_low[j],
        "precisione_high_pct": 100 * p_high[j],
        "utenti_con_successo_pct": 100 * H[:, j].mean(),
        "utenti_con_successo_low_pct": 100 * h_low[j],
        "utenti_con_successo_high_pct": 100 * h_high[j],
        "richiamo_attivi_pct": 100 * R[:, j].mean(),
        "richiamo_low_pct": 100 * r_low[j],
        "richiamo_high_pct": 100 * r_high[j],
        "collegamenti_recuperati": int(round(P[:, j].sum() * K)),
    })
methods_df = pd.DataFrame(method_rows).set_index("method")

# Controllo: i collegamenti ricostruiti dalla precisione coincidono con quelli salvati.
saved_hits = results.groupby("method")["hits"].sum().reindex(METHODS)
assert (methods_df["collegamenti_recuperati"] == saved_hits).all()

COMPARISONS = [
    # Metodo combinato contro ciascun metodo singolo.
    ("Reciprocità + PPR", "Personalized PageRank"),
    ("Reciprocità + PPR", "Reciprocità + grado"),
    ("Reciprocità + PPR", "PageRank globale"),
    ("Reciprocità + PPR", "Grado entrante"),
    # PageRank personalizzato contro la popolarità e contro la reciprocità.
    ("Personalized PageRank", "PageRank globale"),
    ("Personalized PageRank", "Grado entrante"),
    ("Personalized PageRank", "Reciprocità + grado"),
]
column = {method: j for j, method in enumerate(METHODS)}
assert all(a in column and b in column for a, b in COMPARISONS)

comparison_rows = []
for a, b in COMPARISONS:
    ja, jb = column[a], column[b]
    observed = P[:, ja].mean() - P[:, jb].mean()
    resampled = boot_P[:, ja] - boot_P[:, jb]
    low, high = np.quantile(resampled, [0.025, 0.975])
    comparison_rows.append({
        "confronto": f"{a} meno {b}",
        "differenza_punti_pct": 100 * observed,
        "low_95": 100 * low,
        "high_95": 100 * high,
        "differenza_collegamenti": int(round((P[:, ja].sum() - P[:, jb].sum()) * K)),
        "frazione_ricampionamenti_diff_non_positiva": float(np.mean(resampled <= 0)),
        "intervallo_esclude_zero": bool(low > 0 or high < 0),
    })
comparisons_df = pd.DataFrame(comparison_rows).set_index("confronto")

methods_df.to_csv(CONFIRM_DIR / "bootstrap_metodi.csv")
comparisons_df.to_csv(CONFIRM_DIR / "bootstrap_confronti.csv")

bootstrap_record = {
    "input": RESULTS_PATH.name,
    "input_sha256": hashlib.sha256(RESULTS_PATH.read_bytes()).hexdigest(),
    "users": N_USERS,
    "active_users_for_recall": int(active.sum()),
    "methods": METHODS,
    "resamples": BOOT_RESAMPLES,
    "seed_precision_and_hit_rate": BOOT_SEED,
    "seed_recall": BOOT_SEED + 1,
    "pairing": "same resampled users for all methods",
    "interval": "percentile, 2.5% and 97.5%",
}
(CONFIRM_DIR / "bootstrap_protocollo.json").write_text(
    json.dumps(bootstrap_record, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"Utenti: {N_USERS}; utenti attivi per il richiamo: {int(active.sum())}")
print(f"Ricampionamenti: {BOOT_RESAMPLES}; semi: {BOOT_SEED} e {BOOT_SEED + 1}")

print("\nMetodi, in percentuale, con intervalli al 95%:")
display(methods_df.round(3))

print("\nConfronti appaiati sulla precisione, in punti percentuali:")
display(comparisons_df.round(3))

print("\nFile salvati in:", CONFIRM_DIR)


<a id="exploratory"></a>
## 8. Optional exploratory analyses

These cells preserve the additional analyses from the final research notebook,
separately from the thesis comparison: fixed and adaptive reciprocity quotas,
their paired bootstrap, and a 300-user zero-score/tie audit. They are disabled by
default and run only after the confirmation experiment. Their outputs should not
be confused with the pre-existing five-method evaluation.


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_quotas.py
# Cella da eseguire nel notebook Colab dopo il confronto dei cinque metodi.
# Usa le liste salvate: non esegue operazioni sulla GPU.
import hashlib
import json
import time
import numpy as np
import pandas as pd
from IPython.display import display

QUOTA_METHODS = ["PPR + quota fissa (5)", "PPR + quota adattiva"]
ALL_METHODS_CONFIRM = METHODS_CONFIRM + QUOTA_METHODS
base_path = CONFIRM_DIR / "recommendation_results.csv"
base_results = pd.read_csv(base_path)
assert len(base_results) == N_CONFIRM * len(METHODS_CONFIRM)
assert not base_results.duplicated(["user_id", "method"]).any()

quota_protocol = {
    "base_protocol": "recommendation_protocol.json",
    "base_results_sha256": hashlib.sha256(base_path.read_bytes()).hexdigest(),
    "users": N_CONFIRM,
    "k": K_CONFIRM,
    "methods_added": QUOTA_METHODS,
    "profile": "fraction of initial outgoing contacts with a reverse edge",
    "self_links_in_profile": "excluded",
    "adaptive_requested_slots": "round(k * fraction), halves rounded upwards",
    "fixed_requested_slots": 5,
    "available_candidates": "initial followers not already followed, excluding self",
    "reserved_slots": "minimum of requested slots, k and available candidates",
    "ordering": "PPR order from the saved Reciprocity + PPR list",
    "completion": "saved PPR list, skipping duplicates",
    "quota_semantics": "minimum reservation; PPR completion may add other followers",
    "future_links_used_for_selection": False,
}
quota_protocol_path = CONFIRM_DIR / "quota_policy_protocol.json"
if quota_protocol_path.exists():
    assert json.loads(quota_protocol_path.read_text(encoding="utf-8")) == quota_protocol
else:
    quota_protocol_path.write_text(
        json.dumps(quota_protocol, indent=2, ensure_ascii=False), encoding="utf-8"
    )


def decode_saved_ranking(value):
    original = np.asarray(json.loads(value), dtype=np.int64)
    internal = np.searchsorted(user_ids, original)
    assert np.all(internal < user_ids.size)
    assert np.array_equal(user_ids[internal], original)
    assert len(internal) == np.unique(internal).size == K_CONFIRM
    return internal.astype(np.int32)


def ranking_with_quota(ppr_top, hybrid_top, available, requested):
    quota = min(K_CONFIRM, int(requested), int(available.size))
    head = hybrid_top[np.isin(hybrid_top, available)]
    assert head.size >= quota
    selected = head[:quota].tolist()
    seen = set(selected)
    if len(selected) < K_CONFIRM:
        for node in ppr_top:
            node = int(node)
            if node not in seen:
                selected.append(node)
                seen.add(node)
            if len(selected) == K_CONFIRM:
                break
    result = np.asarray(selected, dtype=np.int32)
    assert len(result) == np.unique(result).size == K_CONFIRM
    return result, quota


ppr_lookup = base_results[
    base_results["method"] == "Personalized PageRank"
].set_index("user_id")
hybrid_lookup = base_results[
    base_results["method"] == "Reciprocità + PPR"
].set_index("user_id")
assert set(ppr_lookup.index) == set(user_ids[confirm_users].tolist())
assert set(hybrid_lookup.index) == set(ppr_lookup.index)

quota_rows = []
quota_diagnostics = []
quota_start = time.perf_counter()

for internal_user in confirm_users:
    u = int(internal_user)
    original_id = int(user_ids[u])
    base_row = ppr_lookup.loc[original_id]
    ppr_top = decode_saved_ranking(base_row["recommended_ids"])
    hybrid_top = decode_saved_ranking(
        hybrid_lookup.loc[original_id, "recommended_ids"]
    )

    followed = G0.indices[G0.indptr[u]:G0.indptr[u + 1]]
    followers = GT0.indices[GT0.indptr[u]:GT0.indptr[u + 1]]
    followed = followed[followed != u]
    followers = followers[followers != u]
    excluded = np.union1d(followed, np.asarray([u], dtype=np.int32))
    available = np.setdiff1d(followers, excluded, assume_unique=True)
    mutual_count = int(np.intersect1d(followed, followers, assume_unique=True).size)
    degree = int(followed.size)
    fraction = mutual_count / degree if degree else 0.0
    requested_adaptive = (
        (2 * K_CONFIRM * mutual_count + degree) // (2 * degree) if degree else 0
    )

    fixed_top, fixed_quota = ranking_with_quota(ppr_top, hybrid_top, available, 5)
    adaptive_top, adaptive_quota = ranking_with_quota(
        ppr_top, hybrid_top, available, requested_adaptive
    )
    # Le quote estreme devono riprodurre i due metodi salvati.
    assert np.array_equal(ranking_with_quota(ppr_top, hybrid_top, available, 0)[0], ppr_top)
    assert np.array_equal(
        ranking_with_quota(ppr_top, hybrid_top, available, K_CONFIRM)[0], hybrid_top
    )

    # Valutazione successiva alla generazione delle liste.
    truth = Gnew.indices[Gnew.indptr[u]:Gnew.indptr[u + 1]]
    reciprocal_truth = np.intersect1d(truth, followers, assume_unique=True)
    assert int(base_row["new_links"]) == truth.size
    assert int(base_row["new_reciprocal_links"]) == reciprocal_truth.size

    for method, suggested in zip(QUOTA_METHODS, [fixed_top, adaptive_top]):
        assert np.intersect1d(suggested, excluded).size == 0
        matched = np.intersect1d(suggested, truth, assume_unique=True)
        reciprocal_matched = np.intersect1d(matched, reciprocal_truth, assume_unique=True)
        hits = int(matched.size)
        reciprocal_hits = int(reciprocal_matched.size)
        nonreciprocal_hits = hits - reciprocal_hits
        row = base_row.to_dict()
        row.update({
            "user_id": original_id,
            "method": method,
            "hits": hits,
            "hits_reciprocal": reciprocal_hits,
            "hits_nonreciprocal": nonreciprocal_hits,
            "precision_at_10": hits / K_CONFIRM,
            "precision_reciprocal_at_10": reciprocal_hits / K_CONFIRM,
            "precision_nonreciprocal_at_10": nonreciprocal_hits / K_CONFIRM,
            "recall_at_10": hits / truth.size if truth.size else np.nan,
            "hit_at_10": int(hits > 0),
            "recommended_ids": json.dumps(user_ids[suggested].tolist()),
        })
        quota_rows.append(row)

    quota_diagnostics.append({
        "user_id": original_id,
        "initial_following_without_self": degree,
        "initial_mutual_contacts": mutual_count,
        "initial_mutual_fraction": fraction,
        "available_reciprocal_candidates": int(available.size),
        "adaptive_requested": requested_adaptive,
        "adaptive_reserved": adaptive_quota,
        "fixed_reserved": fixed_quota,
        "adaptive_selected_reciprocal": int(np.isin(adaptive_top, available).sum()),
    })

results_with_quotas = pd.concat([base_results, pd.DataFrame(quota_rows)], ignore_index=True)
assert len(results_with_quotas) == N_CONFIRM * len(ALL_METHODS_CONFIRM)
assert not results_with_quotas.duplicated(["user_id", "method"]).any()
summary_with_quotas = results_with_quotas.groupby("method").agg(
    precision_at_10=("precision_at_10", "mean"),
    precision_reciprocal_at_10=("precision_reciprocal_at_10", "mean"),
    precision_nonreciprocal_at_10=("precision_nonreciprocal_at_10", "mean"),
    recall_at_10=("recall_at_10", "mean"),
    hit_rate_at_10=("hit_at_10", "mean"),
    hits=("hits", "sum"),
    hits_reciprocal=("hits_reciprocal", "sum"),
    hits_nonreciprocal=("hits_nonreciprocal", "sum"),
).reindex(ALL_METHODS_CONFIRM)
assert np.allclose(
    summary_with_quotas["precision_at_10"],
    summary_with_quotas["precision_reciprocal_at_10"]
    + summary_with_quotas["precision_nonreciprocal_at_10"],
)

quota_diagnostics_df = pd.DataFrame(quota_diagnostics)
results_with_quotas.to_csv(CONFIRM_DIR / "recommendation_results_with_quotas.csv", index=False)
summary_with_quotas.to_csv(CONFIRM_DIR / "recommendation_summary_with_quotas.csv")
quota_diagnostics_df.to_csv(CONFIRM_DIR / "quota_diagnostics.csv", index=False)

print(f"Valutazione delle quote completata in {time.perf_counter() - quota_start:.1f} s.")
print("Metriche in percentuale:")
display(summary_with_quotas.iloc[:, :5].mul(100).round(3))
print("Collegamenti recuperati:")
display(summary_with_quotas[["hits", "hits_reciprocal", "hits_nonreciprocal"]])
print("Distribuzione delle quote adattive riservate:")
print(quota_diagnostics_df["adaptive_reserved"].value_counts().sort_index().to_string())


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_bootstrap_quotas.py
# Cella: bootstrap appaiato sui 5.000 utenti del campione di conferma.
# Usa solo i file salvati, quindi funziona anche dopo un riavvio della sessione.
# Nessuna operazione sulla GPU.
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CONFIRM_DIR = (Path.cwd() / "flickr_ppr" / "flickr_5000")  # cambiare se i file sono su Drive
RESULTS_PATH = CONFIRM_DIR / "recommendation_results_with_quotas.csv"

protocol = json.loads(
    (CONFIRM_DIR / "recommendation_protocol.json").read_text(encoding="utf-8")
)
quota_protocol = json.loads(
    (CONFIRM_DIR / "quota_policy_protocol.json").read_text(encoding="utf-8")
)

METHODS = list(protocol["methods"]) + list(quota_protocol["methods_added"])
N_USERS = int(protocol["n_users"])
K = int(protocol["k"])
BOOT_RESAMPLES = int(protocol["bootstrap_resamples"])  # 10000
BOOT_SEED = int(protocol["bootstrap_seed"])            # 20261005
CHUNK = 500  # ricampionamenti elaborati per volta, per limitare la memoria

results = pd.read_csv(RESULTS_PATH)
assert len(METHODS) == 7
assert set(results["method"]) == set(METHODS)
assert len(results) == N_USERS * len(METHODS)
assert not results.duplicated(["user_id", "method"]).any()


def user_by_method(column):
    """Una riga per utente, una colonna per metodo, nello stesso ordine."""
    table = results.pivot(index="user_id", columns="method", values=column)
    return table.reindex(columns=METHODS)


precision = user_by_method("precision_at_10")
users = precision.index
hit = user_by_method("hit_at_10").loc[users]
recall = user_by_method("recall_at_10").loc[users]

assert precision.shape == (N_USERS, len(METHODS))
assert not precision.isna().any().any()
assert not hit.isna().any().any()

P = precision.to_numpy(dtype=np.float64)
H = hit.to_numpy(dtype=np.float64)

# Il richiamo è definito solo per gli utenti con almeno un nuovo collegamento:
# per ogni utente deve essere definito per tutti i metodi o per nessuno.
assert (recall.isna().all(axis=1) == recall.isna().any(axis=1)).all()
active = recall.notna().all(axis=1).to_numpy()
R = recall.to_numpy(dtype=np.float64)[active]


def bootstrap_means(matrices, n_resamples, seed, chunk=CHUNK):
    """Ricampiona gli utenti con reinserimento.

    Lo stesso ricampionamento vale per tutti i metodi e per tutte le
    matrici passate insieme: il confronto è quindi appaiato per utente.
    """
    rng = np.random.default_rng(seed)
    n_rows = matrices[0].shape[0]
    assert all(m.shape[0] == n_rows for m in matrices)
    outputs = [np.empty((n_resamples, m.shape[1])) for m in matrices]

    done = 0
    while done < n_resamples:
        size = min(chunk, n_resamples - done)
        idx = rng.integers(0, n_rows, size=(size, n_rows))
        for out, m in zip(outputs, matrices):
            out[done:done + size] = m[idx].mean(axis=1)
        done += size
    return outputs


boot_P, boot_H = bootstrap_means([P, H], BOOT_RESAMPLES, BOOT_SEED)
(boot_R,) = bootstrap_means([R], BOOT_RESAMPLES, BOOT_SEED + 1)


def interval(samples):
    """Intervallo percentile al 95%."""
    return np.quantile(samples, [0.025, 0.975], axis=0)


p_low, p_high = interval(boot_P)
h_low, h_high = interval(boot_H)
r_low, r_high = interval(boot_R)

method_rows = []
for j, method in enumerate(METHODS):
    method_rows.append({
        "method": method,
        "precisione_pct": 100 * P[:, j].mean(),
        "precisione_low_pct": 100 * p_low[j],
        "precisione_high_pct": 100 * p_high[j],
        "utenti_con_successo_pct": 100 * H[:, j].mean(),
        "utenti_con_successo_low_pct": 100 * h_low[j],
        "utenti_con_successo_high_pct": 100 * h_high[j],
        "richiamo_attivi_pct": 100 * R[:, j].mean(),
        "richiamo_low_pct": 100 * r_low[j],
        "richiamo_high_pct": 100 * r_high[j],
        "collegamenti_recuperati": int(round(P[:, j].sum() * K)),
    })
methods_df = pd.DataFrame(method_rows).set_index("method")

# Controllo: i collegamenti ricostruiti dalla precisione coincidono con quelli salvati.
saved_hits = results.groupby("method")["hits"].sum().reindex(METHODS)
assert (methods_df["collegamenti_recuperati"] == saved_hits).all()

COMPARISONS = [
    # Metodo combinato contro ciascun metodo singolo.
    ("Reciprocità + PPR", "Personalized PageRank"),
    ("Reciprocità + PPR", "Reciprocità + grado"),
    ("Reciprocità + PPR", "PageRank globale"),
    ("Reciprocità + PPR", "Grado entrante"),
    # Analisi esplorativa delle quote.
    ("Reciprocità + PPR", "PPR + quota adattiva"),
    ("Reciprocità + PPR", "PPR + quota fissa (5)"),
    ("PPR + quota adattiva", "PPR + quota fissa (5)"),
    # PageRank personalizzato contro la popolarità e contro la reciprocità.
    ("Personalized PageRank", "PageRank globale"),
    ("Personalized PageRank", "Grado entrante"),
    ("Personalized PageRank", "Reciprocità + grado"),
]
column = {method: j for j, method in enumerate(METHODS)}
assert all(a in column and b in column for a, b in COMPARISONS)

comparison_rows = []
for a, b in COMPARISONS:
    ja, jb = column[a], column[b]
    observed = P[:, ja].mean() - P[:, jb].mean()
    resampled = boot_P[:, ja] - boot_P[:, jb]
    low, high = np.quantile(resampled, [0.025, 0.975])
    comparison_rows.append({
        "confronto": f"{a} meno {b}",
        "differenza_punti_pct": 100 * observed,
        "low_95": 100 * low,
        "high_95": 100 * high,
        "differenza_collegamenti": int(round((P[:, ja].sum() - P[:, jb].sum()) * K)),
        "frazione_ricampionamenti_diff_non_positiva": float(np.mean(resampled <= 0)),
        "intervallo_esclude_zero": bool(low > 0 or high < 0),
    })
comparisons_df = pd.DataFrame(comparison_rows).set_index("confronto")

methods_df.to_csv(CONFIRM_DIR / "bootstrap_metodi_with_quotas.csv")
comparisons_df.to_csv(CONFIRM_DIR / "bootstrap_confronti_with_quotas.csv")

bootstrap_record = {
    "input": RESULTS_PATH.name,
    "input_sha256": hashlib.sha256(RESULTS_PATH.read_bytes()).hexdigest(),
    "users": N_USERS,
    "active_users_for_recall": int(active.sum()),
    "methods": METHODS,
    "resamples": BOOT_RESAMPLES,
    "seed_precision_and_hit_rate": BOOT_SEED,
    "seed_recall": BOOT_SEED + 1,
    "pairing": "same resampled users for all methods",
    "interval": "percentile, 2.5% and 97.5%",
}
(CONFIRM_DIR / "bootstrap_protocollo_with_quotas.json").write_text(
    json.dumps(bootstrap_record, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"Utenti: {N_USERS}; utenti attivi per il richiamo: {int(active.sum())}")
print(f"Ricampionamenti: {BOOT_RESAMPLES}; semi: {BOOT_SEED} e {BOOT_SEED + 1}")

print("\nMetodi, in percentuale, con intervalli al 95%:")
display(methods_df.round(3))

print("\nConfronti appaiati sulla precisione, in punti percentuali:")
display(comparisons_df.round(3))

print("\nFile salvati in:", CONFIRM_DIR)


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_tie_audit.py
# Cella: stima degli spareggi a punteggio zero su un sottocampione.
# Richiede la sessione GPU con gli oggetti già in memoria: G0, GT0, user_ids,
# confirm_users, incoming_gpu, inv_degrees_gpu, dangling_gpu, pagerank_gpu,
# top_k_gpu, hybrid_recommendations, CONFIRM_DIR, K_CONFIRM.
# Se la sessione è ripartita, rieseguire prima le celle di preparazione del grafo
# e delle funzioni GPU, e quelle del campione di conferma.
import json
import time

import cupy as cp
import numpy as np
import pandas as pd
from IPython.display import display

N_CHECK = 300
CHECK_SEED = 20261007  # diverso dai semi già usati

rng_check = np.random.default_rng(CHECK_SEED)
check_users = rng_check.choice(confirm_users, size=N_CHECK, replace=False)

saved = pd.read_csv(CONFIRM_DIR / "recommendation_results.csv")
saved_lists = {
    method: saved[saved["method"] == method].set_index("user_id")["recommended_ids"]
    for method in ["Personalized PageRank", "Reciprocità + PPR"]
}


def wilson(successes, n, z=1.96):
    """Intervallo di Wilson al 95% per una proporzione."""
    if n == 0:
        return (np.nan, np.nan)
    p = successes / n
    center = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (max(0.0, center - half), min(1.0, center + half))


rows = []
check_start = time.perf_counter()

for position, internal_user in enumerate(check_users, start=1):
    u = int(internal_user)
    original_id = int(user_ids[u])

    followed = G0.indices[G0.indptr[u]:G0.indptr[u + 1]]
    followers = GT0.indices[GT0.indptr[u]:GT0.indptr[u + 1]]
    excluded = np.union1d(followed, np.asarray([u], dtype=np.int32))

    rank, info = pagerank_gpu(
        incoming_gpu, inv_degrees_gpu, dangling_gpu,
        user=u, alpha=0.85, tol=1e-8, max_iter=1000,
    )
    ppr_top = top_k_gpu(rank, excluded, k=K_CONFIRM)
    hybrid_top = hybrid_recommendations(u, rank, excluded, ppr_top, k=K_CONFIRM)

    # 1. Le liste ricalcolate coincidono con quelle salvate?
    same_ppr = user_ids[ppr_top].tolist() == json.loads(
        saved_lists["Personalized PageRank"].loc[original_id]
    )
    same_hybrid = user_ids[hybrid_top].tolist() == json.loads(
        saved_lists["Reciprocità + PPR"].loc[original_id]
    )

    # 2. PageRank personalizzato: suggerimenti a punteggio zero e spareggio al confine.
    selected_scores = cp.asnumpy(rank[cp.asarray(ppr_top)])
    zeros_in_ppr = int(np.count_nonzero(selected_scores == 0))

    rest = rank.copy()
    rest[cp.asarray(excluded)] = -cp.inf
    rest[cp.asarray(ppr_top)] = -cp.inf
    best_left_out = float(rest.max().item())
    # Il confine conta solo per l'insieme dei 10, non per l'ordine interno.
    ppr_boundary_tie = bool(selected_scores.min() == best_left_out)

    candidates_positive = int(cp.count_nonzero(rank > 0).item()) - int(
        cp.count_nonzero(rank[cp.asarray(excluded)] > 0).item()
    )
    del rest

    # 3. Metodo combinato: follower disponibili a punteggio zero e taglio a 10.
    available = np.setdiff1d(followers, excluded, assume_unique=True)
    if available.size:
        available_scores = cp.asnumpy(rank[cp.asarray(available)])
    else:
        available_scores = np.empty(0)
    zero_followers = int(np.count_nonzero(available_scores == 0))

    block = hybrid_top[np.isin(hybrid_top, available)]
    block_scores = available_scores[np.searchsorted(available, block)]
    zeros_in_block = int(np.count_nonzero(block_scores == 0))

    truncated = available.size > K_CONFIRM
    if truncated:
        left_out_scores = available_scores[~np.isin(available, block)]
        hybrid_boundary_tie = bool(block_scores.min() == left_out_scores.max())
    else:
        hybrid_boundary_tie = False

    rows.append({
        "user_id": original_id,
        "iterations": int(info["iterations"]),
        "same_ppr_list": same_ppr,
        "same_hybrid_list": same_hybrid,
        "ppr_zero_suggestions": zeros_in_ppr,
        "ppr_boundary_tie": ppr_boundary_tie,
        "ppr_positive_candidates": candidates_positive,
        "available_followers": int(available.size),
        "zero_score_followers": zero_followers,
        "hybrid_zero_in_follower_block": zeros_in_block,
        "hybrid_block_truncated": bool(truncated),
        "hybrid_boundary_tie": hybrid_boundary_tie,
    })
    del rank

    if position % 50 == 0 or position == N_CHECK:
        print(f"{position}/{N_CHECK} utenti — {time.perf_counter() - check_start:.1f} s",
              flush=True)

check_df = pd.DataFrame(rows)
check_df.to_csv(CONFIRM_DIR / "controllo_spareggi_zero.csv", index=False)


def report(label, mask):
    count = int(mask.sum())
    low, high = wilson(count, N_CHECK)
    return {
        "controllo": label,
        "utenti": count,
        "su": N_CHECK,
        "percentuale": 100 * count / N_CHECK,
        "low_95_pct": 100 * low,
        "high_95_pct": 100 * high,
    }


summary_rows = [
    report("Lista PPR ricalcolata identica a quella salvata", check_df["same_ppr_list"]),
    report("Lista combinata ricalcolata identica a quella salvata",
           check_df["same_hybrid_list"]),
    report("Lista PPR con almeno un suggerimento a punteggio zero",
           check_df["ppr_zero_suggestions"] > 0),
    report("Lista PPR con spareggio al confine dei 10",
           check_df["ppr_boundary_tie"]),
    report("Follower disponibili con punteggio zero (almeno uno)",
           check_df["zero_score_followers"] > 0),
    report("Blocco dei follower nel combinato con punteggi zero",
           check_df["hybrid_zero_in_follower_block"] > 0),
    report("Blocco dei follower troncato a 10",
           check_df["hybrid_block_truncated"]),
    report("Blocco troncato con spareggio al confine",
           check_df["hybrid_boundary_tie"]),
]
summary_df = pd.DataFrame(summary_rows).set_index("controllo")

(CONFIRM_DIR / "controllo_spareggi_zero.json").write_text(
    json.dumps({
        "users_checked": N_CHECK,
        "sampling_seed": CHECK_SEED,
        "population": "confirmation sample of 5000 users",
        "alpha": 0.85, "tol": 1e-8, "k": int(K_CONFIRM),
        "interval": "Wilson 95%",
    }, indent=2),
    encoding="utf-8",
)

print("\nRiepilogo, con intervalli di Wilson al 95%:")
display(summary_df.round(2))
print("\nCandidati con punteggio positivo per utente (min, mediana, max):",
      int(check_df["ppr_positive_candidates"].min()),
      int(check_df["ppr_positive_candidates"].median()),
      int(check_df["ppr_positive_candidates"].max()))
print("File salvati in:", CONFIRM_DIR)


In [ ]:
%%writefile _pagerank_runtime/scripts/flickr_run.py
from pathlib import Path
import os
source_dir = Path(__file__).resolve().parent
namespace = {'__name__': '__main__'}
names = ['flickr_00.py', 'flickr_01.py', 'flickr_02.py', 'flickr_03.py', 'flickr_04.py', 'flickr_05.py', 'flickr_08.py', 'flickr_09.py', 'flickr_10.py', 'flickr_bootstrap.py']
if os.environ.get('THESIS_FLICKR_EXPLORATORY') == '1':
    names += ['flickr_quotas.py', 'flickr_bootstrap_quotas.py']
if os.environ.get('THESIS_FLICKR_TIE_AUDIT') == '1':
    names += ['flickr_tie_audit.py']
for name in names:
    path = source_dir / name
    print('\nRunning', name, flush=True)
    namespace['__file__'] = str(path)
    exec(compile(path.read_text(encoding='utf-8'), str(path), 'exec'), namespace)


<a id="run"></a>
## 9. Run selected experiments

Enable the desired flags in Section 1, then run all cells in order. The source cells
above must execute before this cell. Each experiment has its own directory and
each subprocess releases its GPU memory on exit. Existing Flickr checkpoints are
reused only after checking the stored sample and protocol. Use a new run directory
if you intentionally change the protocol.


In [ ]:
if (RUN_FLICKR_EXPLORATORY or RUN_FLICKR_TIE_AUDIT) and not RUN_FLICKR:
    raise ValueError('Enable RUN_FLICKR for the optional Flickr analyses.')

gap_selected = any([RUN_GAP_PREPARATION, RUN_GAP_POWER, RUN_GAP_KERNEL_COMPARISON,
                    RUN_GAP_WEIGHTED_KERNELS, RUN_GAP_MONTE_CARLO, RUN_GAP_VALIDATION])
if gap_selected:
    if sys.platform != 'linux':
        raise RuntimeError('Run the GAP workflow on Linux/Colab (gcc, tar and wget are required).')
    for tool in ('gcc', 'tar', 'wget'):
        if shutil.which(tool) is None:
            raise RuntimeError(f'Missing GAP preparation tool: {tool}')

if RUN_WEB_BENCHMARK or RUN_WEB_KERNEL_COMPARISON:
    run_program('web_benchmark.py', 'web', gpu=True,
                extra_env={'THESIS_WEB_KERNELS': str(int(RUN_WEB_KERNEL_COMPARISON))})

if RUN_CONVERGENCE:
    graph = download_web_google(RUN_DIR / 'web')
    result_dir = run_program('figure_convergenza.py', 'convergence', graph)
    for figure in ('fig_convergenza.png', 'fig_fattore.png'):
        display(Image(filename=str(result_dir / figure)))

if RUN_TWITTER_MONTE_CARLO:
    run_program('montecarlo.py', 'twitter', 'twitter', '--N', '1e6,1e7,1e8',
                '--seeds', '1,2,3', '--batch', '1000000', '--workdir', '.', gpu=True)

if RUN_GAP_PREPARATION:
    run_program('prepara_gap_kernel.py', 'gap')
if RUN_GAP_POWER:
    run_program('power_gap_follow.py', 'gap', '--workdir', '.', gpu=True)
if RUN_GAP_KERNEL_COMPARISON:
    run_program('benchmark_vector_unweighted_gap.py', 'gap', '--workdir', '.', gpu=True)
if RUN_GAP_WEIGHTED_KERNELS:
    run_program('benchmark_kernel_gap.py', 'gap', '--workdir', '.', gpu=True)
if RUN_GAP_MONTE_CARLO:
    run_program('montecarlo.py', 'gap', 'gap', '--N', '1e7,1e8,1e9', '--seeds', '1,2',
                '--batch', '1000000', '--workdir', '.', gpu=True)
if RUN_GAP_VALIDATION:
    run_program('verify_gap_follow.py', 'gap', '--workdir', '.', gpu=True)

if RUN_FLICKR:
    run_program('flickr_run.py', 'flickr', gpu=True, extra_env={
        'THESIS_FLICKR_EXPLORATORY': str(int(RUN_FLICKR_EXPLORATORY)),
        'THESIS_FLICKR_TIE_AUDIT': str(int(RUN_FLICKR_TIE_AUDIT)),
    })
print('Selected experiments completed. Results directory:', RUN_DIR)


<a id="sources"></a>
## 10. Sources and organisation

### Data and methodological references
* [SNAP web-Google](https://snap.stanford.edu/data/web-Google.html)
* [SNAP ego-Twitter](https://snap.stanford.edu/data/ego-Twitter.html)
* [SuiteSparse GAP-twitter](https://sparse.tamu.edu/GAP/GAP-twitter)
* [Flickr growth data](https://socialnetworks.mpi-sws.org/data-imc2007.html)
* Bell and Garland, *Efficient Sparse Matrix-Vector Multiplication on CUDA*,
  NVR-2008-004, Section 4.3 (CSR-vector structure; adapted CUDA implementation).
* Avrachenkov et al., *Monte Carlo Methods in PageRank Computation: When One
  Iteration Is Sufficient*, SIAM Journal on Numerical Analysis 45(2), 2007
  (endpoint Monte Carlo).

Datasets are downloaded from their providers when an experiment is enabled;
they are not redistributed in this notebook. See the thesis bibliography for the
full mathematical and experimental references.

### Consolidation notes
* `PageRank_Benchmark_GPU.ipynb`: retained data preparation, references, CPU/GPU
  benchmarks and the supporting web kernel experiment.
* `convergenza.ipynb` and `Montecarlo (1).ipynb`: replaced repeated shell history
  with the complete scripts and ordered execution controls.
* `correzione_csr_colab.ipynb`: its runtime-dependent orientation repair is
  superseded by the final follower-to-followed preparation and power scripts.
* `Recommender.ipynb`: retained final sampling, ranking, checkpointing and evaluation;
  the pilot sample is reconstructed but the preliminary pilot evaluation is omitted.
  Exploratory quota policies and the tie audit are optional sections.
* Removed empty cells, duplicate downloads, session-specific `pkill` commands,
  Colab export dialogs, cached notebook outputs and dependencies on manually uploaded scripts.
* Adapted hard-coded working paths and supplied missing imports for standalone use.
  Baseline and exploratory Flickr bootstrap outputs have distinct filenames.

Original benchmark precision, thresholds, random seeds and repetition counts are
preserved. The original comments sometimes describe exact-arithmetic error bounds;
these do not become strict floating-point guarantees. New hardware or software may
change runtimes and numerical rounding. Historical results above are retained as
reference measurements, not guaranteed outcomes of a rerun.

### Source fingerprints
The relative paths and SHA-256 fingerprints below document the local inputs used
to assemble this notebook. They are provenance records, not runtime dependencies.


| Original source | SHA-256 |
|---|---|
| `PageRank_Benchmark_GPU.ipynb` | `4ae1089d2c2d2ac916736bffb363d6c9b1537ca3c492abede513f4bc79b9c7b5` |
| `FLICKR_PPR/risultati_finali/Recommender.ipynb` | `a76610d0e4f49db193f060ec2030c8548a2da64b3d500e6f0890bd92c1b16f65` |
| `FIGURE_CONVERGENZA/convergenza.ipynb` | `4dc5888f90be6806c81d060ba155af7dabfd5ae8b0d8cb65d438d23919ca4eb6` |
| `VERIFY_risultati/Montecarlo (1).ipynb` | `33b17af54e5ef89f5d1cff76dbd3e1a114bc8d41b92def89dffd14a22269bddc` |
| `correzione_csr_colab.ipynb` | `3a25b99a03d551c0e1efe5c74f1a07000f722aa92120b73a6a51bd8bd39d672d` |
| `benchmark_web_google_gpu (1).json` | `41e5c37d21a7c0f0bb17b9c9428e2dbaa46f9d84bc370e684656c4d4f7fa26f5` |
| `benchmark_vector_unweighted_gap.json` | `802d53c4fa956a3b5f41a0cfa3cdfcf43305474246495610938be69d270fe6bb` |
| `FLICKR_PPR/risultati_finali/flickr_5000/bootstrap_metodi.csv` | `598fa1888904b3e35b477b2a520f08d4362b3dd7cf6fc160696d86a6465665fd` |
| `montecarlo.py` | `5ccd72a498bcc11911bbd58bc106020eefebf156a80e2700f61dbf98017b3f8f` |
| `benchmark_kernel_web.py` | `bbcfa02daf7b45da43820a802efbc21c4196001125b5fce17c6df6e564d5f28f` |
| `FIGURE_CONVERGENZA/figure_convergenza_colab.py` | `cad04dc609f6b784b090e0e015e28955b5a8273c4ba8d36e4139e16f67131e6c` |
| `prepara_gap_kernel.py` | `20f858910738c59bace519995dc84b95384b8bcb7fb83292559968d18bd84fb5` |
| `power_gap_follow.py` | `832fb8dee0795de3408c8c412c52a309763c218c8e6d9fa438991515e6ce145e` |
| `benchmark_vector_unweighted_gap.py` | `15295fa4406dde083e7c4b4ca25558c6da4a591e95f1ad4b7856473bda406686` |
| `benchmark_kernel_gap.py` | `eb4c23578d0d15927ed0cabbf21f1e6f82db9f39c4be5dd7331c37aba6c85224` |
| `risultati_gpu_25set.zip::verify_gap_follow.py` | `f66d38995cbf51ebd2a37ea580e06df660c99864ccf79566a199091788738368` |
